# Redshift

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/redshift.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## What is Amazon Redshift?

Amazon Redshift is a fully managed, massively parallel, columnar data warehouse on AWS. A cluster has a leader node that plans and coordinates queries plus compute nodes divided into slices that store data and run query fragments. RA3 nodes separate compute from Redshift Managed Storage, so you can scale compute without moving data, and Redshift Serverless provides the same engine without cluster management.

```sql
-- The shape of a typical Redshift warehouse query
select
  d.calendar_month,
  p.category,
  sum(f.net_amount) as revenue
from analytics.fact_sales f
join analytics.dim_date d on f.date_key = d.date_key
join analytics.dim_product p on f.product_key = p.product_key
where d.calendar_year = 2026
group by d.calendar_month, p.category
order by d.calendar_month, revenue desc;
```

The leader node parses SQL, builds and optimizes the plan, compiles it, and coordinates the compute nodes. It holds the catalog and metadata, not user data.

Compute nodes are split into slices. Each slice owns a share of CPU, memory and disk, and a table's rows live on those slices. Adding nodes adds slices, which adds scan bandwidth and memory. Slice counts differ by node type: dc2.large has 2 per node, ra3.xlplus has 4, ra3.4xlarge has 12, and ra3.16xlarge has 48.

MPP means each slice scans only its own blocks and the leader merges the partial results. That is why a scan of one huge fact table still finishes quickly, and why a single oversized input file can bottleneck a load.

RA3 adds Redshift Managed Storage, which separates compute from storage. Local SSD becomes a cache rather than the source of truth, so compute can be resized without copying all the data. AQUA is an RA3 feature that offloads some scans and aggregation to an on-storage cache layer. It is enabled per cluster and has no SQL DDL.

Common mistake: assuming the leader node does the heavy scanning or stores table data. The leader plans and coordinates; the slices do the work and own the rows. Watching leader CPU to explain a slow scan looks in the wrong place. Slice-level metrics and the query plan show where the work actually went.

## How do distribution styles and sort keys affect Redshift performance?

Distribution style decides which slice owns each row, which determines whether a join needs a network redistribution. Sort key decides physical order on each slice, which drives zone-map pruning and merge-join efficiency.

+ `DISTSTYLE KEY` colocates matching keys, removing redistribution for that join.
+ `DISTSTYLE ALL` copies a small table to every slice, turning a join into a local lookup.
+ `DISTSTYLE EVEN` round-robins rows; good for tables with no dominant join key.
+ `DISTSTYLE AUTO` lets Redshift choose and change as the workload evolves.
+ `SORTKEY` compound is ordered by priority; `INTERLEAVED` gives equal weight to several columns at the cost of expensive vacuum.

```sql
create table analytics.fact_sales (
  sale_id      bigint       encode az64,
  date_key     integer      encode az64,
  product_key  integer      encode az64,
  store_key    integer      encode az64,
  net_amount   numeric(12,2) encode az64,
  quantity     integer      encode az64
)
diststyle key
distkey (product_key)
sortkey (date_key, store_key);

create table analytics.dim_product (
  product_key integer not null,
  category    varchar(64),
  brand       varchar(64)
)
diststyle all
sortkey (product_key);
```

The fact table colocates on `product_key` with the small dimension copied everywhere, so the common star-schema join has no data movement.

AUTO starts small tables at ALL. A large table becomes AUTO(EVEN), unless Redshift finds a recurring join column and makes it AUTO(KEY). EVEN round-robins rows, which is fine when no key dominates. KEY hashes the distkey so matching values land on the same slice. ALL copies the entire table to every slice and only makes sense for small dimensions.

Sort keys drive zone-map pruning. Compound is the default and follows the leading columns first, which suits date-first range filters. Interleaved gives several columns equal weight, costs more to maintain, and is meant for low-cardinality filters.

```sql
alter table analytics.fact_sales alter diststyle key distkey (product_key);
alter table analytics.dim_product alter diststyle all;

alter table analytics.fact_sales alter sortkey (date_key, store_key);
alter table analytics.fact_sales alter sortkey auto;
alter table analytics.stg_clicks alter sortkey none;
```
Interleaved sort keys can be converted to compound, never the reverse. ALTER and VACUUM cannot run on the same table at the same time. Sort key columns default to RAW encoding.

Common mistake: choosing an interleaved sort key for a high-cardinality column such as an order id or a timestamp. Interleaved is built for low-cardinality filters and needs VACUUM REINDEX, which scans and rebuilds the sort metadata. If the workload filters mainly on one date column, compound is cheaper. Also check that the distkey really is the busiest join key, because a wrong KEY style forces redistribution on every join.

## How do you load data efficiently into Redshift?

Bulk load with `COPY` from Amazon S3 using an IAM role, compressed columnar files, and many files so every slice works in parallel. Avoid single-row `INSERT` for volume, and run `ANALYZE` (and `VACUUM` when needed) after large changes.

```sql
copy analytics.fact_sales
from 's3://acme-lake/manifests/fact_sales.manifest'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
manifest
region 'us-east-1'
statupdate on
compupdate on;
```

```bash
# How many objects are waiting to load, and how big are they?
aws s3 ls s3://acme-lake/curated/fact_sales/ --recursive --human-readable --summarize
```

Rules that matter: use one file per slice (or a manifest with many files), load Parquet or ORC over CSV, never load through the leader node with a single huge file, and keep the S3 bucket in the same region as the cluster.

COPY reads S3 in parallel across slices, so the number of files matters as much as the format. Aim for a multiple of the cluster's slice count, roughly one file per slice. A manifest lists the exact set of files and is the safest way to control the load.

Options you will actually use: IGNOREHEADER, DELIMITER, GZIP, ENCRYPTED, TRUNCATECOLUMNS, ACCEPTINVCHARS, EXPLICIT_IDS, REGION, MAXERROR. STATUPDATE ON refreshes table statistics at the end of the load. COMPUPDATE PRESET tells COPY to reuse the table's existing encodings instead of sampling the incoming data.

```sql
copy analytics.stg_orders
from 's3://acme-lake/raw/orders/'
iam_role 'arn:aws:iam::123456789012:role/redshift-staging-role'
csv
ignoreheader 1
delimiter ','
gzip
truncatecolumns
acceptinvchars
maxerror 10;
```

Common mistake: loading one multi-gigabyte CSV file. That is a single slice of work, so the rest of the cluster idles. Another is setting MAXERROR high to get past a bad file. The load succeeds, but bad rows are recorded in STL_LOAD_ERRORS and you learn about them from a report instead of the load. Check STL_LOAD_ERRORS after any load that reports skipped rows.

## What are the key benefits of using Amazon Redshift?

+ Fully managed MPP warehouse with columnar storage and compression.
+ RA3 separates compute from managed storage, so storage grows without resizing.
+ Redshift Spectrum queries S3 data lake files without loading them.
+ Concurrency scaling absorbs burst concurrency; short query acceleration speeds dashboards.
+ Mature SQL surface: window functions, stored procedures, materialized views, UDFs, cross-database queries.
+ Deep AWS integration: IAM, Glue, Lake Formation, KMS, CloudWatch, Kinesis, Lambda.
+ Data sharing and zero-ETL integrations with operational databases.

```sql
-- A workload only a columnar MPP warehouse handles comfortably
select
  customer_id,
  sum(net_amount) as lifetime_value,
  rank() over (order by sum(net_amount) desc) as value_rank
from analytics.fact_sales
where date_key >= 20240101
group by customer_id
qualify value_rank <= 100;
```

Each benefit is a real mechanism. Spectrum runs on a separate fleet, so S3 scans do not compete with local compute. Concurrency scaling adds transient clusters for read bursts and is controlled by max_concurrency_scaling_clusters; queries that ran on one show `compute_type = 'primary-scale'` in `SYS_QUERY_HISTORY`. Short query acceleration gives short queries a dedicated path. The result cache answers repeated identical queries without re-running them. Data sharing exposes data across clusters without copies.

```sql
set enable_result_cache_for_session to off;

select customer_id, sum(net_amount) as lifetime_value
from analytics.fact_sales
group by customer_id;
```

Common mistake: treating a listed benefit as automatic. The result cache is skipped when a query uses non-deterministic functions and is invalidated by any data change in a referenced table. Concurrency scaling, short query acceleration, and Spectrum all depend on cluster configuration and IAM setup. Configure and verify the feature you rely on rather than assuming it is on.

## How does Amazon Redshift achieve scalability?

Redshift scales in three dimensions: node count and type (elastic or classic resize, or Serverless base capacity), storage (RA3 Redshift Managed Storage grows independently), and concurrency (concurrency scaling adds transient clusters for read bursts). AQUA accelerates scans on RA3 by pushing work to a hardware cache.

```bash
# Serverless: raise base capacity (RPUs) for a heavy month-end window
aws redshift-serverless update-workgroup \
  --workgroup-name analytics-wg \
  --base-capacity 256

# Provisioned: elastic resize an RA3 cluster
aws redshift modify-cluster \
  --cluster-identifier analytics-cluster \
  --node-type ra3.4xlarge \
  --number-of-nodes 12
```

Elastic resize adds or removes nodes quickly and keeps the cluster available. Classic resize moves all the data and takes much longer. On RA3, Redshift Managed Storage separates storage from compute, so you resize for CPU and memory without copying the data. Serverless scales by RPUs and has no nodes to choose. Concurrency scaling is a separate axis: it adds transient clusters for read traffic, governed by max_concurrency_scaling_clusters.

```sql
select count(*) as slice_count
from stv_slices;
```
The slice count is the unit that matters. It changes with node type, so a resized cluster plans the same SQL differently. Counting slices after a resize confirms the new parallelism.

Common mistake: resizing compute to fix slow queries that are really a distribution or sort-key problem. More slices spread the same skewed work wider, and the slowest slice still gates the query. Check skew_rows and unsorted in SVV_TABLE_INFO and look at the plan before paying for a resize.

## What is columnar storage, and why is it important in Redshift?

Columnar storage keeps each column's values together on disk instead of storing whole rows. Analytical queries touch a few columns but many rows, so columnar layout reads far less data, compresses better because values in a column share a type and range, and lets zone maps skip whole blocks. Redshift combines this with per-column encodings.

```sql
-- Reads only four of the wide table's columns and compresses to those blocks
select date_key, store_key, sum(net_amount) as revenue, sum(quantity) as units
from analytics.fact_sales
where date_key between 20261001 and 20261031
group by date_key, store_key;
```

On each slice, a column is stored in 1 MB blocks. Every block keeps min and max metadata, so a range predicate can skip whole blocks that cannot match. Fewer blocks touched means less I/O. Values in one column also share a type and range, so they compress better together than mixed row values would.

Redshift chooses encodings per column. Numeric and date columns default to AZ64, char and varchar to LZO, and sort key, boolean and float columns to RAW. ENCODE AUTO is the table-level default and lets Redshift change encodings as data grows.

```sql
alter table analytics.dim_product alter column category encode zstd;
alter table analytics.fact_sales alter column net_amount encode az64;

analyze compression analytics.fact_sales;
```
ANALYZE COMPRESSION only reports recommended encodings. It does not change any data, so you still apply the suggestions yourself.

Common mistake: expecting a columnar store to be fast at point lookups. A query like `select * from analytics.fact_sales where sale_id = 12345` reads blocks from every column just to assemble one row, which is slower than a row store would be. Columnar wins on wide scans over a few columns, not on fetching single rows.

## How is data distributed in Amazon Redshift?

The leader node parses and plans; compute nodes are partitioned into slices, and each slice owns a subset of every table's rows. Distribution style controls the mapping. During a join, if the joining rows are not colocated, Redshift either redistributes one side, broadcasts the smaller side, or uses a merge join on a common sort key. Minimizing that movement is the core of physical design.

```text
leader node  -> parse, plan, coordinate
   |
   +-- compute node 1 [slice 0][slice 1]
   +-- compute node 2 [slice 0][slice 1]
   +-- compute node N [slice 0][slice 1]
```

When joined tables are not colocated, the planner has three choices. It can redistribute one side on the join key, broadcast the smaller side to every slice, or use a merge join with no movement when both sides are already sorted on the join key. Statistics and table size drive the choice.

A small dimension with DISTSTYLE ALL often removes the problem entirely, because every slice already holds a copy.

```sql
explain
select p.category, sum(f.net_amount) as revenue
from analytics.fact_sales f
join analytics.dim_product p on f.product_key = p.product_key
group by p.category;
```
Read the plan labels. DS_DIST_NONE means no data movement, DS_BCAST_INNER means the inner side was broadcast, and DS_DIST_BOTH means both sides were redistributed. DS_DIST_BOTH on a large join is usually the thing to remove.

Common mistake: distributing two large tables on different keys and joining them anyway. Each execution then redistributes one or both sides, and the cost grows with data volume. Choose one distribution key that serves the busiest join, or accept EVEN and lean on sort keys where some movement is unavoidable.

## What is a Redshift cluster?

A Redshift cluster is the provisioned unit of the warehouse: a leader node, one or more compute nodes of a chosen type, parameter and security groups, and a database. Modern deployments often use RA3 nodes or Redshift Serverless, which removes node management while keeping the same SQL engine.

```bash
aws redshift describe-clusters \
  --cluster-identifier analytics-cluster \
  --query 'Clusters[0].{Status:ClusterStatus,Nodes:NumberOfNodes,Type:NodeType,DB:DBName}'
```

A provisioned cluster bundles a leader node, compute nodes of one type, a parameter group, a security group, a cluster subnet group, and at least one database. Clients connect to the cluster endpoint on port 5439. The parameter group holds settings such as the WLM JSON configuration.

RA3 nodes rest on Redshift Managed Storage, while dc2 nodes rely on local SSD only. Redshift Serverless has no cluster or nodes at all; you choose a workgroup and a base capacity in RPUs, and the engine is the same one that runs on provisioned clusters.

Common mistake: assuming more nodes always make a query faster. Node count helps only when the work spreads evenly across slices. A small result, a skewed table, or a load built from one unsorted file will not speed up with more nodes. Also remember that Serverless runs the same engine, so the SQL and tuning advice carry over unchanged.

## How can you optimize query performance in Amazon Redshift?

+ Distribute and sort for the dominant joins and filters (or trust `AUTO`).
+ Keep statistics fresh with `ANALYZE`; stale stats cause bad join order and scan choices.
+ Run `VACUUM` after heavy delete/update churn so space is reclaimed and rows are sorted.
+ Use columnar compressed files for `COPY`, and load in parallel.
+ Materialize expensive repeated aggregations as materialized views.
+ Use `EXPLAIN` to confirm there is no unexpected `DS_DIST_BOTH` or full-table scan.
+ Watch WLM queue wait and use concurrency scaling or short query acceleration for dashboards.

```sql
analyze analytics.fact_sales;
vacuum sort only analytics.fact_sales to 75 percent;

explain
select p.category, sum(f.net_amount)
from analytics.fact_sales f
join analytics.dim_product p on f.product_key = p.product_key
where f.date_key between 20261001 and 20261031
group by p.category;
```

Start with the table-level facts before changing any SQL.

```sql
select "schema", "table", diststyle, sortkey1, size, pct_used,
       unsorted, stats_off, skew_rows, tbl_rows
from svv_table_info
where "schema" = 'analytics'
order by size desc;
```

Read the numbers carefully. size is in 1 MB blocks. stats_off of 0 means current statistics and 100 means out of date. unsorted is the percent of rows not in sort order. skew_rows is the largest slice row count divided by the smallest; a large ratio means one slice does most of the work. ANALYZE is incremental, so refreshing statistics is cheaper than it sounds. VACUUM handles space reclamation and sort order, and VACUUM SORT ONLY or VACUUM DELETE ONLY targets one job at a time.

Common mistake: running a full VACUUM on a schedule regardless of need, or skipping ANALYZE after a large load. VACUUM cannot run inside a transaction block and cannot run at the same time as ALTER on the same table. ANALYZE is the cheap command and fixes most bad plans. Check stats_off and unsorted first, then vacuum only the tables that need it.

## What is a sort key in Amazon Redshift?

A sort key stores rows on each slice in a defined order. Range-restricted predicates on the leading sort column let zone maps skip blocks, and sorted data makes merge joins cheap. Compound sort keys prioritize columns left to right; interleaved sort keys weight up to eight columns equally for multi-dimensional filters, but they require `VACUUM REINDEX` and cost more to maintain.

```sql
-- Time-series fact: date_key first makes range scans prune aggressively
create table analytics.fact_events (
  event_ts   timestamp encode az64,
  event_date date      encode az64,
  user_id    bigint    encode az64,
  event_type varchar(32) encode zstd,
  payload    varchar(4096) encode zstd
)
diststyle even
compound sortkey (event_date, user_id);
```

Zone maps are the reason a sort key pays off. On each slice every column is stored in 1 MB blocks, and each block keeps min/max metadata. A predicate on the leading sort column lets the scan skip whole blocks, so touching fewer distinct blocks means less I/O. Compound is the default and its benefit follows the leading columns left to right. Sort key columns default to RAW encoding, which is worth knowing before you override them. A table has one sort key, and `SORTKEY AUTO` lets Redshift choose the columns for you.

```sql
-- Change the leading sort columns on an existing table
alter table analytics.fact_events alter sortkey (event_date, user_id);

-- Hand the choice to Redshift, or drop the sort key entirely
alter table analytics.fact_events alter sortkey auto;
alter table analytics.fact_events alter sortkey none;

-- Reclaim sort order without a full VACUUM
vacuum sort only analytics.fact_events to 75 percent;
```

Common mistake: reaching for an interleaved sort key when the filters are one range on a single date column. Interleaved weights several columns equally, costs more to maintain, and needs `VACUUM REINDEX`. You can alter an interleaved key to compound, never the reverse. It only earns its keep for low-cardinality filters on several columns at once.

## What is a distribution key in Amazon Redshift?

A distribution key is the column (or the `ALL`/`EVEN` strategy) that determines which slice stores each row. Choosing the join key of the largest fact-to-fact join colocates matching rows and removes network redistribution, which is usually the single biggest performance decision in a Redshift model.

```sql
-- Two facts that are frequently joined on customer_key share that distkey
create table analytics.fact_orders (
  order_id     bigint,
  customer_key integer,
  order_date   date,
  amount       numeric(12,2)
)
diststyle key distkey (customer_key)
sortkey (order_date);

create table analytics.fact_returns (
  return_id    bigint,
  customer_key integer,
  return_date  date,
  amount       numeric(12,2)
)
diststyle key distkey (customer_key)
sortkey (return_date);
```

Slice count drives the decision. A dc2.large node has 2 slices, an ra3.4xlarge has 12, and an ra3.16xlarge has 48. `DISTSTYLE EVEN` spreads rows round robin, `KEY` hashes one column, and `ALL` copies the whole table to every slice. `AUTO` starts a small table at ALL and moves a large one to AUTO(EVEN), or AUTO(KEY) on its join column. Check `skew_rows`, the ratio of the busiest to the quietest slice, in `SVV_TABLE_INFO` after a change.

```sql
-- Move a fact table onto the column it is joined on
alter table analytics.fact_orders alter diststyle key distkey (customer_key);

-- A small dimension is a fine candidate for a full copy on every slice
create table analytics.dim_product (
  product_key  integer,
  category     varchar(64),
  product_name varchar(256)
)
diststyle all
sortkey (product_key);
```

Common mistake: putting a high-cardinality column such as `order_id` in `DISTSTYLE KEY`. Hashing a unique id spreads rows evenly but does nothing for joins, so every join still redistributes. Pick the column you actually join on. The opposite error is `DISTSTYLE ALL` on a large fact table, which multiplies storage and load cost on every slice.

## Can you change the sort key or distribution key of an existing table in Redshift?

Yes in many cases. `ALTER TABLE ... ALTER SORTKEY` changes a compound sort key, and `ALTER TABLE ... ALTER DISTSTYLE` changes the distribution style; interleaved sort keys are the main exception, and the operation can be expensive because it reorganizes data. The zero-risk pattern is still to build a new table with the target design, copy the data, and rename.

```sql
-- In-place, supported for many conversions but potentially long-running
alter table analytics.fact_sales alter sortkey (date_key, store_key);
alter table analytics.fact_sales alter diststyle key distkey (product_key);

-- Rebuild pattern, which you can schedule and roll back
create table analytics.fact_sales_new (
  sale_id      bigint,
  date_key     integer,
  product_key  integer,
  store_key    integer,
  net_amount   numeric(12,2),
  quantity     integer
)
diststyle key distkey (product_key)
sortkey (date_key, store_key);

insert into analytics.fact_sales_new
select * from analytics.fact_sales;

analyze analytics.fact_sales_new;

alter table analytics.fact_sales rename to fact_sales_old;
alter table analytics.fact_sales_new rename to fact_sales;
```

`ALTER TABLE` and `VACUUM` cannot run concurrently on the same table, so schedule reorganizing changes in their own window instead of alongside a nightly vacuum. Interleaved sort keys are the one-way door: you can alter them to compound, never back. `AUTO` is also available for both diststyle and sortkey. `SVV_ALTER_TABLE_RECOMMENDATIONS` lists tables whose current design the system believes is costing you, which is a reasonable starting point for a review.

```sql
-- Inspect the design and its symptoms before and after a change
select database, schema, table, diststyle, sortkey1, sortkey_num,
       skew_rows, unsorted, stats_off
from svv_table_info
where schema = 'analytics';

-- Undo a hand-picked key and let Redshift decide again
alter table analytics.fact_sales alter diststyle auto;
alter table analytics.fact_sales alter sortkey auto;
```

Common mistake: firing a long `ALTER TABLE ... ALTER DISTSTYLE` while a `VACUUM` or a large `COPY` runs on the same table. The statements queue behind each other and the window blows out. For a big table the rebuild-and-rename pattern stays safer, because you can prepare and load the new table first and spend only a short transaction on the swap.

## What is a Redshift Spectrum?

Redshift Spectrum is the Redshift feature that queries data directly in Amazon S3 - Parquet, ORC, JSON, CSV, Avro, and Iceberg - using external tables in the AWS Glue Data Catalog. You pay per terabyte scanned and per Spectrum compute, and you avoid loading or transforming the data first.

Spectrum is not infrastructure you provision; it is a capability driven by external tables. `CREATE EXTERNAL SCHEMA` points Redshift at a Glue Data Catalog database, and `CREATE EXTERNAL TABLE` describes the S3 location and format. Scans run on a separate Spectrum fleet rather than your cluster's slices, so they consume little local disk. Billing is per terabyte scanned plus the Spectrum compute charge, which makes partition pruning a cost control and not just a speed trick.

```sql
-- Declare how big the external table is, so the planner can size it
alter table spectrum_lake.orders
set table properties ('numRows'='1000000');

-- Query S3 directly, no load step
select dt, count(*) as order_count, sum(amount) as revenue
from spectrum_lake.orders
where dt between '2026-09-01' and '2026-09-30'
group by dt
order by dt;
```

Common mistake: expecting `ANALYZE` to collect statistics for an external table. It does not. Spectrum table stats come from the `numRows` table property, supplied by you or by the Glue crawler. Missing or stale `numRows` makes the optimizer misestimate the scan and choose a poor join order, which shows up as an unexpectedly expensive query.

## How does Redshift Spectrum work?

The leader node compiles the query, then Spectrum compute nodes - a separate, elastically scaled fleet - read and filter the S3 objects. Predicate and partition projection is pushed down so only the needed columns, row groups, and partitions are read. The result is returned to the cluster, where it can be joined with local tables.

```sql
create external schema spectrum_lake
from data catalog
database 'acme_lake'
iam_role 'arn:aws:iam::123456789012:role/redshift-spectrum-role'
create external database if not exists;

create external table spectrum_lake.orders (
  order_id    varchar(64),
  customer_id varchar(64),
  order_date  date,
  amount      decimal(12,2)
)
partitioned by (dt varchar(10))
stored as parquet
location 's3://acme-lake/orders/';

alter table spectrum_lake.orders
add partition (dt = '2026-10-04')
location 's3://acme-lake/orders/dt=2026-10-04/';

select customer_id, sum(amount) as revenue
from spectrum_lake.orders
where dt = '2026-10-04'
group by customer_id;
```

Pushdown happens at three levels. Partition pruning drops whole S3 prefixes when the predicate matches a partition column, exactly like Hive. Column pruning reads only the referenced columns, but only for columnar formats such as Parquet and ORC; a CSV or JSON table must read the whole object. Row group metadata inside a Parquet file lets Spectrum skip blocks as well, so file size and row group size directly affect scan cost and therefore the bill.

```sql
-- Register one new partition after files land
alter table spectrum_lake.orders
add partition (dt = '2026-10-05')
location 's3://acme-lake/orders/dt=2026-10-05/';

-- Or reconcile every partition in one statement
msck repair table spectrum_lake.orders;
```

Common mistake: storing the S3 data as CSV and expecting Parquet-like behaviour. A text table cannot prune columns, so a query that touches two fields still reads and parses every byte. A second trap is filtering on a function of the partition column, such as `cast(dt as date) = ...`, which stops the planner from pruning partitions at all.

## Can you join tables from Redshift and Redshift Spectrum in a single query?

Yes. The optimizer decides what to push to Spectrum and what to run locally, and can broadcast a local dimension to the Spectrum scan or redistribute the S3 result. This is the standard pattern for a lakehouse with hot data in Redshift and cold history in S3.

```sql
-- Local dimension joined to an S3 fact table
select
  p.category,
  sum(s.amount) as revenue
from spectrum_lake.orders s
join analytics.dim_product p on s.product_id = p.product_id
where s.dt between '2026-09-01' and '2026-09-30'
group by p.category
order by revenue desc;
```

The performance lever is external table design: partition on the filter column, store Parquet, keep partitions reasonably large, and run `MSCK REPAIR` or add partitions when new data lands.

The optimizer picks a join strategy from the relative sizes. A small local dimension can be broadcast to the Spectrum scan, while a large S3 result may be redistributed or materialized on the cluster. The local table's distkey still governs the local side of the join. Because scanned bytes are billed, pushing the selective filter to the S3 side is the main cost lever, and a query that reads only one or two partitions is far cheaper than one that reads the month.

```sql
-- Filter the S3 side hard and keep the local side narrow
create table analytics.monthly_revenue as
select
  p.category,
  date_trunc('month', s.order_date) as month,
  sum(s.amount) as revenue
from spectrum_lake.orders s
join analytics.dim_product p
  on s.product_id = p.product_id
where s.dt between '2026-09-01' and '2026-09-30'
  and p.is_active = true
group by 1, 2;
```

Common mistake: joining an external table to a local table on mismatched types, such as `varchar` in S3 against `integer` in Redshift. The implicit cast can block pushdown and force a full scan plus conversion on the cluster. Define the external table columns to match, and always confirm the partition column appears directly in the `WHERE` clause.

## How can you load data into Amazon Redshift?

The main paths are `COPY` from S3 (or DynamoDB, EMR, or a remote host over SSH), `INSERT`/`INSERT ... SELECT` for small volumes, `CREATE TABLE AS` from an external table, streaming through Kinesis Data Firehose, zero-ETL integrations from Aurora/RDS, and Redshift Data API or JDBC/ODBC from an application.

```sql
-- COPY from a manifest of columnar files
copy analytics.fact_sales
from 's3://acme-lake/manifests/fact_sales.manifest'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
manifest;

-- ELT from the lake: transform while loading from Spectrum
create table analytics.fact_sales_2026 as
select order_id, customer_id, order_date, amount
from spectrum_lake.orders
where dt between '2026-01-01' and '2026-12-31';
```

`COPY` is the workhorse and has a wide option surface. Real options include `IGNOREHEADER n`, `TRUNCATECOLUMNS`, `ACCEPTINVCHARS`, `GZIP`, `ENCRYPTED`, `EXPLICIT_IDS`, `REGION`, and `DELIMITER`. Use them to absorb messy source files instead of pre-cleaning in S3. Small volumes can go in with `INSERT ... SELECT` or `CREATE TABLE AS`, streams land through Kinesis Data Firehose, and zero-ETL integrations copy from Aurora or RDS without a pipeline you maintain.

```sql
copy analytics.dim_product
from 's3://acme-lake/staging/dim_product/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
csv
ignoreheader 1
region 'us-east-1'
gzip
truncatecolumns
acceptinvchars
maxerror 10;
```

Common mistake: staging one uncompressed CSV file and then blaming the cluster for a slow load. One file is one unit of work, so a single slice reads it. Split the input, compress it, and prefer Parquet. The other common error is looping `INSERT` one row at a time; each statement costs a transaction and a compile.

## What is the recommended way to load large amounts of data into Redshift?

Stage compressed, columnar files in S3, split into many files, and run a single parallel `COPY` with an IAM role, a manifest, `COMPUPDATE` and `STATUPDATE` on, and `MAXERROR` tuned to your tolerance. Then `ANALYZE` the table and `VACUUM` if the load caused churn.

```bash
# Example: convert CSV to Parquet and upload many files, then COPY in parallel
aws s3 cp ./curated/fact_sales/ s3://acme-lake/curated/fact_sales/ --recursive --exclude "*" --include "*.parquet"
```

```sql
copy analytics.fact_sales
from 's3://acme-lake/curated/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
compupdate on
statupdate on
maxerror 0;

analyze analytics.fact_sales;
```

Beware the anti-patterns: one giant file, CSV with a single uncompressible stream, row-by-row inserts, and loading into a table whose sort key forces a full resort on every batch.

Aim for roughly one file per slice, or a small multiple of the slice count. A dc2.large node has 2 slices, an ra3.4xlarge has 12, and an ra3.16xlarge has 48. Files in the few-hundred-MB range keep S3 request overhead low while still giving every slice work. Run one `COPY` per table, then `ANALYZE` to refresh stats; `ANALYZE` is incremental, so running it after each load is cheap and keeps plans honest.

```sql
copy analytics.fact_sales
from 's3://acme-lake/manifests/fact_sales.manifest'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
manifest
compupdate preset
statupdate on
maxerror 0;

analyze analytics.fact_sales;
vacuum sort only analytics.fact_sales to 75 percent;
```

Common mistake: leaving `COMPUPDATE ON` for every small incremental load. Each one triggers a full-table compression analysis, which is wasted work once the table is already encoded. Use `COMPUPDATE PRESET` for routine loads and reserve `ON` for the first load or after a column change. The mirror-image mistake is skipping `ANALYZE`, which leaves stale stats behind a perfectly sorted table.

## Can you load data into Redshift from multiple files in parallel?

Yes, and that is the point of the design: each file can be loaded by a slice, so throughput scales with the number of files up to the slice count. Use a prefix in `COPY`, or a manifest file for explicit control, and keep files roughly the same size so no slice finishes long before the others.

```sql
-- Prefix load: every object under the prefix is read in parallel
copy analytics.fact_sales
from 's3://acme-lake/curated/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet;

-- Manifest load: explicit list, which avoids picking up stray objects
copy analytics.fact_sales
from 's3://acme-lake/manifests/fact_sales.manifest'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
manifest
format as parquet;
```

If the file count is much larger than the slice count, Redshift still parallelizes well, but thousands of tiny files add S3 request overhead; target files of a few hundred MB.

Parallelism is bounded by the slice count, so the goal is a multiple of slices, not the largest possible number of files. Thousands of tiny objects add S3 request and file-open overhead per slice, and that overhead can dominate a small load. Keep the files close to the same size, because the load finishes when the slowest slice finishes, and one outsized file becomes the tail of the whole job.

```sql
-- Produce a matching set of output files straight from the cluster
unload ('select * from analytics.fact_sales where order_date >= ''2026-01-01''')
to 's3://acme-lake/curated/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
format as parquet
parallel on
manifest
maxfilesize 256 mb
allowoverwrite;
```

Common mistake: pointing a prefix `COPY` at a folder that also contains a manifest, a staging marker, or files from a previous run. A prefix load reads every object under it, which can duplicate rows or fail outright. Use an explicit manifest whenever the folder is not exclusively load files, and check the file count before you run it.

## How can you monitor the performance of Amazon Redshift?

Use CloudWatch metrics for cluster-level health (CPU, disk space, queue length, commit latency), the Redshift console performance tab for query-level views, and the system tables for the details: `STL_QUERY`, `STL_QUERYTEXT`, `SVL_QUERY_SUMMARY`, `SVL_QUERY_REPORT`, `STL_WLM_QUERY`, and `SVV_TABLE_INFO` for table design. Redshift Advisor and `SVV_ALTER_TABLE_RECOMMENDATIONS` suggest design changes.

```sql
-- Longest-running queries in the last day
select
  q.query,
  trim(t.text) as sql_text,
  q.starttime,
  datediff(second, q.starttime, q.endtime) as elapsed_s,
  q.aborted
from stl_query q
join stl_querytext t
  on q.query = t.query and t.sequence = 0
where q.starttime >= dateadd(day, -1, getdate())
order by elapsed_s desc
limit 20;
```

```sql
-- Tables that need attention: high skew, stale stats, or high unsorted percentage
select
  schema, "table", diststyle, sortkey1,
  nvl(unsorted, 0) as unsorted_pct,
  nvl(stats_off, 0) as stats_off_pct,
  nvl(skew_rows, 0) as skew_rows,
  tbl_rows
from svv_table_info
where nvl(unsorted, 0) > 20 or nvl(stats_off, 0) > 10
order by unsorted_pct desc;
```

STL tables such as `STL_QUERY` and `STL_QUERYTEXT` hold one row per query step or segment and are trimmed periodically, so they are good for live debugging but not for a permanent baseline. SVL views are the durable user-facing layer. `SVL_QUERY_SUMMARY` reports one row per step with `avgtime` and `maxtime` in microseconds, plus `rows`, `bytes`, `workmem` and `is_diskbased`. A step with `is_diskbased = 't'` spilled to disk, which usually means the plan needs more memory or a different sort or distribution key. `SVL_QUERY_METRICS_SUMMARY` rolls metrics up per query. `SYS_QUERY_HISTORY` keeps recent completed queries, and `STV_WLM_QUERY_STATE` shows what is running right now.

```sql
-- Find the steps that spilled to disk; these are the real tuning targets
select query, stm, seg, step, avgtime, maxtime, rows, bytes, workmem, is_diskbased
from svl_query_summary
where is_diskbased = 't'
  and query > pg_last_query_id() - 100
order by workmem desc
limit 20;
```

Common mistake: judging cluster health from elapsed time alone. A query can be slow because one step spilled to disk, because a large table is skewed across slices, or because the queue made it wait before it started. Check `SVL_QUERY_SUMMARY` for the spill, `SVV_TABLE_INFO.skew_rows` for the imbalance, and `STL_WLM_QUERY` for queue time. Also remember `SVV_TABLE_INFO` is superuser-only and hides empty tables the caller does not own, so a non-superuser dashboard can silently show nothing.

## What is the purpose of the VACUUM command in Amazon Redshift?

`VACUUM` reclaims space left by deletes and re-sorts rows that became unsorted after loads, which restores zone-map effectiveness and merge-join order. It has variants: `VACUUM FULL` (sort and reclaim), `VACUUM SORT ONLY`, `VACUUM DELETE ONLY`, `VACUUM REINDEX` (interleaved sort keys), and `VACUUM ... TO threshold PERCENT`.

```sql
-- Reclaim space and resort the whole table
vacuum full analytics.fact_sales;

-- Only resort, and stop when the table is 95 percent sorted
vacuum sort only analytics.fact_sales to 95 percent;

-- Rebuild interleaved sort keys (only needed for INTERLEAVED tables)
vacuum reindex analytics.fact_events;
```

VACUUM is I/O heavy and takes an exclusive-ish lock during the merge; run it in a maintenance window, target only tables that need it, and prefer `VACUUM SORT ONLY` when space is not the problem.

VACUUM cannot run inside a transaction block, so wrapping it in an explicit `BEGIN` and `COMMIT` fails. It also takes a strong lock while it merges: `ALTER TABLE` and `VACUUM` cannot run on the same table at the same time. The default target is 95 percent sorted, and `VACUUM ... TO 75 PERCENT` relaxes that so it finishes sooner on very large tables. Sort key columns default to RAW encoding, so a re-sort really does rewrite them. `VACUUM BOOST` is a variant for large tables that uses the BOOST feature to speed the sort.

```sql
-- Inspect the benefit before paying the I/O cost
select "table", unsorted, vacuum_sort_benefit, tbl_rows, size
from svv_table_info
where schema = 'analytics'
order by vacuum_sort_benefit desc
limit 10;

vacuum boost analytics.fact_sales;
```

Common mistake: running `VACUUM REINDEX` on a table with a compound sort key. REINDEX only applies to interleaved sort keys, so it does nothing useful there. Another common mistake is a blanket nightly VACUUM on every table: it is I/O heavy, and most tables never need it. Pick the few that show a real need through `SVV_TABLE_INFO.unsorted` and `vacuum_sort_benefit`.

## What is the difference between VACUUM and ANALYZE commands in Redshift?

| Aspect | VACUUM | ANALYZE |
|---|---|---|
| Purpose | Reclaim space, re-sort rows, rebuild interleaved keys | Refresh table and column statistics |
| Affects | Physical layout and zone maps | Optimizer cardinality estimates |
| Cost | High, I/O heavy, can block | Moderate, samples rows |
| Frequency | After heavy delete/update or out-of-order loads | After significant data changes |
| Risk if skipped | Bloated storage, poor pruning, slow merges | Bad join order and plan choices |

```sql
vacuum sort only analytics.fact_sales to 95 percent;
analyze analytics.fact_sales;
```

They solve different problems: a table can be perfectly sorted and still get a terrible plan from stale statistics, and a table with fresh statistics can still scan bloated, unsorted blocks.

ANALYZE is incremental and samples rows, so it is cheap enough to run after most loads; it refreshes only the statistics that look stale. `SVV_TABLE_INFO.stats_off` shows 0 for current statistics and 100 for fully out-of-date statistics. `ANALYZE COMPRESSION` is a different tool: it only reads the table and reports recommended encodings, it does not collect statistics, and it does not change data. Note the lock difference too: VACUUM cannot run inside a transaction block, while ANALYZE can.

```sql
-- Which tables have stale stats, and what encodings does Redshift suggest?
select schema, "table", stats_off, tbl_rows
from svv_table_info
where stats_off > 10
order by stats_off desc;

analyze analytics.fact_sales;
analyze compression analytics.fact_sales;
```

Common mistake: running `ANALYZE COMPRESSION` and assuming the recommended encodings were applied. It only produces a report; you still apply the change with `ALTER COLUMN ... ENCODE` or by letting COPY run with `COMPUPDATE`. The other common mistake is treating ANALYZE as a substitute for VACUUM. Fresh statistics fix join order and pruning choices, but they do not reclaim space or re-sort unsorted rows.

## How can you secure data in Amazon Redshift?

+ IAM controls who can call the Redshift APIs; database users/roles control query access.
+ Encryption at rest with KMS or CloudHSM; TLS for data in transit.
+ VPC, security groups, private subnets, and PrivateLink for network isolation.
+ Column-level access control and row-level security for least privilege.
+ IAM authentication and Secrets Manager for credential handling.
+ Audit logging (connection, user activity, user) and CloudTrail for API history.
+ Lake Formation and Spectrum permissions for the S3 side.

```sql
-- Row-level security scoped to the current user's region
create role emea_analyst;
grant usage on schema analytics to role emea_analyst;
grant select on analytics.fact_sales to role emea_analyst;

create rls policy emea_only
  with (region varchar(32))
  using (region = current_setting('app.region'));

attach rls policy emea_only
  on analytics.fact_sales to role emea_analyst;

-- Column-level grants keep PII out of reach
grant select (order_id, order_date, amount) on analytics.fact_orders to role emea_analyst;
```

Row-level security is enforced by a policy that carries a column definition in `WITH` and a predicate in `USING`. The predicate can call `current_user` or read a lookup table instead of relying on a session setting. Column-level grants only restrict access while the role does not also hold table-level SELECT: a table-wide `GRANT SELECT` overrides the column list. Audit logging writes connection, user activity and user logs to S3, separately from CloudTrail, which records the API history.

```sql
-- A mapping table lets one policy serve many regions
create table analytics.user_region (username varchar(64), region varchar(32));

create rls policy region_lookup
  with (region varchar(32))
  using (region in (
    select r.region
    from analytics.user_region r
    where r.username = current_user
  ));

alter table analytics.fact_sales row level security on;
```

Common mistake: granting SELECT on the whole table for convenience and then adding column-level grants expecting them to hide PII. In Redshift the table-level grant wins, so the column list restricts nothing. Audit that no role holds broad table SELECT, then grant only the columns each role actually needs.

## Can you encrypt data in Amazon Redshift?

Yes. Redshift encrypts data at rest with AES-256 using either AWS KMS keys or CloudHSM keys (with automatic key rotation), and supports encryption in transit with SSL/TLS. You can also encrypt the S3 side of a `COPY`/`UNLOAD` with SSE-S3, SSE-KMS, or client-side keys, and use `COPY ... ENCRYPTED` for client-side encrypted load files.

```bash
# Enable KMS encryption on an existing cluster (requires a reboot)
aws redshift modify-cluster \
  --cluster-identifier analytics-cluster \
  --encrypted \
  --kms-key-id alias/redshift-prod
```

```sql
-- Client-side encrypted load: the file is encrypted before it reaches S3
copy analytics.fact_orders
from 's3://acme-secure/orders/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
master_symmetric_key '${MASTER_KEY}'
encrypted;
```

Encryption is chosen when the cluster is created and cannot be switched on in place without a reboot. KMS keys can rotate automatically, while CloudHSM is the option when you must hold the key material yourself. In transit, Redshift uses SSL/TLS and clients should not disable it. On the S3 side of COPY and UNLOAD you can choose SSE-S3, SSE-KMS, or client-side encryption, and the cluster and its snapshots inherit the same at-rest setting.

```sql
-- Client-side encrypted export: files are encrypted before they leave the cluster
unload ('select order_id, order_date, amount from analytics.fact_orders')
to 's3://acme-secure/unload/orders/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
master_symmetric_key '${MASTER_KEY}'
encrypted
format as parquet;
```

Common mistake: assuming a new cluster is encrypted by default and only discovering otherwise during an audit. A cluster created without encryption stays unencrypted, and turning encryption on later requires `modify-cluster` plus a reboot. Plan for that reboot, or create the cluster encrypted from the start.

## What is the maximum size of a single Amazon Redshift cluster?

Provisioned RA3 clusters support up to 128 compute nodes per cluster as documented at the time of writing, and Redshift Managed Storage scales far beyond the local disk of older DS2/DC2 nodes. Redshift Serverless removes the node-count question entirely and scales capacity in RPUs. AWS raises these limits periodically, so quote the architecture ("RA3 separates storage, so capacity is not tied to node count") rather than a hardcoded ceiling.

```bash
# Current limits for a workgroup/cluster in your account
aws redshift describe-clusters --cluster-identifier analytics-cluster \
  --query 'Clusters[0].[NumberOfNodes,NodeType,ClusterStatus]'
```

Capacity is easier to reason about per slice than per cluster. A slice owns a CPU, memory and a disk share, and a table's rows live on slices. Slice counts per node are fixed by node type: dc2.large 2, dc2.8xlarge 16, ra3.xlplus 4, ra3.4xlarge 12, ra3.16xlarge 48. On RA3, Redshift Managed Storage holds the data and the local SSD is only a cache, so the useful storage limit is not the sum of local disks. Redshift Serverless replaces node counts with RPUs.

Common mistake: quoting one hardcoded ceiling as if it were permanent. AWS raises node and storage limits regularly, and the answer changes with the platform: DC2, RA3, or Serverless. Explain the model instead. On RA3 storage is decoupled from compute, on Serverless capacity is measured in RPUs, and both can grow without adding nodes.

## How can you take backups of Amazon Redshift clusters?

Redshift takes automated snapshots on a retention schedule (1 to 35 days, default 1) and supports manual snapshots you keep until deleted. RA3 and Serverless snapshots can be shared across accounts, copied to another region, and restored into a new cluster or namespace.

```bash
# Automated snapshot retention
aws redshift modify-cluster \
  --cluster-identifier analytics-cluster \
  --automated-snapshot-retention-period 14

# On-demand snapshot before a risky migration
aws redshift create-cluster-snapshot \
  --cluster-identifier analytics-cluster \
  --snapshot-identifier analytics-pre-migration
```

Automated snapshots are created on a schedule and when enough data changes, then kept for the retention period. Manual snapshots are kept until you delete them, which makes them the right tool before a migration or another risky change. The retention that Redshift accepts is 1 to 35 days and defaults to 1. RA3 and Serverless snapshots can be shared across accounts, copied to another region, and restored into a new cluster or namespace.

Common mistake: deleting a cluster and assuming the automated snapshots survive. Deleting a cluster removes its automated snapshots unless you take a final snapshot first; manual snapshots are what remain. Take a manual snapshot before any risky change, and do not run production on the minimum retention.

## What is the difference between a Redshift snapshot and a Redshift cluster backup?

In Redshift a "backup" is delivered as snapshots: a point-in-time, cluster-level copy stored in S3, including data and cluster metadata (users, schemas, configuration). The distinction people usually mean is automated snapshots versus manual snapshots, or cluster snapshots versus a cross-region copy or a data-share. There is no separate continuous-backup product to configure.

```bash
aws redshift describe-cluster-snapshots \
  --cluster-identifier analytics-cluster \
  --query 'Snapshots[].[SnapshotIdentifier,SnapshotType,SnapshotCreateTime,Status]'
```

Redshift has no separate continuous backup product to enable. Snapshots are the backup: a point-in-time copy of the whole cluster, data plus catalog metadata such as users and schemas. The distinctions that matter in practice are automated versus manual, cluster snapshot versus cross-region copy, and snapshot versus data share. A data share gives live read access to data rather than a restorable copy, so it is not a backup.

Common mistake: treating a snapshot as a table-level backup. A snapshot restores a whole cluster or namespace, so recovering one table means restoring beside production and copying the rows back. A data share is also not a backup, because it follows the source data and disappears when access is revoked.

## Can you restore a Redshift snapshot to a different AWS region?

Yes. Redshift supports cross-region snapshot copy: enable it on the source cluster with a destination region and retention, and Redshift copies new automated and manual snapshots (encrypted with a KMS key in the destination) so you can restore or use them there. You can also copy a single snapshot on demand.

```bash
# Enable ongoing cross-region snapshot copy
aws redshift enable-snapshot-copy \
  --cluster-identifier analytics-cluster \
  --destination-region us-west-2 \
  --retention-period 7

# Copy one snapshot now
aws redshift copy-cluster-snapshot \
  --source-snapshot-identifier analytics-pre-migration \
  --target-snapshot-identifier analytics-pre-migration-west \
  --source-region us-east-1 \
  --target-region us-west-2 \
  --target-snapshot-kms-key-id alias/redshift-dr
```

Snapshot copy runs in the background. Each new automated or manual snapshot is copied after it is created, so the destination region lags the source by minutes. The copied snapshot keeps its own retention period, separate from the source cluster. Data transfer and destination storage are billed, so retention has a cost. To bring the data back, restore the copy as a new cluster in the destination region:
```bash
aws redshift restore-from-cluster-snapshot \
  --cluster-identifier analytics-dr \
  --snapshot-identifier analytics-pre-migration-west \
  --region us-west-2
```
Restoring is not an in-place operation. It creates a fresh cluster. Cluster settings are carried over, but the destination region must already have the IAM roles, subnet groups, and parameter groups the cluster needs.

Common mistake: expecting the snapshot to be usable in the destination region the moment it is created, or restoring with a KMS key the destination account cannot use. Cross-region copy re-encrypts with a key in the destination region, so the source cluster role must be able to use that key. If the key policy blocks it, the copy fails silently and the snapshot never appears in the destination region. Check the snapshot copy status before relying on it for disaster recovery.

## How can you resize a Redshift cluster?

Use elastic resize for a fast change to a supported target configuration (adds or removes nodes or changes some node types with a short reboot), classic resize for arbitrary changes (which migrates data and takes longer), or move to Redshift Serverless to avoid node management. RA3 also lets you scale compute and storage independently.

```bash
# Elastic resize: double the nodes
aws redshift modify-cluster \
  --cluster-identifier analytics-cluster \
  --number-of-nodes 8 \
  --node-type ra3.4xlarge \
  --elastic-resize

# Classic resize to a different node family
aws redshift resize-cluster \
  --cluster-identifier analytics-cluster \
  --cluster-type multi-node \
  --number-of-nodes 16 \
  --node-type ra3.16xlarge
```

Elastic resize is fast because Redshift remaps data onto the new slices with only a short reboot. It supports a limited set of target configurations, usually doubling or halving the node count within the same family. A target it cannot handle is rejected or falls back to a classic resize. Classic resize provisions a new cluster and copies every table, so it takes hours on a large warehouse and needs space for the copy. After any resize, run ANALYZE. The optimizer statistics were gathered for the old slice count, and rows have moved to different slices, so zone maps and cardinality estimates are stale.

Common mistake: resizing during a busy window without taking a fresh snapshot first. Elastic resize reboots the cluster, so in-flight queries fail and clients must retry. Classic resize holds the cluster in a resizing state where write queries cannot run. Schedule the change in a maintenance window, take a manual snapshot, and re-run ANALYZE and VACUUM afterwards so the new slice layout is efficient.

## What is the difference between dense compute and dense storage node types in Redshift?

| Family | Storage | Best for | Notes |
|---|---|---|---|
| DC2 (dense compute) | Local SSD | High-performance, compute-heavy, smaller data | Fast local I/O, storage tied to nodes |
| DS2 (dense storage) | Local HDD | Large data at low cost, sequential scans | Legacy; being superseded by RA3 |
| RA3 | Managed storage (SSD cache + S3) | Most production warehouses | Scale compute and storage independently |

```text
RA3: compute nodes hold a local SSD cache; the authoritative data lives in
Redshift Managed Storage, so growing storage does not require adding nodes.
```

For new deployments, RA3 or Serverless is the modern default: DS2 is legacy and RA3 removes the "add nodes to get disk" trade-off.

Storage is tied to the node on DC2 and DS2, so capacity grows only by adding nodes. DC2 uses local SSD for fast I/O. DS2 uses local HDD for cheap capacity and sequential scans. RA3 keeps only a local SSD cache, while Redshift Managed Storage in S3 holds the authoritative data, which separates compute from storage. Slice counts differ by node, and slices are the unit of parallel work: dc2.large has 2, dc2.8xlarge has 16, ra3.xlplus has 4, ra3.4xlarge has 12, and ra3.16xlarge has 48. More slices mean more parallel scan and load throughput.

```sql
-- Node and slice layout of the current cluster
select count(*) as slice_count from stv_slices;

select node, slice
from stv_slices
order by node, slice;
```

Common mistake: sizing parallelism by node count instead of slice count. A dc2.8xlarge is one node but has 16 slices, and distribution spreads rows across slices, not nodes. Splitting a load into one file per node under-parallelizes it by the slice factor. Split input files to a multiple of the slice count for COPY and UNLOAD.

## What is the maximum number of tables you can create in an Amazon Redshift database?

The documented limit varies by node type - 9,900 tables per database on smaller nodes (for example `ra3.large`) and 200,000 on `ra3.4xlarge`/`ra3.16xlarge`, while Redshift Serverless allows 200,000. System tables are excluded, and AWS has raised these limits over time, so verify the current quota. In practice the limit is never the constraint - super-wide schemas with thousands of tiny tables are, because each adds planning and maintenance overhead.

```sql
-- How many user tables does this database actually have?
select count(*) as user_tables
from svv_tables
where table_schema not in ('pg_catalog', 'information_schema');
```

The limit is per database, not per cluster, so each database carries its own table count. Every table also creates storage and catalog entries on every slice, so thousands of tiny tables add real overhead even when rows are small. Planning and DDL operations get slower because the leader node must walk a larger catalog. This is why a design that creates one table per day or per customer ages badly. If you need to know where the pressure is, count tables by schema:
```sql
select table_schema, count(*) as table_count
from svv_tables
where table_schema not in ('pg_catalog', 'information_schema')
group by table_schema
order by table_count desc;
```

Common mistake: assuming the table limit is the real scaling risk. A schema with thousands of tiny tables stays far below the documented maximum but still slows planning and maintenance. Prefer one larger table with a partition or date column over many per-day tables, and archive cold data to Spectrum instead of splitting it into more relations.

## How can you optimize storage in Amazon Redshift?

+ Choose the right compression encoding per column (or let `COPY` pick with `COMPUPDATE`).
+ Set sort and distribution keys so blocks hold related rows and zone maps work.
+ Run `VACUUM` to reclaim deleted space and re-sort.
+ Drop or archive cold tables and use Spectrum for rarely queried history.
+ On RA3, managed storage grows automatically; on DS2/DC2, watch per-node disk usage.

```sql
-- Where is storage going, and how much is unsorted or skewed?
select
  schema, "table",
  diststyle, sortkey1,
  tbl_rows,
  size as size_mb,
  nvl(unsorted, 0) as unsorted_pct,
  nvl(skew_rows, 0) as skew_rows
from svv_table_info
order by size_mb desc
limit 20;
```

VACUUM does the space work. A full VACUUM re-sorts rows and reclaims space left by deletes and updates. `VACUUM SORT ONLY` only re-sorts, and `VACUUM DELETE ONLY` only reclaims. The `TO n PERCENT` clause stops early once the table is sorted enough. VACUUM cannot run inside a transaction block.
```sql
-- Reclaim space and re-sort a busy table
vacuum full analytics.fact_sales to 90 percent;

-- See what encodings the data would prefer
analyze compression analytics.fact_sales;
```

Encoding is the other lever. `ENCODE AUTO` lets Redshift pick and change per-column encodings, and `COPY ... COMPUPDATE ON` updates them during a load. Sort key columns default to RAW encoding, which is larger but fast to scan. On RA3, managed storage grows on its own, so the win from VACUUM and encoding is scan speed and cost, not freeing a fixed disk. On DC2 and DS2, disk space is fixed per node, so deleted rows and poor encodings can genuinely fill the cluster.

Common mistake: expecting VACUUM to fix an interleaved sort key. Vacuuming an interleaved table does not restore the sort benefit. Interleaved tables need `VACUUM REINDEX`, which rebuilds the interleaved index and is expensive. A better route is often to alter the table to a compound sort key, because interleaved can be changed to compound but not the reverse.

## What is the maximum size of a single table in Amazon Redshift?

A single table is bounded by the cluster's usable storage rather than by an internal per-table cap. On RA3 the table lives in Redshift Managed Storage and can grow into petabytes across the cluster, while DS2/DC2 tables are bounded by local disk and are usually kept smaller through archival. Treat any specific number as version-dependent and verify the current service quotas.

```sql
-- Largest tables in bytes
select schema, "table", size as size_mb, tbl_rows
from svv_table_info
order by size_mb desc
limit 5;
```

Rows of one table are spread across every slice, so a table's ceiling is the cluster's usable capacity, not a per-table number. On RA3 that capacity is Redshift Managed Storage, which scales as data grows. On DC2 and DS2 it is the sum of local disks, which is why those clusters keep large tables smaller and archive cold history. Watch how much of each table's allocated blocks are actually used. `pct_used` near 100 means the table is packed, while low values point at wasted space.
```sql
select schema, "table", size as size_mb, pct_used, tbl_rows, unsorted
from svv_table_info
where pct_used is not null
order by size_mb desc
limit 5;
```

Common mistake: assuming a huge table will hit a row-count limit and fail cleanly. It usually fails on disk capacity, slice skew, or a VACUUM that never finishes. One oversized slice can exhaust local space long before the cluster looks full. Check `skew_rows` and `skew_sortkey1` before blaming size, and note that `SVV_TABLE_INFO` hides empty tables unless the caller owns them.

## What is the purpose of the UNLOAD command in Redshift?

`UNLOAD` runs a query and writes the result to Amazon S3 in parallel across slices, producing multiple files. It is the standard way to export curated data for the lake, hand data to another system, or archive a partition before dropping it. `UNLOAD` writes Parquet, ORC, or delimited text with compression and partitioning.

```sql
unload ('select * from analytics.fact_sales where date_key between 20260101 and 20260131')
to 's3://acme-export/fact_sales/2026-01/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
format as parquet
partition by (date_key)
allowoverwrite
manifest;
```

Note that `UNLOAD` is server-side: the data never flows through the client, and it parallelizes automatically. Use `PARALLEL OFF` only when a single ordered file is genuinely required.

UNLOAD writes the result of a query to S3 from the compute nodes, so it never streams through the client. With `PARALLEL ON` it writes one file per slice, which gives the best throughput and lets a downstream engine read the files in parallel. `MAXFILESIZE` caps file size, and each run can emit a manifest that lists the parts. `ALLOWOVERWRITE` lets a rerun replace existing objects in the prefix.
```sql
unload ('select date_key, product_key, sum(amount) as revenue
           from analytics.fact_sales
          where date_key >= 20260101
          group by date_key, product_key')
to 's3://acme-export/product-revenue/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
format as csv
header
parallel on
maxfilesize 96 mb
manifest
allowoverwrite;
```

Common mistake: using `PARALLEL OFF` just to get a single file, then reading it back with a parallel engine. The leader node writes that one file serially, so a large export becomes slow, and the single file is a single unit of work for whatever reads it next. Keep parallel output and match the file count to the slice count. Also add `ALLOWOVERWRITE` when a job may rerun, since UNLOAD fails if the prefix already holds objects.

## How can you manage Redshift query concurrency?

Three tools work together: WLM queues (automatic or manual) allocate memory and slots to query classes, short query acceleration (SQA) routes short queries ahead of long ones, and concurrency scaling adds transient clusters for read-heavy bursts. Query monitoring rules (QMR) can abort, log, or hop a query that exceeds thresholds like runtime or returned rows.

```bash
# Automatic WLM with a query monitoring rule that aborts runaway queries
aws redshift modify-cluster \
  --cluster-identifier analytics-cluster \
  --wlm-json-configuration '[{"name":"default","auto_wlm":true,"query_concurrency_scaling":"auto","rules":[{"rule_name":"abort_long","predicate":[{"metric_name":"query_execution_time","operator":">","value":3600}],"action":"abort"}]}]'
```

```sql
-- Which queue waited the longest in the last day?
select service_class, count(*) as queries, avg(total_queue_time) as avg_queue_us
from stl_wlm_query
where queue_start_time >= dateadd(day, -1, getdate())
group by service_class
order by avg_queue_us desc;
```

Automatic WLM decides queue memory and concurrency on demand. Manual WLM fixes slots and a memory percentage per queue, which is predictable but can strand memory when queues are idle. Query monitoring rules act on a running query: log it, hop it to another queue, or abort it. Concurrency scaling adds transient clusters for read-heavy bursts, and queries that ran on one show `compute_type = 'primary-scale'` in `SYS_QUERY_HISTORY`.
```sql
-- Did queries overflow onto concurrency scaling clusters?
select compute_type, count(*) as queries
from sys_query_history
where start_time >= dateadd(day, -7, getdate())
group by compute_type
order by queries desc;
```

Short query acceleration gives short queries a dedicated express path so they do not sit behind long scans. It is a queue setting, and the cluster predicts which queries are short enough to use it. Query monitoring rules cover runtime, returned rows, and other metrics. Concurrency scaling only helps read queries. A write-heavy workload, such as a large COPY or a long-running UPDATE, waits for its slot instead of scaling out.

Common mistake: looking for a SQL command to change WLM. There is no `ALTER WORKLOAD MANAGEMENT` statement. WLM is cluster configuration, set through the console, the `wlm_json_configuration` parameter group, or the Redshift API. Configure it there and inspect the effect through the WLM system views.

## What is the purpose of the ANALYZE command in Redshift?

`ANALYZE` collects table and column statistics - row counts, distinct values, null fractions, and block-level range information - that the optimizer uses to estimate cardinality and choose join order, join type, and distribution method. Without fresh statistics the optimizer guesses, and a guessed plan on a large table is usually a slow one.

```sql
-- Analyze one table
analyze analytics.fact_sales;

-- Only re-analyze columns used in predicates or joins (cheaper on wide tables)
analyze analytics.fact_sales predicate columns;

-- Analyze every table in the connected database after a big load
analyze;
```

Run `ANALYZE` after loads that change a meaningful fraction of rows, after large deletes, and before benchmarking a query. `COPY ... STATUPDATE ON` updates statistics as part of the load.

`ANALYZE` is incremental. It only gathers statistics for tables and columns that changed since the last run, which is why repeating it after a small load is cheap. It also runs automatically in the background and during `COPY` when STATUPDATE is on. Statistics go stale after large deletes as well as inserts, so check how far out of date they are before blaming the query.
```sql
-- 0 means current, 100 means fully out of date
select schema, "table", stats_off, tbl_rows
from svv_table_info
where stats_off > 10
order by stats_off desc, tbl_rows desc;
```

`ANALYZE COMPRESSION` is a different command. It reads the table, compares encodings, and returns recommendations. It changes no data and collects no optimizer statistics, so it is not a substitute for `ANALYZE`. Because it scans data, it can be slow on a large table. Run it when you suspect encoding choices are making scans read more blocks than necessary.

Common mistake: confusing `ANALYZE` with `ANALYZE COMPRESSION`. The first gathers the row counts, distinct values, and range metadata the optimizer needs. The second only reports encoding recommendations and leaves the table untouched. Running `ANALYZE COMPRESSION` and expecting better plans, or never running `ANALYZE` after a big delete, both produce slow queries that look like a cluster problem.

## Can you perform cross-database queries in Amazon Redshift?

Yes. On RG and RA3 provisioned clusters and on Redshift Serverless you can query across databases in the same cluster using three-part names (`database.schema.table`). Cross-database queries work on local databases and on databases accessed through data sharing; they do not extend to a different cluster's local databases unless those are shared.

```sql
-- Query a table in another database on the same cluster
select c.customer_id, c.email, sum(s.amount) as revenue
from sales_db.analytics.fact_sales s
join crm_db.public.customers c on s.customer_id = c.customer_id
where s.order_date >= date '2026-01-01'
group by c.customer_id, c.email
order by revenue desc
limit 100;
```

Requirements: RG or RA3 provisioned nodes (or Serverless), the databases on the same cluster, and appropriate privileges on the objects in the other database. DS2/DC2 clusters cannot do it.

Cross-database queries are resolved inside a single cluster. The leader node holds catalog metadata for every local database, so a three-part name is just another namespace to the planner. The rows still sit on the cluster's slices, which means a join across databases redistributes or broadcasts exactly like a join inside one database. Data sharing is the bridge that makes tables from another cluster appear as a database here, which is why a three-part name can reach them; there is no direct path to a foreign cluster's local databases. The feature is available on RA3 and RG provisioned clusters and on Redshift Serverless.

Real SQL, including views and grants, can span databases:
```sql
create view analytics.v_customer_revenue as
select c.customer_id, c.email, sum(s.amount) as revenue
from sales_db.analytics.fact_sales s
join crm_db.public.customers c
  on s.customer_id = c.customer_id
group by c.customer_id, c.email;

grant select on analytics.v_customer_revenue to group reporting;
```

Common mistake: assuming a three-part name reaches any cluster you can connect to. It reaches only databases in the same cluster, plus databases shared in through data sharing, and it is not supported at all on DS2 or DC2. The second trap is privileges: being able to connect is not the same as having SELECT on crm_db.public.customers, so the object rights must be granted in the source database too.

## What is the difference between a leader node and a compute node in Amazon Redshift?

The leader node is the query planner and coordinator: it parses SQL, optimizes the plan, distributes work, and merges results. It also stores metadata but no user data. Compute nodes store the data in slices and execute the query fragments; each node reports results back to the leader.

```text
client -> leader node (plan, coordinate) -> compute nodes/slices (scan, join, aggregate) -> leader -> client
```

You scale compute for query throughput and storage for data volume; on RA3 the two are decoupled, while on DS2/DC2 adding storage means adding nodes.

Each compute node is divided into slices, and a slice owns a fixed share of that node's CPU, memory and disk. A table's rows live on slices, so distribution decides which slice touches which rows. Slice counts are fixed per node type: dc2.large has 2 per node, dc2.8xlarge 16, ra3.xlplus 4, ra3.4xlarge 12 and ra3.16xlarge 48. The leader node never stores user data; it keeps the catalog, plans queries and merges results returned by the slices.

Because planning is centralized, a burst of many concurrent small queries can pressure the leader even while compute is idle. Add compute nodes to widen scan throughput. On RA3, Redshift Managed Storage separates storage from compute, so the local SSD acts as a cache and you can grow storage without adding nodes. AQUA is an RA3-only, cluster-level accelerator for some scans and aggregations; it is enabled per cluster and has no SQL DDL.

Common mistake: diagnosing a slow query as a leader-node problem when the real issue is distribution. The leader does not scan rows, so uneven work across slices, not the leader, is usually why one node looks busy. The mirror mistake is expecting to scale the leader alone; sustained heavy concurrency is handled with WLM and concurrency scaling, not a larger leader.

## How can you improve query performance for small tables in Amazon Redshift?

Use `DISTSTYLE ALL` so the table is replicated to every slice, which turns joins against it into local lookups and avoids redistribution. Keep its sort key small and keep statistics fresh. Do not distribute a small lookup table `EVEN` or by a key that does not match the join.

```sql
create table analytics.dim_currency (
  currency_code char(3) not null,
  currency_name varchar(64),
  usd_rate      numeric(12,6)
)
diststyle all
sortkey (currency_code);

analyze analytics.dim_currency;

-- With DISTSTYLE ALL this join needs no redistribution of the fact table
select f.sale_id, f.net_amount * c.usd_rate as usd_amount
from analytics.fact_sales f
join analytics.dim_currency c on f.currency_code = c.currency_code;
```

`DISTSTYLE ALL` is wrong for anything but genuinely small, slowly changing tables: it multiplies storage by the slice count and slows every write.

`DISTSTYLE ALL` writes a full copy of the table onto every slice. A join against it becomes a local lookup, so the large table never has to redistribute or broadcast rows for that join. The price is storage multiplied by the slice count and slower writes on every INSERT, COPY, UPDATE and DELETE, because each write touches all copies. `AUTO` distribution starts genuinely small tables at ALL and moves larger tables to AUTO(EVEN), or to AUTO(KEY) on the join column.

Verify the choice from the catalog rather than assuming:
```sql
select "schema", "table", diststyle, size, skew_rows
from svv_table_info
where "schema" = 'analytics'
  and "table" = 'dim_currency';
```
A `size` that is a meaningful fraction of the cluster and a high `skew_rows` ratio are signals that the table is no longer small enough to replicate.

Common mistake: tagging a table ALL because it is small today. A dimension that grows into millions of rows multiplies storage and write cost on every slice, and the benefit evaporates once it no longer fits comfortably in memory. Recheck `diststyle` and `size` as the table grows, and move a grown table back with `alter table analytics.dim_currency alter diststyle auto;`.

## Can you create indexes on tables in Amazon Redshift?

Redshift has no secondary or B-tree indexes like an OLTP database. The closest analogues are sort keys (including interleaved sort keys), zone maps, and distribution keys. If an interviewer asks about "indexes", explain those mechanisms and when to use an interleaved sort key instead.

```sql
-- Interleaved sort key: equal weight to multiple filter columns
create table analytics.fact_events (
  event_date date,
  country    varchar(2),
  event_type varchar(32),
  user_id    bigint
)
interleaved sortkey (event_date, country, event_type);

-- A query that filters on several of those columns benefits from the interleave
select country, count(*)
from analytics.fact_events
where event_date between date '2026-10-01' and date '2026-10-07'
  and event_type = 'purchase'
group by country;
```

Interleaved sort keys cost more to maintain because `VACUUM REINDEX` is required; prefer a compound sort key unless queries genuinely filter on several columns with equal weight.

Redshift has no B-tree index. The nearest analogue is the zone map. Each column is stored on a slice in 1 MB blocks, and every block keeps the minimum and maximum value for that column. A range or equality predicate can skip whole blocks whose min and max cannot match, so the physical sort order of a column decides how much I/O a filter avoids. A compound sort key helps mostly through its leading columns; an interleaved sort key spreads the benefit across several columns.

```sql
-- Compound sort key: the default, and the leading columns carry the benefit
create table analytics.fact_events (
  event_date date,
  country    varchar(2),
  event_type varchar(32),
  user_id    bigint
)
compound sortkey (event_date, country, event_type);

-- An interleaved key can be altered to compound; the reverse is not supported
alter table analytics.fact_events alter sortkey compound (event_date);
```

Common mistake: converting a sort key to interleaved to "index" several filter columns. Interleaved keys need `VACUUM REINDEX` to stay effective, cost more to maintain, and only pay off for low-selectivity filters on the interleaved columns. Because ALTER only lets you go from interleaved to compound, not back, the choice is effectively one-way; start compound unless queries genuinely weight several columns equally.

## What is the maximum number of columns you can have in a table in Amazon Redshift?

The documented limit is 1,600 columns per table. Wide tables are legal but expensive: every column adds block metadata, and `SELECT *` over 1,600 columns forces the engine to read them all. Model wide event data with `SUPER`/JSON or separate tables rather than hundreds of sparse columns.

```sql
-- Count the columns a table actually exposes
select count(*) as column_count
from svv_columns
where table_schema = 'analytics' and table_name = 'fact_sales';
```

The ceiling is 1,600 columns, but the real cost is metadata. Every column gets its own sequence of 1 MB blocks per slice, so a table with hundreds of mostly NULL columns still creates that many block streams to open, scan and maintain. Wide rows also consume the per-row size budget faster. A table can be fully legal at 1,200 columns and still be a poor design once VACUUM, ANALYZE and scan cost are counted.

For event-like data, model the payload as `SUPER` and query it with navigation syntax, or split storage into a narrow core table plus detail tables keyed by the event id. Both scan less than one sparse wide table because only the columns a query mentions are read. If a wide table is unavoidable, project explicit columns everywhere and keep sort keys short; only a few leading columns realistically drive block pruning.

Common mistake: treating the 1,600-column limit as a design target and using `select *` to reach the data. A wide sparse table multiplies block metadata and slows maintenance, and `select *` reads every one of those column streams even when the query needs three of them. Name the columns you need, and prefer semi-structured `SUPER` or a split table over hundreds of nullable fields.

## What happens when a compute node fails in Amazon Redshift?

Redshift keeps replicas of data across nodes, so when a compute node fails the leader routes work for its slices to other nodes and the cluster continues, typically with reduced performance. The node is replaced automatically and Redshift re-replicates the data. RA3 with Multi-AZ deployments adds an available-zone failure story; RA3/DS2 clusters also support relocation to another AZ during recovery.

```bash
# Watch for node replacement and cluster events
aws redshift describe-events \
  --source-identifier analytics-cluster \
  --source-type cluster \
  --duration 1440
```

Data on one slice is replicated onto other nodes, so a failed compute node does not lose that data on a multi-node cluster. The leader marks the affected slices unavailable and routes their work to the nodes holding the replicas. Queries keep running, but scans and joins that touch those slices are slower while the cluster is degraded. Redshift emits cluster events for the failure, for the node replacement and for the completion of recovery.

Replication is what makes this possible, and also what makes the recovery window long. Re-replicating data competes with live queries for network and disk, and the larger the affected slices, the longer the degraded period. RA3 clusters can be configured Multi-AZ, which keeps a mirrored copy in a second Availability Zone and supports failover with minimal data loss, and RA3 and DS2 clusters can be relocated to another Availability Zone during recovery.

Common mistake: assuming every cluster has this safety net. A single-node cluster has no second node to hold replicas, so a compute failure there is an outage, not a slowdown. Also do not treat node replacement as transparent to workloads: long-running COPY or maintenance work can fail and must be rerun, so watch the events until recovery completes instead of assuming the node's return means normal service.

## How can you automate data loading into Amazon Redshift?

Common patterns: AWS Glue jobs or Glue DataBrew for transformation plus `COPY`, Step Functions or Airflow to orchestrate, EventBridge Scheduler plus the Redshift Data API for scheduled SQL, Lambda for event-driven loads, and Kinesis Data Firehose for streaming. The Redshift Data API is the modern way to run SQL without managing a JDBC connection.

In [ ]:
# EventBridge/Lambda style: run a COPY through the Redshift Data API
import time

import boto3

client = boto3.client("redshift-data")
response = client.execute_statement(
    WorkgroupName="analytics-wg",
    Database="analytics",
    Sql="""
        copy analytics.fact_sales
        from 's3://acme-lake/curated/fact_sales/'
        iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
        format as parquet;
    """,
)
statement_id = response["Id"]

# There is no built-in waiter; poll until the statement finishes.
while client.describe_statement(Id=statement_id)["Status"] not in ("FINISHED", "FAILED", "ABORTED"):
    time.sleep(2)

Patterns that survive production: make each load idempotent (staging table plus `MERGE`, or partition swap), record every run in an audit table, and alert on `MAXERROR` breaches rather than ignoring them.

The Redshift Data API runs SQL as an asynchronous statement against a provisioned cluster or a Serverless workgroup. It returns an identifier, and the caller polls or waits on an event until the status is FINISHED, FAILED or ABORTED. Credentials are handled by the service, so no JDBC driver, connection pool or VPC path is needed from Lambda, Step Functions or EventBridge Scheduler. Group related statements into one call to cut round trips and keep the whole run idempotent.

The load itself should still be written so a rerun is safe:
```sql
copy analytics.stg_fact_sales
from 's3://acme-lake/curated/fact_sales/dt=2026-10-07/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet;

merge into analytics.fact_sales as t
using analytics.stg_fact_sales as s
  on t.sale_id = s.sale_id
when matched then update set amount = s.amount, order_date = s.order_date
when not matched then insert (sale_id, customer_id, order_date, amount)
values (s.sale_id, s.customer_id, s.order_date, s.amount);

truncate analytics.stg_fact_sales;
```

Common mistake: scheduling COPY with no duplicate guard and no run record. A Lambda retry or a Step Functions catch block can run the same load twice and double the row count. Load into a staging table and MERGE on a natural key, or swap a partition, then write one audit row per run with status, rows loaded and the MAXERROR setting. Alert when MAXERROR is reached instead of letting a partial load pass silently.

## How does Redshift handle data compression?

Redshift compresses each column independently and chooses an encoding based on the data type and value distribution. Common encodings include `AZ64` (numeric and date types, good default), `ZSTD`/`LZO` (general), `BYTEDICT`/`TEXT255` (low-cardinality strings), `DELTA`/`DELTA32K` (sorted integers and dates), `RUNLENGTH` (repeated values), and `MOSTLY8`/`MOSTLY16`/`MOSTLY32` (small integers). You can let `COPY ... COMPUPDATE ON` choose, or set encodings explicitly.

```sql
-- Inspect current encodings
select "column", type, encoding
from pg_table_def
where schemaname = 'analytics' and tablename = 'fact_sales';

-- Let COPY choose encodings for new data
copy analytics.fact_sales
from 's3://acme-lake/curated/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
compupdate on;
```

Compression is chosen per column, not per table, from the data type and the actual value distribution. The documented defaults are AZ64 for numeric and date columns, LZO for char and varchar, and RAW for sort keys, booleans and floats. `ENCODE AUTO` is the table default, so a new column starts unencoded until load or DDL assigns an encoding. `COPY ... COMPUPDATE ON` lets the load choose encodings, and `COMPUPDATE PRESET` keeps whatever is already defined.

`ANALYZE COMPRESSION` is the advisory command; it reports suggested encodings and does not change any data. You can also override one column while leaving the table under ENCODE AUTO:
```sql
-- Report better encodings; no data is modified
analyze compression analytics.fact_sales;

-- Pin one column and let ENCODE AUTO handle the rest
alter table analytics.fact_sales alter column customer_id encode zstd;
```

Common mistake: expecting `COMPUPDATE ON` to fix a bad encoding on a table that already has data. It mainly helps an empty table or newly added columns, so a poor choice made at first load tends to persist. Also avoid TEXT255 or BYTEDICT on a high-cardinality column, and avoid compressing sort key columns away from RAW. Run `ANALYZE COMPRESSION`, then load into a fresh table to re-encode wholesale.

## Can you resize a Redshift cluster while it is running?

Yes. An elastic resize changes the node count or moves between certain node types with a short reboot and minimal downtime; a classic resize migrates data and makes the cluster read-only during the operation. Serverless workgroups scale base capacity without any node-level resize at all.

```bash
# Elastic resize (fast, short reboot)
aws redshift modify-cluster \
  --cluster-identifier analytics-cluster \
  --number-of-nodes 6 \
  --node-type ra3.4xlarge \
  --elastic-resize

# Classic resize (arbitrary target, longer, read-only window)
aws redshift resize-cluster \
  --cluster-identifier analytics-cluster \
  --cluster-type multi-node \
  --number-of-nodes 12 \
  --node-type ra3.4xlarge
```

Elastic resize keeps the cluster available for most of the change. It adjusts the node count or a compatible node type quickly, then reboots briefly, so open connections drop and clients must reconnect. Classic resize can target a very different size or node family, but it copies every table to the new nodes and the cluster stays read-only until that copy finishes, which can take hours at multi-terabyte scale. On RA3, storage lives in Redshift Managed Storage, so an elastic resize mainly re-provisions compute and the data does not have to move.

```sql
-- After any resize, re-check slice count and row skew
select count(*) as slice_count
from stv_slices;

select "table", diststyle, skew_rows, skew_sortkey1
from svv_table_info
where "table" in ('fact_sales', 'dim_product');
```

Common mistake: resizing to fix a slow dashboard when the real problem is a skewed distribution key or stale statistics. A bigger cluster redistributes the same uneven data and stays slow. Check `skew_rows` and `stats_off` first, and remember that a resize changes the slice count, so distribution and sort choices deserve a fresh look afterwards.

## How does Redshift handle data distribution in a cluster?

The leader assigns each slice a portion of every table according to the table's distribution style. `KEY` hashes the distribution column, `EVEN` round-robins, `ALL` replicates small tables, and `AUTO` lets Redshift choose and adapt. During joins the optimizer either redistributes, broadcasts, or merge-joins the inputs, so distribution design is about eliminating the most expensive movement for your dominant queries.

```sql
-- Which tables are skewed, and by how much?
select schema, "table", diststyle, skew_rows, skew_sortkey1
from svv_table_info
where skew_rows > 1.5
order by skew_rows desc;
```

Distribution decides where a row lives, and that decides how much data moves at join time. When both sides are joined on their distribution keys, each slice joins locally. When one side is small, the planner broadcasts it to every slice. Otherwise it redistributes one or both sides across the network. Redistribution and broadcast are the expensive parts, so the distkey should be the column joined most often, with many distinct values so no slice receives a hot share.

```sql
create table analytics.fact_sales (
  sale_id     bigint,
  product_key integer,
  date_key    integer,
  amount      numeric(12,2)
)
diststyle key distkey (product_key)
sortkey (date_key);

-- Small dimensions are cheapest as a full copy on every slice
create table analytics.dim_product (
  product_key integer,
  category    varchar(64)
)
diststyle all;
```

Common mistake: choosing a distribution key just because it looks like a primary key, even when it is skewed or rarely joined. A date column as distkey pushes a single day of rows to one slice. Also remember that ALL is only cheap for small tables; replicating a large fact table multiplies storage and slows every load.

## What is the purpose of the COPY command in Amazon Redshift?

`COPY` is the high-throughput bulk loader. It reads from S3, DynamoDB, EMR, or a remote host and loads directly into slices in parallel, applying compression, encoding selection, statistics updates, and error handling along the way. It is dramatically faster and cheaper than row-by-row `INSERT`.

```sql
copy analytics.fact_sales
from 's3://acme-lake/curated/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
compupdate on
statupdate on
maxerror 10;
```

COPY divides the input into slices and loads them in parallel. Throughput comes from file layout: aim for a number of files that is a multiple of the slice count, roughly one file per slice, and keep the files similar in size. A single giant gzip file is read by one slice while the rest sit idle. A manifest lists the exact files to load and lets the load fail cleanly when one file is missing.

```sql
copy analytics.fact_sales
from 's3://acme-lake/curated/fact_sales/manifest.json'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
manifest
compupdate preset
statupdate on
maxerror 0;
```

Common mistake: raising MAXERROR to push a failing load through. The statement reports success while rows are silently missing. Load into a staging table, inspect STL_LOAD_ERRORS, then move the clean rows forward. Also note that a COPY does not refresh statistics unless you ask, so stale stats after a large load can send the optimizer into a bad plan.

## Can you use Redshift as an OLTP (Online Transaction Processing) database?

No, not as a general-purpose OLTP database. Redshift is an OLAP columnar, MPP warehouse optimized for large scans and aggregations; it lacks row-level secondary indexes and is not designed for thousands of concurrent single-row transactions. It does support ACID transactions, `MERGE`, and UDFs, so modest operational-style writes are fine, but a high-TPS application belongs on Aurora, DynamoDB, or another OLTP engine.

```sql
-- Occasional operational-style upsert is fine ...
merge into analytics.dim_customer t
using analytics.stg_customer s
on t.customer_id = s.customer_id
when matched then update set t.email = s.email, t.segment = s.segment
when not matched then insert (customer_id, email, segment)
values (s.customer_id, s.email, s.segment);

-- ... but thousands of these per second is an OLTP workload, not a warehouse one.
```

Redshift stores each column in separate 1 MB blocks, so a scan reads only the columns it needs. That is excellent for aggregation and poor for point access: there is no row-level B-tree, and a single-row lookup still touches a whole block. Columnar storage also means an UPDATE rewrites blocks and leaves dead rows behind. Each small statement carries commit overhead that a row store hides, so per-statement latency grows quickly as write volume rises.

Common mistake: using Redshift as the system of record for an application that issues thousands of single-row INSERTs per second. Latency and commit overhead climb, and the steady drip of tiny writes creates dead rows that VACUUM must later reclaim. Land the operational traffic in Aurora or DynamoDB first, then batch it into the warehouse.

## How does Redshift handle data backup and durability?

Redshift replicates data within the cluster so a node failure does not lose it, and continuously backs the cluster up to S3 as automated snapshots with a configurable retention window (1 to 35 days). Manual snapshots persist until deleted, and snapshots can be copied to another region or shared across accounts. RA3 Multi-AZ deployments add synchronous standby capacity for zone failures.

```bash
aws redshift describe-clusters --cluster-identifier analytics-cluster \
  --query 'Clusters[0].[ClusterStatus,AutomatedSnapshotRetentionPeriod,Encrypted]'
```

Automated snapshots are taken continuously, roughly every 8 hours or after 5 GB of change per node, and are kept for the retention window of 1 to 35 days. The default retention is short, so long recovery windows need an explicit change. A snapshot restores into a new cluster rather than in place. Manual snapshots persist until deleted, which is what you take before a risky migration. Cross-region snapshot copy and snapshot sharing let another region or account restore the data, and RA3 Multi-AZ keeps a synchronous standby in a second zone.

Test restores on a schedule instead of assuming they work. Restore a snapshot into a throwaway cluster and run a row-count check on the key tables. Snapshots are also the only recovery path for an accidental DELETE or TRUNCATE, because there is no undo inside the warehouse.

Common mistake: treating automated snapshots as a backup nobody has to test, and leaving retention at the default. A bad load noticed two days later is already gone from automated snapshots unless retention was raised or a manual snapshot was taken beforehand.

## What is the purpose of the Redshift Query Optimizer?

It is a cost-based optimizer: it rewrites the query, enumerates join orders and join types, estimates cardinality from table and column statistics, and picks the plan with the lowest estimated cost. It decides whether to redistribute or broadcast, whether to use a merge join on sort keys, and where to push predicates. Its blind spot is stale or missing statistics, which is why `ANALYZE` matters so much.

```sql
analyze analytics.fact_sales;

explain
select p.category, sum(f.net_amount)
from analytics.fact_sales f
join analytics.dim_product p on f.product_key = p.product_key
where f.date_key = 20261004
group by p.category;
```

EXPLAIN prints the plan tree the optimizer chose: scan nodes, join nodes, and an estimated cost. Labels such as DS_BCAST_INNER mean a broadcast, DS_DIST_BOTH means both sides are redistributed, and DS_DIST_NONE means the join ran locally on each slice. Predicates are pushed to the scan so zone maps can skip whole 1 MB blocks. Because the optimizer estimates cardinality from statistics, wrong row counts lead to a bad join order or a broadcast that should have been a local merge join.

```sql
-- Refresh stats only on the columns used for joins and filters
analyze analytics.fact_sales (product_key, date_key);

select "table", tbl_rows, stats_off
from svv_table_info
where "table" = 'fact_sales';
```

Common mistake: trusting the estimated row counts in a plan when `stats_off` is high. A table that has doubled since the last ANALYZE can be estimated at a fraction of its real size, so the optimizer picks a broadcast that floods the network. Re-run ANALYZE after large loads and confirm the plan against the rows each step actually moved.

## Can you run Redshift queries from third-party BI tools?

Yes. Redshift speaks PostgreSQL-compatible SQL and exposes JDBC/ODBC drivers plus native connectors, so Tableau, Power BI, Looker, Metabase, DBeaver, and similar tools can query it. For BI workloads, plan the concurrency side: WLM queues, short query acceleration, concurrency scaling, and materialized views for repeated aggregates.

```sql
-- A materialized view keeps a common dashboard aggregate cheap
create materialized view analytics.mv_revenue_daily
auto refresh yes
as
select
  date_trunc('day', order_date) as order_day,
  region,
  sum(amount) as revenue
from analytics.fact_orders
group by order_day, region;
```

BI tools connect over JDBC or ODBC and mostly issue SELECT statements. Two things break first at scale: connection count and query concurrency. A dashboard refresh can open one connection per chart, so pool and reuse sessions instead. On the cluster side, WLM queues, short query acceleration, and concurrency scaling absorb bursts, while the result cache serves identical repeated queries without touching compute.

```sql
-- Are dashboard queries being served by concurrency scaling?
select query_id, service_class_name, compute_type, elapsed_time, queue_time
from sys_query_history
order by start_time desc
limit 20;
```

Common mistake: letting every BI user run SELECT * across a wide fact table on a live connection. It defeats column pruning, spikes concurrency, and pushes the cluster into queueing. Point dashboards at narrow materialized views or pre-aggregated tables, and set a query timeout so one runaway chart cannot hold a slot.

## Can you create temporary tables in Amazon Redshift?

Yes. Temp tables exist for the session (default) or the transaction (`ON COMMIT DROP`/`ON COMMIT DELETE ROWS`), and they are useful for staging within an ETL step. Create them with `CREATE TEMP TABLE ... AS` so no explicit distribution design is needed for short-lived data.

```sql
create temp table stg_orders as
select order_id, customer_id, order_date, amount
from spectrum_lake.orders
where dt = '2026-10-04';

-- ... transform ...

drop table stg_orders;

-- Transaction-scoped variant
create temp table stg_load (id bigint, payload varchar(1024))
on commit drop;
```

A temp table is visible only inside the session that created it, and it is not part of any snapshot. A connection pool can reuse the same physical session later, so a leftover temp table may collide with a new one of the same name. ON COMMIT DROP ties the table to the surrounding transaction, while the default keeps it until the session ends. Constraints such as PRIMARY KEY on a temp table are informational and are not enforced.

```sql
-- Give a temp table the same distkey as the big table it joins
create temp table stg_orders (
  order_id    bigint,
  customer_id integer,
  amount      numeric(12,2)
)
diststyle key distkey (customer_id);

insert into stg_orders
select order_id, customer_id, amount
from analytics.fact_orders
where order_date >= dateadd(day, -7, getdate());
```

Common mistake: assuming a temp table survives the session, or creating one on a pooled connection and finding it gone, or already present, on the next use. Do not use temp tables to pass state between ETL steps that may run on different connections.

## How can you monitor query performance in Amazon Redshift?

Use the console's query monitoring and performance insights, CloudWatch for cluster metrics, and the system tables for detail. The most useful are `STL_QUERY`/`STL_QUERYTEXT`, `SVL_QUERY_SUMMARY` (step and time breakdown), `SVL_QUERY_REPORT` (slice-level skew), `STL_WLM_QUERY` (queue waits), and `STL_ALERT_EVENT_LOG` (nested loops, missing stats, and other alerts).

```sql
-- Find the slowest step of the slowest query, and check for slice skew
select
  query,
  segment,
  step,
  max(rows) as max_rows,
  avg(rows) as avg_rows,
  max(rows) - avg(rows) as skew,
  max(elapsed_time) as elapsed_us
from svl_query_report
where query = 1234567
group by query, segment, step
order by elapsed_us desc
limit 10;
```

```sql
-- Alert events in the last day: what did the optimizer complain about?
select query, event, solution
from stl_alert_event_log
where event_time >= dateadd(day, -1, getdate())
order by event_time desc
limit 20;
```

Choose the view by how long the data must live. STL views carry per-segment and per-slice detail but are trimmed periodically, so they are for the query you are debugging right now. SVL views keep durable summary rows. SYS_QUERY_HISTORY and the other SYS views are the long-term record. CloudWatch carries cluster-level signals such as CPU, disk space, and queue length, which flag a problem even when no single query stands out.

```sql
-- Longest-running queries in the last day, with their queue wait
select query_id, username, elapsed_time, queue_time, error_message
from sys_query_history
where start_time >= dateadd(day, -1, getdate())
order by elapsed_time desc
limit 20;
```

Common mistake: digging into an STL view for a slow query from last week, when those rows may already have been trimmed. Also do not treat one STL_QUERY row as one query: the rows are per query step or segment, so the granularity matters when you count or average.

## Can you perform updates and deletes on tables in Amazon Redshift?

Yes. Redshift supports `UPDATE`, `DELETE`, and `MERGE` with full ACID transactions and snapshot isolation. Internally, an update or delete marks rows for deletion and writes new versions; the old rows stay until `VACUUM` reclaims space, which is why heavy mutation churn causes bloat and unsorted data.

```sql
-- Update and delete are normal SQL
update analytics.fact_orders
set status = 'shipped'
where order_id = 'o-1042' and status = 'packed';

delete from analytics.fact_orders
where order_date < date '2020-01-01';

-- MERGE is the idiomatic upsert for batch ELT
merge into analytics.fact_orders t
using analytics.stg_orders s
on t.order_id = s.order_id
when matched then update set t.status = s.status, t.amount = s.amount
when not matched then insert (order_id, customer_id, order_date, status, amount)
values (s.order_id, s.customer_id, s.order_date, s.status, s.amount);
```

For very large rewrites, the faster pattern is `CREATE TABLE AS SELECT` (or `INSERT INTO ... SELECT`) into a new table and rename, which avoids touching rows one at a time.

An update or delete is physical work, not an in-place edit. Redshift writes a new version of each changed row and marks the old version deleted. Readers using the same snapshot keep seeing the old rows until the transaction commits, so a query never sees a half-applied change. The dead rows stay inside the 1 MB blocks on the slice, and the table also drifts out of sort order. Look at the table shape before and after cleanup:
```sql
select "table", size, pct_used, unsorted, stats_off, skew_rows, estimated_visible_rows
from svv_table_info
where "table" = 'fact_orders';

vacuum delete only analytics.fact_orders;
vacuum sort only analytics.fact_orders;
```

Transactions use snapshot isolation, so two statements that touch the same rows serialize, and a long `MERGE` can block concurrent writers. The `MERGE` join key must match at most one target row. If the staging table carries duplicate keys, Redshift raises a duplicate key error instead of silently picking a winner. Maintenance also serializes: `VACUUM` and `ALTER TABLE` cannot run concurrently on the same table, so a background vacuum competes with a distribution or sort key change.

Common mistake: Deleting in a loop and expecting the space to come back. Dead rows are reclaimed only by `VACUUM DELETE ONLY`, and each statement adds more delete overhead. When a change touches most of a table, rewrite it instead: `CREATE TABLE AS SELECT` the surviving rows into a new table, then rename it into place.

## What is the purpose of the Redshift COPY command's IGNOREHEADER option?

`IGNOREHEADER n` tells `COPY` to skip the first `n` lines of each input file, which is how you load CSV files that begin with a header row. Use `IGNOREHEADER 1` for a single header line; the option is per file, not per load, so every file must have the same structure.

```sql
copy analytics.stg_orders
from 's3://acme-lake/raw/orders_2026-10-04.csv'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
csv
ignoreheader 1
region 'us-east-1';
```

`IGNOREHEADER` applies to every file in the load, not just the first. If one file has a header and another does not, the second file's first data line is dropped as if it were a header. The count is independent of `CSV`: Redshift skips that many physical lines, then parses the rest as CSV. It applies to delimited text only. Parquet and ORC files have no header line, so the option is meaningless there.

Combine it with the other text options the feed actually needs:
```sql
copy analytics.stg_orders
from 's3://acme-lake/raw/orders/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
csv
ignoreheader 1
gzip
emptyasnull
truncatecolumns
acceptinvchars
region 'us-east-1';
```
Skipped lines are not counted against `MAXERROR`. A manifest load can mix gzip and plain files as long as every file has the same leading line count.

Common mistake: Assuming `IGNOREHEADER` can skip trailing totals lines or blank rows. It only skips leading lines. Blank rows and a trailing footer still fail parsing or land as data. Handle those upstream, or load with a small `MAXERROR` and review `STL_LOAD_ERRORS`.

## What is the purpose of the Redshift COPY command's MAXERROR option?

`MAXERROR n` allows up to `n` rows to fail during the load before `COPY` aborts (default `MAXERROR 0`). The rejected rows are written to `STL_LOAD_ERRORS` with the reason, so a tolerant load must be paired with a review of that table. Use a small tolerance for messy external feeds and zero for internal, well-validated data.

```sql
copy analytics.stg_orders
from 's3://acme-lake/raw/orders_2026-10-04.csv'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
csv
ignoreheader 1
maxerror 100;

-- Exactly which rows were rejected, and why
select line_number, colname, raw_line, err_code, err_reason
from stl_load_errors
where filename like '%orders_2026-10-04.csv%'
order by line_number
limit 50;
```

`MAXERROR` is a running count for the whole load, and the default is 0. Redshift checks it as rows are read. Once the limit is exceeded, `COPY` aborts, and because `COPY` is atomic no rows from that statement become visible. A load that finishes under the limit succeeds, but the rejected rows are not written to a dead-letter location. They exist only as rows in `STL_LOAD_ERRORS`, and those STL views are trimmed over time, so save the detail if you need an audit trail.

`lines_scanned` in `STL_LOAD_COMMITS` counts lines read, not rows committed, so a tolerated bad row hides in the difference.
```sql
copy analytics.stg_orders
from 's3://acme-lake/raw/orders/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
csv
ignoreheader 1
maxerror 100;

select query, trim(filename) as file, lines_scanned, curtime
from stl_load_commits
where filename like '%orders%'
order by curtime desc
limit 20;
```

Common mistake: Setting a generous `MAXERROR` to keep the pipeline green and never checking the error view. The load reports success while quietly dropping rows, so downstream totals drift. Alert whenever the rejected count is above zero, and treat repeated errors on the same column as schema drift.

## Can you create materialized views in Amazon Redshift?

Yes. `CREATE MATERIALIZED VIEW` stores the precomputed result, and `AUTO REFRESH YES` keeps it current as base tables change. The optimizer can automatically rewrite matching queries to use the materialized view, so dashboards speed up without changing their SQL. Refresh incrementally or fully, depending on the query.

```sql
create materialized view analytics.mv_sales_by_category
diststyle even
sortkey (order_day)
auto refresh yes
as
select
  date_trunc('day', f.order_date) as order_day,
  p.category,
  sum(f.amount) as revenue
from analytics.fact_sales f
join analytics.dim_product p on f.product_key = p.product_key
group by order_day, p.category;

refresh materialized view analytics.mv_sales_by_category;

-- Freshness check
select name, state, is_stale, autorefresh
from stv_mv_info;
```

Limitations: not every query shape is supported, an auto-refresh view can lag the base tables briefly, and a materialized view costs storage. Use them for repeated, expensive aggregates rather than one-off analysis.

A refresh has two shapes. Incremental refresh reads only the changed base-table rows and patches the view. Full refresh recomputes the whole query. A view whose query shape is not supported for incremental refresh falls back to a full rebuild, which can cost far more than the query it was meant to accelerate. `REFRESH MATERIALIZED VIEW` runs synchronously in your session, while `AUTO REFRESH YES` refreshes in the background and can lag the base tables for a while.

Track which shape each refresh took and how long it ran:
```sql
create materialized view analytics.mv_customer_ltv
diststyle even
auto refresh yes
as
select customer_id, sum(amount) as lifetime_value
from analytics.fact_sales
group by customer_id;

select mv_name, refresh_type, status, start_time, duration
from sys_mv_refresh_history
where mv_name = 'mv_customer_ltv'
order by start_time desc
limit 10;
```

Common mistake: Building a materialized view over staging tables that are truncated and reloaded, or over very volatile base tables. Every load invalidates the view and triggers another refresh, so total maintenance cost can exceed the savings. Point materialized views at stable dimensions and fact tables, and measure refresh duration before rolling one out broadly.

## What is the purpose of the Redshift UNLOAD command's MANIFEST option?

`MANIFEST` makes `UNLOAD` write a manifest file that lists the exported data files. Downstream tools read the manifest to discover exactly which files belong to the export, which avoids both "read the whole prefix" and "read a half-finished export" bugs, and it is what `COPY` expects when loading a multi-file export back in.

```sql
unload ('select * from analytics.fact_sales where date_key = 20261004')
to 's3://acme-export/fact_sales/2026-10-04/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
format as parquet
allowoverwrite
manifest;

-- The manifest is written alongside the data files:
-- s3://acme-export/fact_sales/2026-10-04/manifest-<id>
```

```sql
-- Reload exactly those files elsewhere
copy analytics.fact_sales_reload
from 's3://acme-export/fact_sales/2026-10-04/manifest-1234'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
manifest
format as parquet;
```

`MANIFEST VERBOSE` writes the same file list with extra detail per entry, such as row counts and file sizes, so a loader can size the job and spot an export that is short. With `PARTITION BY`, data lands under Hive-style prefixes and the manifest still lists every underlying file. The manifest file name is generated by Redshift, so capture it from the `UNLOAD` output or read it from `SYS_UNLOAD_HISTORY`. The loader reads that single file, not the prefix.

```sql
unload ('select * from analytics.fact_sales where date_key >= 20260101')
to 's3://acme-export/fact_sales_2026/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
format as parquet
partition by (date_key)
manifest verbose
maxfilesize 128 mb
allowoverwrite;

select query_id, file_format, status, unloaded_rows, unloaded_files_count
from sys_unload_history
order by start_time desc
limit 10;
```

Common mistake: Pointing `COPY` at the output prefix instead of the manifest. That can pull in files from an earlier or still-running export. The manifest pins the exact file set. Also, do not rename or delete files after the manifest is written, because it still references the old paths and the reload fails on the missing objects.

## Can you enable automatic compression encoding for columns in Amazon Redshift?

Yes. `COPY ... COMPUPDATE ON` (or `TRUE`) analyzes incoming data and applies the best encoding per column; `COMPUPDATE PRESET` applies the default encoding without analyzing, and `OFF` disables it. You can also change an encoding explicitly on an existing column.

```sql
copy analytics.fact_sales
from 's3://acme-lake/curated/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
compupdate on;

-- Explicit encoding change on an existing column
alter table analytics.fact_sales
alter column net_amount encode az64;
```

Changing an encoding rewrites the column, so do it in a maintenance window on large tables.

`COMPUPDATE ON` samples the incoming data, chooses an encoding per column, and applies it, which adds analysis time to the load. `COMPUPDATE PRESET` skips the analysis and applies the type defaults: AZ64 for numeric and date, LZO for char and varchar, RAW for sort key, boolean, and float columns. `OFF` leaves existing encodings untouched. Encodings on a table that already holds data change only through `ALTER TABLE ... ALTER COLUMN ... ENCODE`, which rewrites that column.

```sql
analyze compression analytics.fact_sales;

alter table analytics.fact_sales
alter column status encode zstd;

alter table analytics.fact_sales
alter column region_code encode bytedict;
```
`ANALYZE COMPRESSION` only reports what it recommends. It reads the data but does not change a single encoding, so the `ALTER` statements remain your decision.

Common mistake: Re-running `COPY ... COMPUPDATE ON` on every load of a large table to chase space savings. The analysis repeats each time and lengthens the load. Once encodings are set, use `COMPUPDATE PRESET`, and revisit encodings deliberately with `ANALYZE COMPRESSION` rather than on every batch.

## What is the purpose of the Redshift ANALYZE command's VERBOSE option?

`ANALYZE ... VERBOSE` prints progress and detail about each table as statistics are collected, which is useful when you are analyzing a large schema and want to see where time is going. Without `VERBOSE`, the command succeeds silently.

```sql
analyze verbose analytics.fact_sales;
```

For scheduled maintenance, prefer non-verbose `ANALYZE` and inspect `SVV_TABLE_INFO.stats_off` to decide what needs re-analysis.

`ANALYZE` is incremental. It refreshes statistics for columns whose data changed and skips unchanged ones, so repeated runs on a quiet table are cheap. Narrowing it to the columns that appear in predicates is usually enough for the optimizer. Redshift also runs auto-analyze in the background after writes, and those actions appear in `SVL_AUTO_WORKER_ACTION`. `ANALYZE` and `ANALYZE COMPRESSION` are different commands: the second only reports encoding recommendations and never collects statistics.

```sql
analyze analytics.fact_sales;

analyze analytics.fact_sales (order_date, customer_id);

select "table", stats_off, tbl_rows, unsorted, skew_rows
from svv_table_info
where schema = 'analytics'
order by stats_off desc;
```
`stats_off` runs from 0 (current) to 100 (out of date), so a sorted list of it gives you the tables that actually need attention.

Common mistake: Treating `ANALYZE` as the default fix for a slow query. Stale statistics are only one cause; a large `unsorted` percentage or severe `skew_rows` in `SVV_TABLE_INFO` needs vacuum or a distribution change, not more stats. Check `EXPLAIN` before adding maintenance, and keep `VERBOSE` for interactive debugging rather than scheduled jobs.

## Can you use Redshift Federated Query to query data in other AWS data sources?

Yes. Federated Query lets Redshift read live data from Amazon RDS and Aurora (PostgreSQL and MySQL), and Amazon DocumentDB, through an external schema backed by a Secrets Manager secret. Redshift pushes filters and aggregations down to the source where possible and joins the result with local tables.

```sql
create external schema crm from postgres
  database 'crm' schema 'public'
  uri 'my-rds-instance.abc123.us-east-1.rds.amazonaws.com'
  iam_role 'arn:aws:iam::123456789012:role/redshift-federated-role'
  secret_arn 'arn:aws:secretsmanager:us-east-1:123456789012:secret:crm-creds';

select c.customer_id, c.email, sum(s.amount) as revenue
from crm.customers c
join analytics.fact_sales s on s.customer_id = c.customer_id
where s.order_date >= date '2026-01-01'
group by c.customer_id, c.email
order by revenue desc
limit 100;
```

Federated Query is for relatively small lookups and enrichment; it is not a substitute for a proper ingestion pipeline when the source is large.

Federated Query is read-only and pull-based. Redshift generates SQL for the remote engine, which runs it and returns matching rows over the network, so only pushdown-able filters and aggregates stay on the source. A join to a local table forces the remote side back to Redshift and redistributes it. That makes federated query good for small dimension lookups and enrichment, and a poor fit for scanning large remote fact tables. The external schema also depends on a Secrets Manager secret and on network reachability from the cluster's VPC.

Use `EXPLAIN` on a federated query to see what was actually pushed to the remote engine:
```sql
explain
select c.customer_id, c.email
from crm.customers c
where c.status = 'active'
  and c.created_at >= date '2026-01-01';

select schemaname, eskind, databasename, esoptions
from svv_external_schemas
where schemaname = 'crm';
```
`SVV_EXTERNAL_SCHEMAS` shows `eskind`, which tells you whether the schema points at a data catalog, a Hive metastore, or a federated relational source.

Common mistake: Joining two large federated tables, or a big local fact table to a federated table, assuming the work executes remotely. The remote result must still cross the network and be redistributed, and a slow remote query holds Redshift resources while it waits. If the remote side is large, stage it locally first.

## How can you export data from Amazon Redshift to Amazon S3?

Use `UNLOAD` for large, parallel, server-side exports; use `CREATE EXTERNAL TABLE AS` (CETAS) when you want the exported data to be immediately queryable as a Spectrum table; and use the Redshift Data API plus an application if you need a small, custom extract.

```sql
-- Export a month of sales as Parquet, partitioned by day
unload ('select * from analytics.fact_sales where date_key between 20261001 and 20261031')
to 's3://acme-export/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
format as parquet
partition by (date_key)
allowoverwrite;

-- Export straight into a queryable external table
create external table spectrum_lake.fact_sales_202610
stored as parquet
location 's3://acme-export/fact_sales/2026-10/'
as
select * from analytics.fact_sales where date_key between 20261001 and 20261031;
```

Choose by size and consumer. `UNLOAD` writes many files in parallel and downstream engines read them concurrently, so target a file count near a multiple of the slice count and bound the size of each file. `PARTITION BY` writes Hive-style prefixes so Spectrum and Athena can prune partitions later. CETAS registers the output in the external catalog so it is immediately queryable, at the cost of a catalog entry. The Data API suits small, application-driven extracts.

```sql
unload ('select customer_id, email from analytics.dim_customer where is_active')
to 's3://acme-export/dim_customer/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
format as csv
header
delimiter '|'
maxfilesize 64 mb
allowoverwrite;

select query_id, status, unloaded_rows, unloaded_files_count, unloaded_location
from sys_unload_history
order by start_time desc
limit 10;
```

Common mistake: Exporting into a prefix that already holds files and forgetting `ALLOWOVERWRITE`, which fails the run, or reloading a whole prefix that now holds two exports and doubling the rows. Give each export a run-stamped prefix, keep the manifest beside the data, and verify the row count before downstream jobs consume the output.

## What is the purpose of the Redshift REINDEX command?

There is no standalone `REINDEX` command in Redshift. The equivalent is `VACUUM REINDEX`, which rebuilds the index structure for tables that use an `INTERLEAVED` sort key. `VACUUM REINDEX` first re-analyzes the distribution of values and then rebuilds the interleaved index, which is expensive and should be scheduled.

```sql
-- Only meaningful for interleaved sort keys
vacuum reindex analytics.fact_events;

-- After the reindex, statistics should be refreshed
analyze analytics.fact_events;

-- Check whether a table uses interleaved sort keys
select schema, "table", sortkey1, sortkey_num
from svv_table_info
where sortkey1 is not null
order by schema, table;
```

For compound sort key tables, use `VACUUM SORT ONLY` instead; calling `VACUUM REINDEX` on them does not do what people expect.

An interleaved sort key gives equal weight to several columns, so a range filter on any one of them can prune blocks. That flexibility decays: the stored value distribution drifts as rows are inserted, and plan quality drops until the index is rebuilt. `VACUUM REINDEX` first re-analyzes the distribution of values in the sort key columns, then rebuilds the index from that fresh distribution. It reads the whole table and rewrites index blocks, so treat it as a maintenance window job. The plain `VACUUM` form does not rebuild an interleaved index, and neither does `VACUUM SORT ONLY`.

```sql
-- Interleaved sort key is declared with the table
create table analytics.fact_events (
  event_id bigint,
  user_key integer,
  event_ts timestamp,
  event_type varchar(32)
)
interleaved sortkey (user_key, event_ts);

-- Rebuild the interleaved index after enough new rows land
vacuum reindex analytics.fact_events;

-- Size up the work and confirm the sort style first
select "table", sortkey1, sortkey_num, unsorted, vacuum_sort_benefit
from svv_table_info
where "table" = 'fact_events';
```

Common mistake: running `VACUUM REINDEX` on a compound sort key table, or converting an interleaved table to compound and then expecting to switch it back. Interleaved can be altered to compound, never the reverse, so the return trip means rebuilding the table (for example with CTAS) rather than a VACUUM. Also remember that VACUUM cannot run inside an explicit transaction block, and VACUUM and ALTER TABLE cannot run concurrently on the same table, so chained maintenance jobs will queue or fail.

## Can you restore a Redshift cluster to a specific point in time?

Yes. Automated snapshots are the basis for point-in-time restore within the retention window (1 to 35 days). Point-in-time restore creates a new cluster from a snapshot as of a chosen time; it does not rewind the existing cluster in place, so the source cluster keeps running and you cut over deliberately.

```bash
# Restore to a specific time into a new cluster
aws redshift restore-from-cluster-snapshot \
  --cluster-identifier analytics-restore-20261004 \
  --snapshot-identifier <automated-snapshot-id> \
  --node-type ra3.4xlarge \
  --number-of-nodes 4 \
  --availability-zone us-east-1a

# Find the snapshot closest to the target time
aws redshift describe-cluster-snapshots \
  --cluster-identifier analytics-cluster \
  --snapshot-type automated \
  --query 'reverse(sort_by(Snapshots,&SnapshotCreateTime))[:5].[SnapshotIdentifier,SnapshotCreateTime]'
```

Automated snapshots drive point-in-time restore. Redshift takes them roughly every 8 hours, and again after about 5 GB of change on the cluster, and keeps them for the retention window you set (1 to 35 days). Manual snapshots are kept until you delete them, so they are the only way to hold a restore point longer than the window. The restore always builds a new cluster with a new identifier, a new endpoint, and a node type and node count you choose, in the same or a different Availability Zone. The source cluster is untouched, so the cutover is a connection-string change in your clients.

```sql
-- Sanity-check the restored copy before pointing clients at it
select "table", tbl_rows, size, stats_off
from svv_table_info
where schema = 'analytics'
order by tbl_rows desc;

-- Confirm the restored cluster answers queries and reports time normally
select current_timestamp as checked_at;
```

Common mistake: expecting point-in-time restore to rewind the live cluster in place. It does not; it creates a second cluster, and anything written to the source after the target time is absent until you replay or reload it. A second mistake is assuming the retention window is unlimited: once an automated snapshot ages out it cannot be restored, so keep a manual snapshot before risky migrations or large ETL backfills.

## What is the purpose of the Redshift EXPLAIN command?

`EXPLAIN` prints the query plan the optimizer chose, with each step (scan, hash join, merge join, aggregate, distribute, broadcast) and its estimated cost and row count. It is the first tool for understanding a slow query: look for `DS_DIST_BOTH`, `DS_BCAST_INNER`, `XN_Nested_Loop`, or a full sequential scan where you expected a range-restricted scan.

```sql
explain
select p.category, sum(f.net_amount)
from analytics.fact_sales f
join analytics.dim_product p on f.product_key = p.product_key
where f.date_key between 20261001 and 20261031
group by p.category;

-- Runtime counterpart: where did the time actually go?
select query, stm, seg, step, avg(elapsed_time) as avg_us, sum(rows) as rows
from svl_query_summary
where query = pg_last_query_id()
group by query, stm, seg, step
order by avg_us desc
limit 10;
```

Redshift does not expose a single `EXPLAIN ANALYZE` statement; you pair `EXPLAIN` with execution statistics from `STL`/`SVL` system views.

`EXPLAIN` never executes the query. It prints the plan the optimizer chose, in a tree of steps, with an abstract cost and an estimated row count per step. Cost is a relative unit, not milliseconds, so compare plans against each other rather than reading it as a runtime. The useful signals are structural: a broadcast where you expected a co-located join, a nested loop over a large input, or a sequential scan on a table you thought a sort key would prune. Where the estimate comes from stale statistics, the plan is only as good as the last ANALYZE.

```sql
-- Compare the estimate with the runtime counters
select query, segment, step_type, run_time, cpu_time, blocks_read, blocks_to_disk
from stl_query_metrics
where query = pg_last_query_id()
order by run_time desc
limit 20;

-- Step detail: time, memory and whether the step spilled
select query, stm, seg, step, avgtime, maxtime, rows, bytes, workmem, is_diskbased
from svl_query_summary
where query = pg_last_query_id()
order by avgtime desc
limit 10;
```

Common mistake: treating the estimated row count in the plan as fact and skipping the runtime views. A large gap between estimated and actual rows usually means stale statistics or a predicate the optimizer cannot estimate, and the fix is `ANALYZE` on the join and filter columns, not a rewrite. Another mistake is assuming `EXPLAIN` alone reveals skew: cross-slice imbalance shows up in `SVL_QUERY_METRICS.cpu_skew` and `io_skew`, and in `SVV_TABLE_INFO.skew_rows`, not in the plan text.

## Can you create user-defined functions (UDFs) in Amazon Redshift?

Yes, in three forms. SQL UDFs wrap a SQL expression and are inlined by the optimizer. Python UDFs (`LANGUAGE plpythonu`) run scalar Python for logic SQL cannot express, although AWS is phasing them out (no new Python UDFs from Patch 198, with end of support after June 30, 2026). Lambda UDFs call an AWS Lambda function from SQL. Redshift also supports stored procedures for multi-statement logic.

```sql
-- SQL scalar UDF, inlined into the query plan
create function analytics.mask_email(email varchar)
returns varchar
immutable
as $$
  select regexp_replace(email, '^[^@]+', '***')
$$ language sql;

select customer_id, analytics.mask_email(email) as masked_email
from analytics.dim_customer
limit 10;

-- Lambda UDF
create external function analytics.score_risk(features varchar)
returns float
volatile
lambda 'risk-score'
iam_role 'arn:aws:iam::123456789012:role/redshift-lambda-role';
```

Python and Lambda UDFs are slower per row, so reserve them for genuinely custom logic, not for operations SQL already does well.

Pick the function form by where it runs. A SQL UDF is a single scalar expression that the optimizer inlines into the calling query, so it costs roughly nothing at runtime and benefits from predicate pushdown. A Python scalar UDF runs sandboxed Python per row on the compute nodes, cannot reach the network, and adds per-row overhead. A Lambda UDF batches rows out to an AWS Lambda function, which is powerful for custom models but adds network latency and cannot be used in some query shapes. Volatility matters: mark a function `IMMUTABLE` only when the result depends solely on the arguments.

```sql
-- SQL UDF: safe to mark immutable because it depends only on the argument
create function analytics.fiscal_quarter(d date)
returns varchar(8)
immutable
as $$
  select 'Q' || cast(extract(quarter from d) as varchar(1))
$$ language sql;

select analytics.fiscal_quarter(order_date) as qtr, count(*)
from analytics.fact_sales
group by 1
order by 1;

-- Inspect and remove a function by signature
select proname, prosrc from pg_proc where proname = 'fiscal_quarter';
drop function analytics.fiscal_quarter(date);
```

Common mistake: tagging a function `IMMUTABLE` or `STABLE` when it reads other tables, calls a Lambda, or uses the clock. The optimizer may fold the call at plan time or reuse one result, so rows get a stale value and the bug appears only after data changes. A second mistake is pushing per-row formatting, concatenation, or numeric casts into a Python or Lambda UDF when a built-in SQL function does the same work inline, which turns a free operation into a per-row round trip.

## How does Redshift handle query optimization for star schema data models?

The optimizer is tuned for star schemas: a large fact table surrounded by small dimension tables. If the fact is distributed on the foreign key and the dimensions are small and `DISTSTYLE ALL`, joins require no redistribution; sort keys on the fact (typically date first) allow range pruning; and the optimizer can push dimension filters into the fact scan.

```sql
-- Balanced star schema
create table analytics.fact_sales (
  sale_id bigint, date_key integer, product_key integer,
  store_key integer, net_amount numeric(12,2)
)
diststyle key distkey (product_key)
sortkey (date_key);

create table analytics.dim_product (
  product_key integer, category varchar(64), brand varchar(64)
)
diststyle all sortkey (product_key);

create table analytics.dim_store (
  store_key integer, region varchar(32), city varchar(64)
)
diststyle all sortkey (store_key);
```

Three decisions do most of the work on a star schema. First, distribution: put the fact's `DISTKEY` on the column used by the biggest joins, and let small, slowly changing dimensions use `DISTSTYLE ALL` so every slice holds a full copy and joins stay local. Second, sort: a compound sort key with the date column first lets range filters prune 1 MB blocks through zone maps. Third, statistics: the optimizer needs current ANALYZE data to estimate cardinality and pick join order. A star schema stops behaving when a low-cardinality distkey creates slice skew or a large dimension is copied to every slice.

```sql
-- Read the plan, then check the physical shape it was planned from
explain
select s.region, sum(f.net_amount)
from analytics.fact_sales f
join analytics.dim_store s on f.store_key = s.store_key
where f.date_key between 20261001 and 20261031
group by s.region;

select "table", diststyle, sortkey1, skew_rows, stats_off
from svv_table_info
where schema = 'analytics'
order by skew_rows desc;
```

Common mistake: choosing a fact `DISTKEY` on a low-cardinality column such as a status flag or a region, which pushes most rows onto a few slices and shows up as high `skew_rows`. Another is applying `DISTSTYLE ALL` to a dimension that is not small, which multiplies storage and load time by the slice count. A third is ordering a compound sort key by something other than the leading filter column, so the date range scan touches far more blocks than it needs.

## Can you schedule query executions in Amazon Redshift?

Yes. Options include Redshift query editor v2 scheduled queries, EventBridge Scheduler invoking the Redshift Data API, Step Functions with the Redshift integrations, AWS Glue triggers, and Airflow with the Redshift operators. The Data API is the modern, connectionless path.

```bash
# EventBridge Scheduler calling the Redshift Data API via a Lambda target
aws scheduler create-schedule \
  --name nightly-revenue-rollup \
  --schedule-expression 'cron(0 4 * * ? *)' \
  --flexible-time-window 'Mode=OFF' \
  --target '{"Arn":"arn:aws:lambda:us-east-1:123456789012:function:run-redshift-sql","RoleArn":"arn:aws:iam::123456789012:role/scheduler-role"}'
```

In [ ]:
# Lambda target: run the SQL and poll for completion
import boto3, time

client = boto3.client("redshift-data")
statement = client.execute_statement(
    WorkgroupName="analytics-wg",
    Database="analytics",
    Sql="refresh materialized view analytics.mv_sales_by_category;",
)["Id"]

while True:
    status = client.describe_statement(Id=statement)["Status"]
    if status in ("FINISHED", "FAILED", "ABORTED"):
        break
    time.sleep(2)
print(status)

Scheduling splits into two layers. The data-warehouse layer is the query editor v2 scheduler, which runs saved SQL on a cron expression, and materialized views with `AUTO REFRESH YES`, which Redshift refreshes when base table changes are detected. The orchestration layer is outside the warehouse: EventBridge Scheduler plus the Redshift Data API, Step Functions, Glue triggers, or Airflow. The Data API is connectionless and asynchronous, so a scheduler only needs permission to call it; the SQL still runs inside the cluster under the caller's identity, and long statements are polled with `describe_statement`.

```sql
-- Warehouse-side scheduling: let Redshift refresh when base data changes
create materialized view analytics.mv_sales_by_category
auto refresh yes
diststyle even
sortkey (category)
as
select p.category, sum(f.net_amount) as revenue
from analytics.fact_sales f
join analytics.dim_product p on f.product_key = p.product_key
group by p.category;

-- Force a refresh when a downstream job needs a guaranteed fresh copy
refresh materialized view analytics.mv_sales_by_category;
```

Common mistake: scheduling operations without an overlap guard. A slow nightly job that runs past the next cron tick stacks runs that fight over the same tables, and `AUTO REFRESH` can lag the base table, so a report may read a view that is seconds to minutes behind. Set the schedule timezone explicitly and give every job an idempotent key or a lock so a retry does not double-load data.

## What is the purpose of the Redshift WLM (Workload Management) configuration?

WLM (Workload Management) divides a cluster's memory and query slots into queues, so that ETL, BI dashboards, and ad-hoc queries do not starve each other. Automatic WLM lets Redshift manage concurrency and memory per queue using query priorities and monitoring rules; manual WLM gives you explicit slot counts and memory percentages. Concurrency scaling and short query acceleration are configured alongside WLM.

```bash
# Automatic WLM with two named queues and priorities
aws redshift modify-cluster \
  --cluster-identifier analytics-cluster \
  --wlm-json-configuration '[
    {"name":"bi","auto_wlm":true,"priority":"highest","query_group":"bi","query_concurrency_scaling":"auto"},
    {"name":"etl","auto_wlm":true,"priority":"low","query_group":"etl"},
    {"name":"default","auto_wlm":true,"priority":"normal"}
  ]'
```

```sql
-- Route a query to a queue with a query group label
set query_group to 'etl';
insert into analytics.fact_sales
select * from analytics.stg_sales;
reset query_group;
```

Automatic WLM decides queue memory and concurrency from demand, guided by each queue's priority and its query monitoring rules. Manual WLM fixes slots and memory percentage per queue, which is predictable but wastes capacity when the workload mix changes. Query monitoring rules are the operational control: a rule can log, change a query's priority, or abort it when it exceeds a threshold such as elapsed time or rows returned. Short query acceleration gives fast queries an express path so they skip the main queues. Concurrency scaling, capped by `max_concurrency_scaling_clusters`, adds transient clusters for read-heavy bursts.

```sql
-- Label a session so it lands in the matching queue
set query_group to 'bi';
select s.region, sum(f.net_amount)
from analytics.fact_sales f
join analytics.dim_store s on f.store_key = s.store_key
group by s.region;
reset query_group;

-- Inspect the configured service classes and how a query was handled
select service_class, name, num_query_tasks, query_working_mem
from stv_wlm_service_class_config
order by service_class;

select query_id, service_class_name, compute_type, short_query_accelerated
from sys_query_history
where query_id = pg_last_query_id();
```

Common mistake: looking for an `ALTER WORKLOAD MANAGEMENT` SQL statement. WLM is cluster configuration and is changed through the console, the `wlm_json_configuration` parameter group, or the Redshift API, with the WLM system views used to observe the result. Another mistake is assigning manual memory percentages per queue that do not sum to 100, which makes the cluster reject or reshape the config, and labelling sessions with a query group that no queue matches so they fall through to the default queue.

## Can you create Redshift tables from existing Amazon S3 data?

Yes. You can create an external table over the S3 data (Spectrum) and query it in place, or create a native table populated from it with `CREATE TABLE AS` or `COPY`. The choice is about whether the data should be local (fast, costs storage) or remote (cheap, slower, no load).

```sql
-- Remote: external table over S3 Parquet
create external table spectrum_lake.orders (
  order_id varchar(64),
  order_date date,
  amount decimal(12,2)
)
partitioned by (dt varchar(10))
stored as parquet
location 's3://acme-lake/orders/';

-- Local: materialize a copy into Redshift for fast BI
create table analytics.orders_2026
diststyle key distkey (order_id)
sortkey (order_date)
as
select order_id, order_date, amount
from spectrum_lake.orders
where dt between '2026-01-01' and '2026-12-31';
```

An external table is metadata only: the rows stay in S3 and Spectrum reads them through a separate fleet, so there is no load time and no local storage cost, but you pay per terabyte scanned. A native table created with `CREATE TABLE AS` or `COPY` pays storage once and then queries from local slices with zone maps, sort keys, and encodings. The practical rule is to land frequently queried, repeatedly joined data locally, and leave cold or huge append-only data external. External tables are read-only from Redshift, and Spectrum relies on the `numRows` table property for size estimates because ANALYZE does not collect external statistics.

```sql
-- External schema and table over partitioned S3 data
create external schema spectrum_lake
from data catalog
database 'acme_lake'
iam_role 'arn:aws:iam::123456789012:role/redshift-spectrum-role'
create external database if not exists;

-- New S3 prefixes need an explicit partition registration
alter table spectrum_lake.orders
add partition (dt = '2026-10-01')
location 's3://acme-lake/orders/dt=2026-10-01/';

-- Local landing table with distribution and sort for BI
create table analytics.orders_2026
diststyle key distkey (order_id)
sortkey (order_date)
as
select order_id, order_date, amount
from spectrum_lake.orders
where dt >= '2026-01-01';
```

Common mistake: treating an external table as writable. `INSERT`, `UPDATE`, and `DELETE` do not apply to it; write the new files to S3 and either add the partition or rely on partition pruning. A second mistake is omitting `numRows`, which leaves the optimizer with a poor size estimate and can produce a bad join plan. A third is scanning an entire S3 prefix on every query because new partitions were never registered, which inflates the per-terabyte Spectrum bill.

## What is the purpose of the Redshift COPY command's TRUNCATECOLUMNS option?

`TRUNCATECOLUMNS` truncates string values that are longer than the target column definition instead of failing the load. It is a pragmatic option for messy source data, but it silently loses data, so use it only where the target length is a deliberate contract and the truncation is acceptable.

```sql
copy analytics.stg_orders
from 's3://acme-lake/raw/orders_2026-10-04.csv'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
csv
ignoreheader 1
truncatecolumns
maxerror 0;

-- Audit: how often did truncation actually bite?
select query, filename, line_number, colname, raw_line, err_reason
from stl_load_errors
where err_code = 1204
order by starttime desc
limit 20;
```

`TRUNCATECOLUMNS` applies only to character columns. A value longer than the declared CHAR or VARCHAR length is cut to fit and the row still loads; without it the same value raises a load error that counts against MAXERROR and, at MAXERROR 0, fails the whole COPY. It does not help with numeric overflow, date parse failures, or bad delimiters - those still error. The cut happens per field on the way in, so the column ends up holding exactly the declared length and the discarded characters are gone, with no warning row in the load output. Treat it as data loss you have chosen to accept, not a cleaning step.

```sql
-- Land wide, measure, then cast into the contracted column
create table analytics.stg_orders_wide (
  order_id      varchar(64),
  customer_note varchar(4000)
);

copy analytics.stg_orders_wide
from 's3://acme-lake/raw/orders_2026-10-04.csv'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
csv
ignoreheader 1
maxerror 0;

-- How much data would TRUNCATECOLUMNS have silently dropped?
select count(*) as too_long,
       max(length(customer_note)) as longest
from analytics.stg_orders_wide
where length(customer_note) > 200;
```

Common mistake: turning on `TRUNCATECOLUMNS` together with `MAXERROR 0` and treating the green load as clean. The statement succeeds, but every over-length value was shortened, so joins on a truncated key stop matching and string comparisons silently disagree. Another is assuming it also covers numeric or date errors: an overflowing numeric still fails the COPY. Before trusting such a load, compare `max(length(col))` against the target width in a wide staging table, or review the load error rows, and record the decision in the pipeline.

## Can you use Redshift with AWS PrivateLink for private network access?

Yes. Redshift supports VPC endpoints powered by AWS PrivateLink, so clients reach the cluster or Serverless workgroup over a private endpoint inside their VPC instead of the public internet. This keeps traffic on the AWS network, avoids public IP allow-lists, and simplifies cross-account access when combined with endpoint policies.

```bash
# Create an interface VPC endpoint for Redshift in the client VPC
aws ec2 create-vpc-endpoint \
  --vpc-id vpc-0abc1234 \
  --service-name com.amazonaws.us-east-1.redshift \
  --vpc-endpoint-type Interface \
  --subnet-ids subnet-0abc1234 subnet-0def5678 \
  --security-group-ids sg-0abc1234
```

Pair PrivateLink with security groups that allow only the application subnets and TLS-required connections.

PrivateLink builds an interface endpoint, which is an elastic network interface with a private IP in each subnet you select. The client connects to that IP and traffic never leaves the AWS network. Turning on private DNS makes the cluster's public hostname resolve to the endpoint IPs inside the VPC, so existing connection strings keep working. One important split: the endpoint carries Redshift client traffic only. COPY and UNLOAD from S3, and Spectrum scans, still need a route to S3 - a gateway VPC endpoint for S3 or a NAT gateway - or those jobs fail even though the cluster itself is reachable.

```bash
# Turn on private DNS so the cluster hostname resolves to the endpoint
aws ec2 modify-vpc-endpoint \
  --vpc-endpoint-id vpce-0abc1234 \
  --private-dns-enabled

# S3 needs its own gateway endpoint; the Redshift interface endpoint does not carry it
aws ec2 create-vpc-endpoint \
  --vpc-id vpc-0abc1234 \
  --service-name com.amazonaws.us-east-1.s3 \
  --vpc-endpoint-type Gateway \
  --route-table-ids rtb-0abc1234
```

Common mistake: enabling PrivateLink and assuming all warehouse traffic is now private. The interface endpoint covers SQL connections; COPY, UNLOAD, and Spectrum egress to S3 uses a separate path and needs an S3 gateway endpoint or NAT, so those jobs fail with an S3 access or timeout error. A second mistake is opening the endpoint security group to 0.0.0.0/0 or to the whole VPC instead of the application security groups, which rebuilds the exposure PrivateLink removed. Allow port 5439 from the client security groups, and place the endpoint in more than one subnet so an Availability Zone outage does not cut access.

## How does Redshift handle query optimization for time-series data?

Time-series queries are range queries on a timestamp or date, so the decisive design choice is the sort key: put the time column first in a compound sort key so zone maps can skip whole blocks outside the range. Partition the data by time (natively or in Spectrum external tables) and prune with predicates. Materialized views on the daily/weekly grain and Spectrum for cold history keep the hot path small.

```sql
create table analytics.fact_events (
  event_ts   timestamp encode az64,
  event_date date      encode az64,
  device_id  bigint    encode az64,
  metric     numeric(12,4)
)
diststyle even
compound sortkey (event_date, device_id);

-- Range scan prunes blocks using the leading sort key
select device_id, avg(metric) as avg_metric
from analytics.fact_events
where event_date between date '2026-10-01' and date '2026-10-07'
group by device_id;
```

The sort key does the heavy lifting. With a compound key like `(event_date, device_id)`, a filter on `event_date` alone prunes 1 MB blocks through zone maps; a filter on `device_id` alone gets no pruning because it is not the leading column. That is why the timestamp or date usually goes first. Distribution is the second lever: `DISTSTYLE EVEN` spreads rows and parallelizes the aggregate, while a `DISTKEY` on the entity you group or join by keeps that work local to a slice. Encodings matter too - AZ64 on timestamps and dates compresses well and scans faster than RAW blocks.

```sql
-- Time-ordered loads keep new rows behind existing ones, so the sort holds
copy analytics.fact_events
from 's3://acme-lake/curated/fact_events/dt=2026-10-07/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
compupdate preset
statupdate on;

-- Late-arriving rows make the table unsorted; this is the cheap fix
vacuum sort only analytics.fact_events;

-- Refresh the estimates used for join order and aggregate sizing
analyze analytics.fact_events;
```

Common mistake: wrapping the sort column in a function. `where date_trunc('day', event_ts) = date '2026-10-07'` and `where event_ts::date = date '2026-10-07'` both hide the raw column from zone maps, so the scan touches every block in the range. Write a half-open range on the raw column instead: `where event_ts >= timestamp '2026-10-07 00:00:00' and event_ts < timestamp '2026-10-08 00:00:00'`. A second mistake is putting `device_id` first in the compound key when the dominant filter is time, which gives up the range pruning that makes time-series scans cheap.

## Can you use Redshift to query and analyze semi-structured data formats like JSON or Avro?

Yes, in two ways. Native `SUPER` columns store semi-structured data in Redshift and are queried with PartiQL and `json_parse`, which is ideal for flexible event attributes you query often. Redshift Spectrum queries JSON, Avro, ORC, Parquet, and Iceberg files in S3 without loading them.

```sql
-- Native SUPER column with PartiQL access
create table analytics.events_super (
  event_id   varchar(64),
  event_ts   timestamp,
  properties super
)
diststyle even sortkey (event_ts);

select
  event_id,
  properties.event_type::varchar as event_type,
  properties.campaign.campaign_id::varchar as campaign_id
from analytics.events_super
where event_ts >= timestamp '2026-10-01 00:00:00';
```

```sql
-- Spectrum external table over JSON, with a serializer/deserializer
create external table spectrum_lake.events_json (
  event_id varchar(64),
  event_ts varchar(32),
  properties varchar(4096)
)
row format serde 'org.openx.data.jsonserde.JsonSerDe'
stored as textfile
location 's3://acme-lake/events-json/';
```

A SUPER column stores semi-structured values in a compact binary form and is schemaless, so you do not declare nested attributes up front. You navigate it with PartiQL dot and bracket notation and cast scalars with `::varchar` and similar. You can land JSON directly into SUPER with `COPY ... FORMAT AS JSON 'auto'`, or parse a string column at query time with `json_parse`; `can_json_parse` tells you whether a string would parse without raising an error. Because the shape is known only at read time, the optimizer cannot use column statistics on nested attributes, so filter on ordinary columns, especially the sort key, to keep scans narrow.

```sql
-- Land raw JSON events directly into a SUPER column
copy analytics.events_super
from 's3://acme-lake/events-json/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as json 'auto'
compupdate off;

-- Guard against malformed rows when parsing a varchar payload
select event_id,
       json_parse(payload).properties.event_type::varchar as event_type
from analytics.events_raw
where can_json_parse(payload);
```

Common mistake: keeping JSON in a `varchar` column and digging it out with string functions and regexes, which is brittle and hides the structure from tooling. Another is filtering on a nested attribute - `where properties.event_type::varchar = 'click'` - and expecting zone-map pruning; nested access is evaluated per row, so the scan still reads the whole range. PartiQL attribute names are case-sensitive, so `properties.Campaign` is not `properties.campaign`, and mixed-case keys return nulls instead of an error. For files, prefer the matching SerDe for the format rather than casting text.

## What is the purpose of the Redshift COPY command's COMPUPDATE option?

`COMPUPDATE` controls whether `COPY` chooses compression encodings for the loaded columns: `ON`/`TRUE` analyzes and applies the best encoding, `PRESET` applies the default without the analysis pass, and `OFF` leaves encodings untouched. `ON` is the usual choice for a table's initial load, and `PRESET` is useful when the extra analysis pass would slow a frequent incremental load without meaningfully changing encodings.

```sql
-- Initial load: pick good encodings
copy analytics.fact_sales
from 's3://acme-lake/curated/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
compupdate on;

-- Frequent small load: keep existing encodings, skip the analysis pass
copy analytics.fact_sales
from 's3://acme-lake/curated/fact_sales_inc/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
compupdate preset;
```

`COMPUPDATE ON` runs an analysis pass over data COPY already read, then writes the chosen encodings. That pass costs time and I/O, which is why a large initial load takes noticeably longer with it on than with `PRESET`. The value shows up later: a well-encoded column scans fewer 1 MB blocks and compresses far better than RAW. If you want the recommendations without loading, run `ANALYZE COMPRESSION` against the table - it reports suggested encodings and does not change any data. You can change one column later with `ALTER TABLE ... ALTER COLUMN c ENCODE ZSTD`, though that rewrites the column.

```sql
-- Inspect recommended encodings without changing data
analyze compression analytics.fact_sales;

-- Load with defaults, then set encodings on hot columns deliberately
copy analytics.fact_sales
from 's3://acme-lake/curated/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
compupdate preset
statupdate on;

alter table analytics.fact_sales alter column customer_note encode zstd;
```

Common mistake: using `COMPUPDATE PRESET` on the first load of a new table. PRESET applies default encodings without analyzing the data - numeric and date columns get AZ64, char and varchar get LZO, and sort key, boolean, and float columns stay RAW - which can be a poor fit for a wide, high-cardinality string column. Another is expecting `COMPUPDATE ON` to re-encode rows already in the table; it does not rewrite existing data. Finally, the analysis pass lengthens every load, so turning it on for small frequent batches buys little and adds latency.

## Can you use Redshift cross-region replication for disaster recovery?

Yes. Cross-region snapshot copy replicates snapshots from the primary region to a destination region on a schedule so you can restore there after a regional failure. It is asynchronous, and the copies are encrypted with the destination KMS key. For a hot standby, restore the latest cross-region snapshot into a cluster in the target region and cut over; full active-active multi-region writes are not a native Redshift feature.

```bash
aws redshift enable-snapshot-copy \
  --cluster-identifier analytics-cluster \
  --destination-region us-west-2 \
  --retention-period 7

aws redshift describe-cluster-snapshots \
  --cluster-identifier analytics-cluster \
  --snapshot-type automated \
  --query 'Snapshots[?SnapshotCreateTime>`2026-09-01`].[SnapshotIdentifier,SnapshotCreateTime]'
```

Snapshot copy is asynchronous and snapshot-based, so the recovery point objective is the snapshot interval, not continuous. Redshift takes automated snapshots roughly every 8 hours and again after about 5 GB of change, and copies those to the destination region shortly after they are created. If the primary region fails, you restore the latest copied snapshot into a new cluster in the destination region, then repoint clients. Destination retention is set separately with `--manual-snapshot-retention-period`, so copies can be kept longer than the source window. Cross-account copies need a snapshot copy grant and a KMS key the destination account can use.

```bash
# Destination account shares a KMS key with the source account
aws redshift create-snapshot-copy-grant \
  --snapshot-copy-grant-name acme-dr-grant \
  --kms-key-id alias/acme-redshift-dr

# Source account authorizes the grant, then enables the copy
aws redshift authorize-snapshot-copy-grant \
  --snapshot-copy-grant-name acme-dr-grant

aws redshift enable-snapshot-copy \
  --cluster-identifier analytics-cluster \
  --destination-region us-west-2 \
  --retention-period 7 \
  --manual-snapshot-retention-period 14 \
  --snapshot-copy-grant-name acme-dr-grant
```

Common mistake: treating cross-region copy as near-zero RPO. The copy follows the snapshot cadence, so a regional failure can lose every write since the last copied snapshot - plan for hours, not seconds. A second mistake is assuming a destination cluster is running and queryable; the feature places snapshots in the destination region, and you restore a cluster from one before it serves traffic. Also check the destination KMS key and grant, because a bad key policy stops copies from landing, so monitor snapshot copy status rather than assuming it works.

## How does Redshift handle data distribution when a new node is added to the cluster?

A resize changes the number of slices. Redshift redistributes each table according to its distribution style: `KEY` rows hash to the new slice count, `EVEN` rows rebalance round-robin, and `ALL` tables are copied to the new slices. Elastic resize streams the movement with a short reboot; classic resize copies through a temporary cluster and requires a read-only window.

```bash
# After a resize, verify distribution health
psql -h analytics-cluster.abc123.us-east-1.redshift.amazonaws.com -U admin -d analytics \
  -c "select schema, \"table\", diststyle, skew_rows from svv_table_info order by skew_rows desc limit 10;"
```

A resize changes the slice count, and distribution is defined against the slice count. Adding one node to a cluster of `ra3.4xlarge` nodes adds 12 slices, since each of those nodes has 12, so a `KEY` table's hash modulo changes and rows move between slices. `EVEN` tables rebalance round-robin, and `ALL` tables are copied to every new slice, which grows their footprint in proportion to slice count. `AUTO` tables are re-evaluated, and one that was `ALL` while small can be reconsidered. The movement is real work: elastic resize streams it with a brief reboot, while classic resize copies through a temporary cluster and the source is read-only.

```sql
-- How many slices does the resized cluster have now?
select count(*) as slice_count from stv_slices;

-- Which tables are still skewed after redistribution settled?
select schema, "table", diststyle, sortkey1, skew_rows, stats_off
from svv_table_info
where skew_rows > 1.5
order by skew_rows desc
limit 20;

-- Flatten a table whose key distribution no longer fits the slice count
alter table analytics.fact_sales alter diststyle even;
vacuum analytics.fact_sales;
```

Common mistake: assuming a resize is finished when the cluster is back online. Redistribution can continue in the background and `skew_rows` stays high until it settles, so check `SVV_TABLE_INFO` before blaming query plans. A second mistake is resizing during a load window: classic resize puts the cluster in a read-only state, so ETL writes fail. Third, `DISTSTYLE ALL` on a dimension that is small relative to one slice gets much more expensive as nodes are added, because every new slice receives a full copy. Recheck distribution styles after every resize, not once.

## Can you use Redshift with AWS Identity and Access Management (IAM) for access control?

Yes, and at several layers: IAM controls the Redshift APIs (create, resize, snapshot, restore); IAM roles and policies grant Spectrum/S3/Glue/Lambda access; database roles control query privileges; IAM database authentication lets users connect with short-lived credentials; and `GetClusterCredentials` issues temporary database credentials without storing passwords.

```sql
-- Database-side roles and grants
create role bi_reader;
grant usage on schema analytics to role bi_reader;
grant select on analytics.mv_sales_by_category to role bi_reader;
grant role bi_reader to user report_svc;
```

```sql
-- IAM role attached to the cluster is used by COPY/Spectrum, not by users
copy analytics.fact_sales
from 's3://acme-lake/curated/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet;
```

The layers are easy to conflate. IAM controls the Redshift control plane - creating clusters, taking snapshots, resizing. The IAM role attached to the cluster is what COPY, UNLOAD, and Spectrum use to read S3 or call Lambda; it is the cluster's identity, not the user's. Inside the database, roles and grants decide who reads which table. IAM database authentication lets a user present a short-lived token (about 15 minutes) instead of a password, and `GetClusterCredentials` mints temporary database credentials for a named `DbUser`, optionally adding `DbGroups`, for a bounded duration. Prefer roles over per-user grants so access stays auditable.

```sql
-- Grant through roles so membership is the audit trail
create role bi_reader;
grant usage on schema analytics to role bi_reader;
grant select on all tables in schema analytics to role bi_reader;

-- Future tables in the schema inherit the same grant
alter default privileges in schema analytics
  grant select on tables to role bi_reader;

grant role bi_reader to user report_svc;
grant role bi_reader to role analyst;

-- Verify what a role can actually see
select has_schema_privilege('bi_reader', 'analytics', 'usage') as schema_ok,
       has_table_privilege('bi_reader', 'analytics.fact_sales', 'select') as table_ok;
```

Common mistake: handing users the cluster's IAM role or storing long-lived passwords because IAM database authentication feels awkward. The role is the cluster identity, and sharing it erases the per-user audit trail; short-lived tokens and `GetClusterCredentials` exist so credentials are not stored. Another is granting privileges to individual users, which drifts as people change teams - grant to roles and manage membership. Also remember that data access and API access are separate: a user who can read the warehouse still cannot create a cluster unless their IAM policy allows it.

## What is the purpose of the Redshift ANALYZE command's DELETE option?

`ANALYZE` has no `DELETE` option - that is a common mix-up with `VACUUM`. `ANALYZE` only collects table and column statistics; `VACUUM DELETE ONLY` is the command that reclaims the space occupied by rows marked for deletion. If statistics are stale or misleading, re-run `ANALYZE`; if storage is bloated by deletes, run `VACUUM DELETE ONLY`.

```sql
-- ANALYZE collects statistics; there is no ANALYZE ... DELETE
analyze analytics.fact_sales;

-- The DELETE option belongs to VACUUM, which reclaims deleted-row space
vacuum delete only analytics.fact_sales;
```

The two commands solve different problems. `ANALYZE` refreshes the statistics the optimizer uses for cardinality estimates and join order, and it is incremental, so it touches the columns that changed rather than rescanning everything. `VACUUM DELETE ONLY` reclaims the space left by rows marked for deletion; until it runs, that space stays allocated and the table keeps growing. Statistics staleness has its own signal: `SVV_TABLE_INFO.stats_off` is 0 when statistics are current and 100 when they are badly out of date, so check it before deciding which command you actually need.

```sql
-- Are the statistics the problem, or is it deleted-row bloat?
select "table", stats_off, unsorted, tbl_rows, size, pct_used
from svv_table_info
where schema = 'analytics'
order by stats_off desc;

-- Refresh statistics only
analyze analytics.fact_sales;

-- Reclaim deleted-row space without re-sorting everything
vacuum delete only analytics.fact_sales to 75 percent;
```

Common mistake: running `VACUUM DELETE ONLY` because a query got slow. If the plan is wrong because estimates are stale, reclaiming deleted-row space changes nothing; run `ANALYZE` instead. The reverse also happens: running `ANALYZE` to shrink a table that has accumulated deleted rows, which it cannot do. Two constraints bite here: `VACUUM` cannot run inside an explicit transaction block, and `VACUUM` and `ALTER TABLE` cannot run concurrently on the same table, so chained maintenance scripts block or error. The `TO 75 PERCENT` form stops once the table is 75 percent sorted, which is cheaper than the default 95 percent threshold but leaves more unsorted rows.

## Can you use Redshift with AWS Glue for data cataloging and ETL?

Yes. Glue Data Catalog can be the metastore behind Spectrum external schemas for S3 and Iceberg data, and Glue jobs (Spark) plus Glue DataBrew can transform and write the Parquet that Redshift loads. Glue crawlers keep the catalog in sync with new S3 partitions, and Redshift can also expose its own tables into Glue for Athena-style access in some configurations.

```sql
-- Spectrum external schema backed by the Glue Data Catalog
create external schema spectrum_lake
from data catalog
database 'acme_lake'
iam_role 'arn:aws:iam::123456789012:role/redshift-spectrum-role'
create external database if not exists;
```

```bash
# Run a Glue job that writes curated Parquet for Redshift COPY
aws glue start-job-run --job-name curate_orders --arguments '{"--dt":"2026-10-04"}'
```

The Glue Data Catalog is a Hive-compatible metastore, and a Spectrum external schema is a thin pointer into it. Reads go through the separate Spectrum fleet rather than the cluster's compute nodes, so scanning S3 does not consume local SSD cache or slice memory. Partition columns are pruned the same way Hive prunes them, which means an up-to-date catalog is what makes partition pruning possible. A crawler rescans a prefix and adds partitions it has not seen, so schedule it after each ETL drop rather than only once.

```sql
create external table spectrum_lake.fact_orders (
  order_id   bigint,
  order_date date,
  amount     numeric(12,2)
)
partitioned by (dt varchar(10))
stored as parquet
location 's3://acme-lake/curated/fact_orders/'
table properties ('numRows'='250000000');

-- Register one new S3 partition without a full crawler run
alter table spectrum_lake.fact_orders
add if not exists partition (dt='2026-10-04')
location 's3://acme-lake/curated/fact_orders/dt=2026-10-04/';
```

Common mistake: assuming `analyze` maintains statistics on an external table. It does not; the optimizer leans on the `numRows` value in TABLE PROPERTIES, so a missing or stale number can underestimate a huge table and produce a bad join order. Set `numRows` deliberately and update it as the lake grows. Also remember that until the catalog has a partition, queries against it silently return no rows from that date.

## How does Redshift handle query optimization for dimensionally modeled data warehouses?

For a dimensional model, the optimizer exploits distribution and sort keys to avoid redistribution, uses zone maps to prune fact blocks by date, and pushes dimension filters into the fact scan. The classic advice still holds: distribute the fact on the busiest foreign key, give small dimensions `DISTSTYLE ALL`, sort the fact by the most common date filter, and keep dimensions sorted by their primary key for merge joins.

```sql
-- Common dimension-filtered fact query
select d.calendar_month, p.brand, sum(f.net_amount) as revenue
from analytics.fact_sales f
join analytics.dim_date d on f.date_key = d.date_key
join analytics.dim_product p on f.product_key = p.product_key
where d.calendar_year = 2026 and p.category = 'electronics'
group by d.calendar_month, p.brand;
```

In a star join the planner wants each fact row to meet its dimension row on the slice that already holds it. If the fact is distributed on the dimension's key, each slice joins locally with no network movement. A genuinely small dimension is best as `DISTSTYLE ALL`, because every slice then owns a full copy and no broadcast is needed. A dimension too large to replicate is distributed on its own primary key, and the fact is distributed on the matching foreign key. Merge joins need both inputs sorted, which is why dimension primary keys are usually also the sort key. Zone maps reward a sort key on whatever column the range filter uses, usually a date key.

```sql
explain
select d.calendar_month, sum(f.net_amount) as revenue
from analytics.fact_sales f
join analytics.dim_date d on f.date_key = d.date_key
where f.date_key between 20260901 and 20260930
  and d.calendar_year = 2026
group by d.calendar_month;

select "table", diststyle, sortkey1, skew_rows, stats_off
from svv_table_info
where "table" in ('fact_sales', 'dim_date');
```

Common mistake: giving `DISTSTYLE ALL` to a dimension that is not actually small. Every slice stores its own copy, so load time and storage multiply with the slice count, and a replicated table that grows becomes a maintenance problem. Check the row count before choosing ALL, and re-check `skew_rows` on the fact after picking a new distribution key; a skewed key turns a local join into one hot slice.

## Can you restore a Redshift cluster to a different cluster?

Yes. A snapshot is independent of the cluster that produced it, so you can restore it into a new cluster - with a different name, node type, node count, or even a different subnet/AZ configuration. This is the standard way to promote a staging snapshot to production, clone a production database for testing, or recover after a failed migration.

```bash
aws redshift restore-from-cluster-snapshot \
  --cluster-identifier analytics-test-clone \
  --snapshot-identifier analytics-prod-2026-10-04 \
  --node-type ra3.4xlarge \
  --number-of-nodes 2 \
  --no-publicly-accessible
```

The restored cluster keeps database contents, users, and cluster metadata from the snapshot; you still need to re-point applications, DNS, and IAM policies.

A restore always lands in a new cluster, so the source keeps running and the two can be compared side by side before any cutover. The target can use a different node type or node count because Redshift redistributes the data while restoring; a much smaller target therefore takes longer and may run short on disk. When only one table is damaged, `restore-table-from-cluster-snapshot` brings that single table back into an existing cluster, which is far cheaper than rebuilding the whole warehouse.

```bash
aws redshift restore-table-from-cluster-snapshot \
  --cluster-identifier analytics-prod \
  --snapshot-identifier analytics-prod-2026-10-04 \
  --source-database-name analytics \
  --source-table-name fact_sales \
  --target-database-name analytics_recovery \
  --new-table-name fact_sales_recovered
```

Common mistake: pointing applications at the restored cluster and forgetting everything tied to the old one. The endpoint and DNS name are new, and IAM roles, security groups, the parameter group, and scheduled jobs all have to be re-checked. An encrypted snapshot also needs access to the same KMS key in the destination region, so a restore that looks like a permission failure is often a key-policy failure.

## What is the purpose of the Redshift VACUUM command's REINDEX option?

`VACUUM REINDEX` rebuilds the interleaved sort-key index for tables defined with `INTERLEAVED SORTKEY`. It re-analyzes the value distribution, drops the old interleaved index, and rebuilds it, which restores the multi-column pruning benefit that degrades after many loads and updates. It is expensive, so schedule it rather than running it casually.

```sql
vacuum reindex analytics.fact_events;
analyze analytics.fact_events;
```

For compound sort key tables, `VACUUM SORT ONLY` is the right tool; `VACUUM REINDEX` is specifically for interleaved keys.

REINDEX runs in phases. It first analyzes the current distribution of values in the interleaved columns to decide their relative weight, then drops the old interleaved index and rebuilds it. That extra analysis is why it takes far longer than `VACUUM SORT ONLY` and why it belongs in a maintenance window. Interleaved keys only pay off when several low-cardinality columns all appear in filters with equal importance. If that is no longer true, the table can be changed to a compound sort key; Redshift does not support changing a compound table back to interleaved.

```sql
-- Interleaved tables need REINDEX to restore multi-column pruning
vacuum reindex analytics.fact_events to 75 percent;

-- Compound sort keys only need a re-sort
vacuum sort only analytics.fact_sales;

select "table", sortkey1, sortkey_num, unsorted, vacuum_sort_benefit
from svv_table_info
where "table" in ('fact_events', 'fact_sales');
```

Common mistake: running `vacuum reindex` against a table with a compound sort key and expecting it to fix unsorted rows. REINDEX rebuilds the interleaved index only; compound tables are repaired with `vacuum sort only`. Also remember that VACUUM cannot run inside a transaction block and cannot run on a table at the same time as an `alter table` on that table.

## Can you enable audit logging in Amazon Redshift?

Yes. Redshift writes three audit log streams: connection log (authentication and connection attempts), user log (DDL and DCL statements), and user activity log (every query before and after processing). You can deliver them to Amazon S3 and/or CloudWatch Logs, and each can be enabled independently.

```bash
aws redshift enable-logging \
  --cluster-identifier analytics-cluster \
  --bucket-name acme-redshift-audit \
  --s3-key-prefix audit/ \
  --logging-enabled true
```

```sql
-- Alternatively, the database-level audit views
select * from stl_connection_log order by record_time desc limit 20;
select * from stl_ddltext order by starttime desc limit 20;
select * from stl_userlog order by starttime desc limit 20;
```

Audit logs support compliance (PCI, HIPAA, SOC 2) and incident investigation; store them in a bucket with Object Lock or lifecycle rules that match your retention policy, and restrict who can read them.

The three streams answer different questions. The connection log captures connect and disconnect events plus authentication outcome, which is what you read during an access investigation. The user log captures DDL and DCL changes such as CREATE, ALTER, and GRANT that alter the warehouse structure or permissions. The user activity log captures every query text before and after processing, which is the heaviest stream and the one most likely to contain sensitive literal values. Each stream is enabled independently, so a cluster can keep the cheap ones on and turn the query stream off if volume or retention is a concern.

```bash
aws redshift describe-logging-status \
  --cluster-identifier analytics-cluster

aws redshift enable-logging \
  --cluster-identifier analytics-cluster \
  --bucket-name acme-redshift-audit \
  --s3-key-prefix audit/ \
  --logging-enabled true
```

Common mistake: enabling logging and never setting a lifecycle rule on the destination bucket. Audit data grows without bound and query text can contain sensitive values, so retention and read access are part of the design, not an afterthought. Do not treat the `stl_*` views as the compliance record either; those views are trimmed periodically, while the delivered S3 and CloudWatch streams are the durable copy.

## How does Redshift handle query optimization for complex join operations?

Redshift can execute nested-loop, hash, and merge joins, and the optimizer picks based on estimated cardinality and whether inputs are already colocated or sorted. For large joins it prefers hash joins with redistribution or broadcast; for sorted inputs it can use merge joins without a sort; nested loops only make sense for tiny outer inputs. `EXPLAIN` shows the chosen method and the movement steps.

```sql
explain
select f.sale_id, c.customer_name
from analytics.fact_sales f
join analytics.dim_customer c on f.customer_id = c.customer_id
where f.date_key = 20261004;
```

Symptoms to look for: `XN_Nested_Loop` on a large table, `DS_DIST_BOTH` on both sides of a big join, or `DS_BCAST_INNER` broadcasting a large inner table. Fixes are distribution-key alignment, `DISTSTYLE ALL` for genuinely small dimensions, and filtering before the join.

The planner estimates rows per join input from statistics and picks movement from that estimate. A small estimate can make it broadcast the inner side with `DS_BCAST_INNER` to every slice; a large estimate can make it redistribute both sides with `DS_DIST_BOTH`. When both inputs are distributed on the join key the join is `DS_DIST_NONE` and no rows cross the network at all. When both inputs are already sorted on the join key it can choose a merge join and skip the sort entirely. `XN_Nested_Loop` is only sensible when the outer input is tiny, so seeing it over a large table usually points at stale statistics.

```sql
-- Filter before the join so fewer rows are redistributed
explain
select f.sale_id, c.customer_name
from (
  select sale_id, customer_id
  from analytics.fact_sales
  where date_key = 20261004
) f
join analytics.dim_customer c on f.customer_id = c.customer_id
where c.state = 'TX';
```

Common mistake: wrapping a join column in a function or a cast and expecting predicate pushdown. An expression such as `upper(c.email)` or `f.customer_id::varchar` blocks zone-map pruning and can force a full scan, and a `limit` inside a subquery does not reduce the rows the join has to move. Keep join columns in their native type and put filters directly on base columns so the scan can skip blocks.

## Can you use Redshift with AWS Key Management Service (KMS) for encryption key management?

Yes. Redshift cluster encryption can use AWS KMS customer managed keys (or CloudHSM), with automatic key rotation. KMS is also what encrypts cross-region snapshot copies in the destination region, and what protects `COPY`/`UNLOAD` S3 objects with SSE-KMS. IAM policies govern who may use the key, which is the real access boundary.

```bash
aws redshift modify-cluster \
  --cluster-identifier analytics-cluster \
  --encrypted \
  --kms-key-id alias/redshift-prod

aws kms describe-key --key-id alias/redshift-prod \
  --query 'KeyMetadata.[KeyId,KeyState,KeyManager]'
```

Use separate keys per environment (dev, staging, prod) and grant decrypt only to the roles that operate the cluster.

A customer managed key never leaves KMS. Redshift calls KMS to wrap and unwrap the data keys that protect cluster volumes and snapshots, so the key policy plus IAM permissions on `kms:Decrypt` and `kms:CreateGrant` are the real boundary on who can read the data. Keys are regional, which is why a cross-region snapshot copy needs a usable key in the destination region and Redshift can create a destination key for that copy. Disabling or scheduling deletion of a key makes its snapshots unreadable even though the snapshot objects still exist.

```bash
aws kms create-grant \
  --key-id alias/redshift-prod \
  --grantee-principal arn:aws:iam::123456789012:role/redshift-prod-role \
  --operations Decrypt GenerateDataKey CreateGrant
```

Common mistake: letting a single key cover every environment and letting many roles use it. A development role with decrypt on the production key can read production data, and one accidental key deletion freezes every environment at once. Use one key per environment, keep the key policy narrow, and keep automatic rotation on so the backing key material is refreshed on schedule.

## What is the purpose of the Redshift COPY command's ACCEPTINVCHARS option?

`ACCEPTINVCHARS` lets `COPY` replace invalid UTF-8 byte sequences with a replacement character instead of failing. The default replacement is a question mark; you can specify another single ASCII character with `ACCEPTINVCHARS AS '^'`. It is the counterpart to `TRUNCATECOLUMNS` for messy external data.

```sql
copy analytics.stg_orders
from 's3://acme-lake/raw/orders_2026-10-04.csv'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
csv
ignoreheader 1
acceptinvchars as '^'
maxerror 50;
```

Because it silently alters values, log and review the affected rows; if encoding quality matters, fix the producer instead.

The substitution happens inside the COPY parser for CHAR and VARCHAR columns, so the replacement byte is what actually lands in the target column and the original value is gone. The substitute must be a single ASCII character, and the default when no `AS` clause is given is a question mark. It is intended for delimited and CSV input, and it is usually paired with `TRUNCATECOLUMNS` when fields are both dirty and too long. Keeping `MAXERROR 0` still fails the load on structural errors, so you keep a hard stop while tolerating encoding noise.

```sql
copy analytics.stg_events
from 's3://acme-lake/raw/events/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
csv
ignoreheader 1
acceptinvchars as '^'
truncatecolumns
maxerror 0;
```

Common mistake: choosing a replacement character that already appears in the data, such as a common separator, so substituted values become impossible to distinguish from real ones. Because the change is silent, load into a staging table, count how many values were altered, quarantine the affected rows, and only then publish the clean result into the dimension or fact table.

## Can you use Redshift with AWS CloudTrail for audit trail and compliance?

Yes. CloudTrail records Redshift control-plane API calls (create/modify/delete cluster, snapshot operations, IAM changes) as events, which you can deliver to S3 and query with Athena or CloudTrail Lake. It complements the database-level audit logs: CloudTrail tells you who changed the infrastructure, Redshift audit logs tell you who ran queries.

```bash
# Confirm that Redshift management events are being recorded
aws cloudtrail lookup-events \
  --lookup-attributes AttributeKey=EventSource,AttributeValue=redshift.amazonaws.com \
  --max-results 20
```

The two log families together give end-to-end evidence: API changes from CloudTrail, query activity from `STL`/`STV` audit views or S3 audit logs.

CloudTrail records control-plane API calls, not SQL. A Redshift management event carries eventName (CreateCluster, ModifyCluster, CreateClusterSnapshot, DeleteCluster, AuthorizeSnapshotAccess and similar), plus userIdentity, sourceIPAddress and requestParameters. Management events are recorded by default and stay in Event history for 90 days; a trail delivers them to S3 for long-term retention, and CloudTrail Lake keeps them queryable for a configured period. Parameter group and IAM changes arrive as their own event sources, which is what an auditor wants when the question is who changed the cluster configuration.

Query-level evidence comes from the database side. Use the durable SVL views and SYS_QUERY_HISTORY for recent activity, and remember that STL_* views are per-step and trimmed periodically, so they are not an archive.
```sql
select query, userid, starttime, elapsed, aborted
from stl_query
where starttime >= dateadd(day, -1, getdate())
order by starttime desc
limit 100;
```
Join STL_QUERYTEXT on the query id when you need the statement text. For retention beyond the view window, ship the database audit logs to S3 from the cluster parameter group and keep CloudTrail trails in the same account so the two timelines can be joined.

Common mistake: expecting CloudTrail to answer who ran a query or who deleted rows. CloudTrail captures cluster and API management activity only; SQL execution is not one of its events. For query attribution, enable database audit logging to S3 and use STL_QUERY, STL_QUERYTEXT, SVL_QUERY_REPORT or SYS_QUERY_HISTORY. Do not treat Event history as a permanent archive either. It holds 90 days; a trail or a Lake event data store is what makes the evidence durable.

## How does Redshift handle query optimization for complex aggregations and window functions?

Simple aggregates use hash or stream aggregation, and window functions require the data to be partitioned and sorted by the window definition (`PARTITION BY ... ORDER BY ...`). The optimizer chooses between sorting the input or using an existing sort key; when the window's `PARTITION BY`/`ORDER BY` matches a table's distribution and sort key, Redshift can avoid a re-sort and a redistribution. `QUALIFY` filters on window results without a subquery.

```sql
select
  customer_id,
  order_date,
  sum(amount) over (
    partition by customer_id
    order by order_date
    rows between 90 preceding and current row
  ) as trailing_90d_revenue,
  row_number() over (partition by customer_id order by order_date desc) as recency_rank
from analytics.fact_orders
where order_date >= date '2026-01-01'
qualify recency_rank <= 3;
```

A window whose `PARTITION BY` matches the distribution key keeps each customer's rows on one slice, which avoids a global reshuffle.

A window function needs its input ordered by PARTITION BY and then ORDER BY. The optimizer satisfies that with a sort, and when rows are not already on the slice that owns each partition it inserts a redistribution first. The cheap case is a table distributed on the partition key and sorted on the window order, which needs neither. Two windows with different specifications force two sorts, so reusing one window definition cuts plan cost. Note that QUALIFY filters after the window is computed, not before.

```sql
select
  customer_id,
  order_date,
  count(*) over w as orders_to_date,
  sum(amount) over w as revenue_to_date,
  max(amount) over w as biggest_order_so_far
from analytics.fact_orders
window w as (
  partition by customer_id
  order by order_date
  rows between unbounded preceding and current row
);
```
Three aggregates share one named window, so the input is partitioned and sorted once. Naming the window also documents the frame, which matters because ROWS counts physical rows while RANGE groups peers that share the same ORDER BY value.

Common mistake: assuming QUALIFY filters rows before the window is evaluated. It does not. The window runs over the rows left by WHERE, then QUALIFY removes rows, so ranking a huge table and keeping the top few still pays for ranking everything. Push the row count down in WHERE first. The second mistake is leaving the frame implicit: without ROWS or RANGE, the default is RANGE UNBOUNDED PRECEDING TO CURRENT ROW, which includes peer rows sharing the ORDER BY value and can inflate running totals.

## Can you use Redshift with Amazon QuickSight for data visualization and analytics?

Yes. QuickSight connects to Redshift directly or through a VPC connection, and can query live or import into SPICE. For large datasets, the common pattern is to expose curated tables and materialized views, use SPICE for dashboards that tolerate periodic refresh, and reserve direct query for near-real-time operational views.

```sql
-- A dashboard-friendly materialized view that QuickSight can query or import
create materialized view analytics.mv_kpi_daily
auto refresh yes
as
select
  date_trunc('day', order_date) as order_day,
  region,
  channel,
  count(distinct order_id) as orders,
  sum(amount) as revenue
from analytics.fact_orders
group by order_day, region, channel;
```

QuickSight's Redshift connector also supports row-level security pass-through, which lets one dashboard serve many tenants.

SPICE is QuickSight's in-memory store. Importing a dataset there moves query cost off the cluster and makes dashboards interactive, but it consumes SPICE capacity and only reflects data as of the last refresh. Direct query sends every interaction to Redshift, so a popular dashboard becomes real warehouse concurrency, which is where WLM queues and concurrency scaling matter. A VPC connection or PrivateLink keeps the traffic off the public endpoint. Use SPICE for wide, frequently viewed dashboards and direct query for small tables that must be current.

Row-level security can live in the warehouse instead of in each dataset, so one governed table serves many teams.
```sql
create row level security policy region_isolation on analytics.fact_orders
  using (region = current_user);

alter table analytics.fact_orders
  attach row level security policy region_isolation;
```
The policy is enforced only after it is attached and the user has SELECT on the table. A dataset built from an aggregate import can still weaken the guarantee, because rows are filtered for the refresh identity and then stored in SPICE. Decide which layer owns the rule and keep it in exactly one place.

Common mistake: pointing every dataset at direct query because it feels more real time. Each dashboard view then becomes warehouse concurrency, and a morning rush of refreshes can queue behind ELT. Import the wide tables into SPICE on a schedule and reserve direct query for small, fresh tables. Also line the refresh schedule up with the load window: a SPICE refresh that fires before COPY finishes quietly publishes yesterday's numbers.

## What is the purpose of the Redshift VACUUM command's DELETE option?

`VACUUM DELETE ONLY` reclaims the space occupied by rows that were deleted or superseded by updates, without re-sorting the table. Use it when storage or block efficiency is the problem but row order is already fine (for example after a large delete of an old partition).

```sql
vacuum delete only analytics.fact_orders;
analyze analytics.fact_orders;

-- Compare before and after
select "table", size as size_mb, nvl(unsorted, 0) as unsorted_pct
from svv_table_info
where "table" = 'fact_orders';
```

If rows are also out of order, use `VACUUM SORT ONLY` or full `VACUUM` instead; running both options in a single command is generally the most efficient.

An update or delete does not immediately free disk. Redshift writes the new version and marks the old rows as deleted, so the table grows and scans keep touching dead blocks until a vacuum rewrites them. VACUUM DELETE ONLY merges those blocks and returns the space, touching deleted rows only. The cost is a scan and temporary space during the rewrite, so schedule it when the change volume justifies it instead of nightly by habit. VACUUM cannot run inside a transaction block, and ALTER and VACUUM cannot run on the same table at the same time.

```sql
select "table",
       size as size_mb,
       pct_used,
       unsorted,
       vacuum_sort_benefit
from svv_table_info
where "schema" = 'analytics'
order by size desc;
```
Read `size` against `pct_used` to see how much allocated space is actually live, and `vacuum_sort_benefit` to judge whether a full VACUUM would pay off. SVV_TABLE_INFO requires superuser and hides empty tables unless the caller owns them.

Common mistake: using VACUUM DELETE ONLY to fix slow range scans. Reclaiming deleted rows does not put rows back in sort order, so the unsorted percentage stays high and zone-map pruning keeps missing blocks. If row order is the problem, use VACUUM SORT ONLY or a full VACUUM. The other mistake is running VACUUM DELETE ONLY on every table every night; when few rows changed, the scan and rewrite buy almost nothing. Look at deleted-row volume and table size before scheduling the work.

## Can you use Redshift with AWS Lambda for serverless data processing?

Yes, three patterns dominate: Lambda UDFs invoked from SQL, event-driven loads where an S3 event triggers a Lambda that runs `COPY` via the Redshift Data API, and lightweight transformation/validation before the load. Lambda is for short, stateless work; heavy transformation belongs in Glue or Redshift SQL itself.

In [ ]:
# S3 event -> Lambda -> Redshift Data API COPY
import boto3, json

redshift = boto3.client("redshift-data")


def handler(event, context):
    key = event["Records"][0]["s3"]["object"]["key"]
    sql = f"""
        copy analytics.stg_orders
        from 's3://acme-lake/{key}'
        iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
        format as parquet
        compupdate preset;
    """
    return redshift.execute_statement(
        WorkgroupName="analytics-wg", Database="analytics", Sql=sql
    )

Combine with a dead-letter queue and an audit table so failed loads are visible, not silent.

The Data API removes the JDBC connection problem. `execute_statement` is asynchronous: it returns an identifier immediately and you poll `describe_statement` until the status is FINISHED, FAILED or ABORTED. That matters at burst scale, because thousands of Lambda invocations no longer each hold a database connection. The caller also does not need VPC networking. Keep statements short and idempotent, set reserved concurrency so a retry storm cannot flood the cluster, and route failures to a dead-letter queue or an audit table rather than letting them disappear.

Lambda can also be reached from SQL as an external function, which pushes a per-row transform into the query.
```sql
create external function analytics.mask_email(varchar)
returns varchar
stable
lambda 'arn:aws:lambda:us-east-1:123456789012:function:mask-email'
iam_role 'arn:aws:iam::123456789012:role/redshift-lambda-invoke';
```
The role needs lambda:InvokeFunction permission. The function is called once per row, so it suits light masking or validation on modest row counts, not millions of rows.

Common mistake: treating `execute_statement` as blocking. It returns before the SQL finishes, so a Lambda that submits a COPY and returns success lets the pipeline move on while the load is still running. Poll the statement status, or orchestrate the call from Step Functions, before reporting success. A second mistake is using the Data API to insert rows one at a time instead of loading a file with COPY. The API is built for statements, not for high-volume ingestion.

## How does Redshift handle query optimization for large-scale joins and aggregations?

At scale the cost is dominated by data movement and spilling. The optimizer tries to push filters and partial aggregations below the join, choose `KEY`-colocated joins when possible, broadcast only genuinely small inputs, and aggregate on the slice before the final combine. When it cannot, a large hash join spills to disk and the query slows sharply.

```sql
-- Pre-aggregate before the join so the shuffle carries far fewer rows
with daily as (
  select customer_id, date_key, sum(net_amount) as revenue
  from analytics.fact_sales
  where date_key between 20260101 and 20261231
  group by customer_id, date_key
)
select c.region, d.calendar_month, sum(daily.revenue) as revenue
from daily
join analytics.dim_customer c on daily.customer_id = c.customer_id
join analytics.dim_date d on daily.date_key = d.date_key
group by c.region, d.calendar_month;
```

Watch `SVL_QUERY_REPORT` for a single slice doing much more work than the others; that indicates skew, which no amount of join tuning will fix.

Read the plan for data-motion labels before changing any SQL. `DS_DIST_NONE` means the join is colocated because both sides are distributed on the join key. `DS_BCAST_INNER` broadcasts the smaller side to every slice, which is fine for a true dimension and fatal for a large table. `DS_DIST_INNER` and `DS_DIST_BOTH` mean redistribution. After the motion, a large hash join needs work memory for the build side; when it does not fit, the step shows `is_diskbased = 't'` in `SVL_QUERY_SUMMARY`.

```sql
explain
select c.region, sum(f.net_amount) as revenue
from analytics.fact_sales f
join analytics.dim_customer c
  on f.customer_id = c.customer_id
group by c.region;
```
The plan shows join order, motion operators and where the aggregate runs. If the fact table is distributed on customer_id and the dimension is DISTSTYLE ALL or keyed on the same column, this join should need no redistribution, which is the shape you want to see.

Common mistake: tuning SQL when the real problem is one slice. If `skew_rows` in SVV_TABLE_INFO is large, one slice holds far more rows and finishes last while the rest idle, and total time is set by that slice. Adding nodes does not fix skew. Pick a higher-cardinality distkey, or fall back to EVEN when the current key concentrates rows. Check for skew before rewriting joins or resizing the cluster.

## Can you use Redshift with Amazon Athena for ad-hoc SQL queries on S3 data?

Yes, in a complementary way. Athena queries the same Glue Data Catalog and S3 objects serverlessly, which is ideal for occasional ad-hoc exploration and one-off lake queries without touching cluster capacity. Redshift Spectrum handles the same data when it must be joined with warehouse tables. Keep one catalog and one file layout so both engines see the same partitions.

```sql
-- Athena over the same Parquet prefix that Redshift Spectrum reads
select
  dt,
  count(*) as orders,
  sum(amount) as revenue
from acme_lake.orders
where dt between '2026-10-01' and '2026-10-07'
group by dt
order by dt;
```

Practical split: Athena for sparse exploration and log forensics; Redshift/ Spectrum for governed, join-heavy, scheduled analytics.

Athena and Spectrum read the same files, but the cost models differ. Athena bills by bytes scanned per query. Spectrum runs on a separate fleet and bills for the S3 scan while the result flows back into the cluster query. Both get faster and cheaper with columnar, compressed, partitioned data, because both prune Hive partitions the same way. Keep a single Glue Data Catalog: external tables created from Redshift with `CREATE EXTERNAL TABLE` land in that catalog, so Athena can query them. Redshift local tables never appear there; only the lake side is shared.

```sql
create external table spectrum.orders (
  order_id bigint,
  dt date,
  amount decimal(12,2)
)
stored as parquet
location 's3://acme-lake/orders/'
table properties ('numRows'='1000000');
```
Set `numRows` so the optimizer has a size estimate. ANALYZE does not collect statistics for external tables, so that table property and partition pruning are the main levers for plan quality.

Common mistake: expecting Athena to see new partitions the moment files land. The Glue table must learn about them through a crawler, `MSCK REPAIR TABLE`, or an explicit partition add, or the new `dt` values are invisible and the query quietly returns partial results. The second mistake is letting different jobs write conflicting layouts to the same prefix. A column type change there can break one engine while the other still reads the old files.

## What is the purpose of the Redshift COPY command's ENCRYPTED option?

`ENCRYPTED` tells `COPY` that the files in S3 were encrypted on the client side before upload. You supply the master symmetric key with `MASTER_SYMMETRIC_KEY`, and Redshift decrypts during load. This is different from server-side encryption (SSE-S3/SSE-KMS), where AWS decrypts the object transparently and `COPY` needs no key.

```sql
copy analytics.fact_orders
from 's3://acme-secure/orders/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as csv
master_symmetric_key '${MASTER_KEY}'
encrypted;
```

```bash
# Server-side encrypted variant: no key needed in COPY, but the role needs kms:Decrypt
aws s3 cp ./orders.parquet s3://acme-secure/orders/ \
  --sse aws:kms --sse-kms-key-id alias/acme-data
```

Client-side encryption means the file left S3 already encrypted with a symmetric key you manage. COPY decrypts it during load using the key supplied with MASTER_SYMMETRIC_KEY, which is a base64-encoded AES-256 key. Redshift does not store that key, so losing it means losing the load. The server-side options are different: SSE-S3 is fully transparent, and SSE-KMS requires kms:Decrypt on the role plus a key policy that allows it, including for cross-account buckets. Decide who owns the key before the pipeline is built; retrofitting key management later is painful.

```sql
unload ('select * from analytics.fact_orders')
to 's3://acme-secure/orders_out/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
format as csv
encrypted
master_symmetric_key '${MASTER_KEY}'
allowoverwrite
parallel off;
```
The same client-side option exists on UNLOAD, so exported files can require the key too. Keep the key in a secret manager and inject it at runtime; never bake it into a script or a saved query.

Common mistake: mixing the cases up. A bucket with default SSE-KMS encryption plus `ENCRYPTED` in COPY is not double protection; it is a mismatch, because COPY expects to decrypt bytes that AWS already handles server-side. Conversely, SSE-KMS without kms:Decrypt on the role fails with an error that looks like an S3 permission problem. Also confirm format support before choosing client-side encryption; server-side encryption is the simpler default for columnar files such as Parquet.

## Can you use Redshift with AWS Step Functions for serverless workflow orchestration?

Yes. Step Functions orchestrates multi-step pipelines with retries, branching, and error handling, and has direct integrations for Redshift (including the Data API), Glue, Lambda, and S3. The typical pipeline is: check S3 for new files, run `COPY`, run `ANALYZE`, refresh materialized views, and publish a completion event.

```json
{
  "Comment": "Daily Redshift ELT",
  "StartAt": "CopyRaw",
  "States": {
    "CopyRaw": {
      "Type": "Task",
      "Resource": "arn:aws:states:::redshift-data:executeStatement",
      "Parameters": {
        "WorkgroupName": "analytics-wg",
        "Database": "analytics",
        "Sql.$": "$.copySql"
      },
      "Retry": [
        {"ErrorEquals": ["States.TaskFailed"], "IntervalSeconds": 60, "MaxAttempts": 3, "BackoffRate": 2}
      ],
      "Next": "RefreshViews"
    },
    "RefreshViews": {
      "Type": "Task",
      "Resource": "arn:aws:states:::redshift-data:executeStatement",
      "Parameters": {
        "WorkgroupName": "analytics-wg",
        "Database": "analytics",
        "Sql": "refresh materialized view analytics.mv_kpi_daily;"
      },
      "End": true
    }
  }
}
```

Step Functions is a good fit for AWS-native shops that do not want to run Airflow; for large DAGs with backfills and data-aware scheduling, Airflow or MWAA is usually more productive.

The Data API integration is asynchronous under the hood. ExecuteStatement returns a statement id, and the Step Functions integration polls DescribeStatement until the SQL finishes, so a state failure reflects a failed statement rather than a failed API call. Use Standard workflows for warehouse jobs: they allow long waits and keep an execution history, while Express workflows are capped at five minutes and run at-least-once. Wrap tasks in Retry with exponential backoff for throttling and serialization conflicts, and add Catch so a failed load routes to an alert instead of ending quietly.

Fan out with a Map state when a load has natural partitions. One COPY per partition key can run in parallel, and MaxConcurrency caps how many hit the cluster at once. Follow the loads with ANALYZE and then REFRESH MATERIALIZED VIEW, so statistics and summaries are current before dashboards read them. Keep statements small: the Data API has a request size limit, so pass an S3 key or a reference instead of a huge generated statement. Express workflows are a poor fit for long loads because the five-minute limit truncates them.

Common mistake: treating the executeStatement call as fire-and-forget. A state that only submits SQL and moves on lets the workflow report success before the data has landed, so downstream refreshes run against stale tables. Let the integration wait for completion, or poll the statement, before the next state starts. The related mistake is assuming the service is free: Step Functions charges per state transition, so tight polling loops and very chatty DAGs add cost that a single SQL statement would not.

## How does Redshift handle query optimization for data filtering and predicate pushdown?

Predicates are pushed as deep as possible: into the scan so zone maps skip blocks, into Spectrum so S3 objects, row groups, and partitions are pruned, and into federated sources so the remote database filters before returning rows. The optimizer also rewrites `WHERE` clauses, folds constants, and removes redundant joins, but it cannot push a non-sargable expression such as a function on the sort-key column.

```sql
-- Sargable: predicate matches the sort key and prunes blocks
select count(*) from analytics.fact_events
where event_date between date '2026-10-01' and date '2026-10-07';

-- Non-sargable: a function on the sort-key column defeats pruning
select count(*) from analytics.fact_events
where date_part(year, event_date) = 2026;
```

Write range predicates on the bare sort-key column, filter before joining when possible, and keep the leading sort key as the column you constrain most often.

Every predicate is classified at plan time. A sargable predicate on the bare sort key becomes a zone-map check: each slice stores each column in 1 MB blocks that carry min and max values, so a range that misses a block skips it entirely. A predicate on the distribution key lets the optimizer discard whole slices before reading. Spectrum pushes filters into the S3 scan, pruning partitions first and then Parquet row groups. Federated queries send the predicate to the remote engine. Equality on a leading sort key is cheapest. A cast, a function, or a leading wildcard makes a predicate non-sargable, so it is applied only after the scan.

```sql
-- Partition first, then zone maps, then a residual filter
select count(*) as orders
from spectrum_lake.orders
where dt = '2026-10-04'                 -- partition pruning
  and order_date >= date '2026-10-01'   -- block / row-group pruning
  and upper(status) = 'SHIPPED';        -- residual filter after scan
```
Check `EXPLAIN` for the effect. A pruned plan shows the scan with a `filter:` on the sort key and a small row estimate. A plan that wraps the key, for example `date_part(year, event_date) = 2026`, still scans every block. A join filter that lands after an `XN_HashJoin` has already read both inputs.

Common mistake: trusting a predicate just because it is written in the WHERE clause. `where event_date::date = date '2026-10-07'` casts the leading sort key and loses block pruning. Filtering the non-leading column of a compound sort key gives almost no benefit. A filter applied after a join does not shrink the join. Rewrite the predicate onto the bare key, or add the column to the sort key, and confirm from the plan that fewer rows are read.

## Can you use Redshift with Amazon Kinesis Data Firehose for real-time data ingestion?

Yes. Firehose buffers streaming records and delivers them to S3 (which Redshift then loads with `COPY`), or uses its Redshift destination, which stages records in S3 and issues a `COPY` for you with configurable buffering, retries, and a manifest. That makes near-real-time ingestion possible without running a consumer.

```sql
-- Firehose delivers to S3; a scheduled COPY loads new files in micro-batches
copy analytics.stg_stream_events
from 's3://acme-firehose/events/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as json 'auto'
compupdate preset
maxerror 100;
```

Firehose buffering (size and interval) is the latency/cost dial: smaller buffers mean fresher data and more, smaller S3 files. The Redshift destination further batches into `COPY`, so sub-minute latency is not its sweet spot - use it for seconds-to-minutes freshness.

Firehose buffers records and flushes when either the buffer size or the buffer interval is reached, then delivers a batch to S3. The Redshift destination stages those batches in S3 and issues a `COPY` with a manifest, so Redshift never talks to Firehose directly. The interval is the freshness dial: a short interval means fresher data and many small files. Use the JSON path for messy streams and convert to Parquet when the schema is stable, because a columnar load reads only the columns the table needs instead of parsing every row field by field.

```sql
-- Load a Firehose batch set through its manifest
copy analytics.stg_stream_events
from 's3://acme-firehose/events/manifest.json'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as json 'auto'
manifest
compupdate preset
statupdate on
maxerror 0;
```
`maxerror 0` fails on the first bad record, which is right for a replayable stream. Keep a dedupe step. Firehose delivery is at-least-once, so a retried batch can land the same event twice. Load raw rows into a staging table and merge into the target on the event key.

Common mistake: shrinking the buffer to chase low latency. Each flush becomes a separate S3 object, so a one-minute interval on a modest stream creates thousands of tiny files and makes `COPY` slow. Firehose cannot do sub-second latency, and the Redshift destination batches again. For true low latency use Kinesis Data Streams with Redshift streaming ingestion, which reads the stream into a materialized view without landing files first.

## What is the purpose of the Redshift VACUUM command's SORTONLY option?

`VACUUM SORT ONLY` re-sorts rows into sort-key order without reclaiming deleted space. Use it after out-of-order loads or large updates made the table unsorted, so zone maps and merge joins work again. It is cheaper than full `VACUUM` and can run to a threshold.

```sql
vacuum sort only analytics.fact_sales to 95 percent;

analyze analytics.fact_sales;

select "table", nvl(unsorted, 0) as unsorted_pct
from svv_table_info
where "table" = 'fact_sales';
```

If the table is both unsorted and bloated, a full `VACUUM` does both in one pass and is usually the better choice.

`VACUUM SORT ONLY` rewrites rows into sort-key order in one scan. It does not reclaim space from deleted rows, so the table keeps its old size on disk. The `TO n PERCENT` threshold stops early once n percent of rows are in sort order, which bounds the time on a table that is only slightly unsorted. `VACUUM BOOST` raises the priority of the run and uses more resources. SORT ONLY cannot fix an interleaved sort key; that needs `VACUUM REINDEX`. A full `VACUUM` does the sort and the space reclaim in one pass.

```sql
-- A bloated and unsorted table: full vacuum does both jobs
vacuum analytics.fact_sales to 95 percent;

-- Only unsorted, little churn: sorting is enough
vacuum sort only analytics.fact_events;

analyze analytics.fact_events;
```
`ANALYZE` is a separate step. VACUUM moves rows but does not refresh column statistics, so run `ANALYZE` after it. VACUUM cannot run inside a transaction block, and no other ALTER or VACUUM can run on the table at the same time.

Common mistake: wrapping a scheduled VACUUM in `BEGIN` and `COMMIT` along with the rest of the load script. Redshift rejects VACUUM inside a transaction block, and the failure can leave the surrounding logic half applied, so run it as its own statement on an AUTOCOMMIT connection. The other mistake is expecting space back from every sort-only run: only deleted rows free space, so a table that is merely unsorted does not shrink.

## Can you use Redshift with AWS Glue DataBrew for data preparation and cleansing?

Yes. DataBrew provides a visual, no-code interface for profiling and transforming datasets, and it can read from and write to Redshift (and S3, which Redshift then loads). Typical uses are standardizing messy source fields, removing duplicates, and profiling data quality before a `COPY`.

```text
Raw CSV in S3 -> DataBrew recipe (trim, cast, standardize, dedupe) -> Parquet in S3 -> COPY -> Redshift
```

```sql
-- After DataBrew writes clean Parquet, load it
copy analytics.stg_customers
from 's3://acme-cleansed/customers/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
compupdate on;
```

DataBrew is a good fit for analyst-driven preparation and one-off cleanup; for repeatable, version-controlled transformation, keep the logic in SQL/dbt or Glue jobs.

DataBrew is a serverless visual tool. A recipe is an ordered list of transformations such as trim, cast, standardize, dedupe, join, and aggregate, applied by a Spark job on AWS-managed nodes. A profile job computes statistics like null counts, distinct counts, and value distributions so quality problems are visible before the load. Output can land in S3 as Parquet or CSV, or be written back through JDBC. Because every run is a Spark job, cost scales with node count and runtime, and each run rereads the input.

Use DataBrew where an analyst needs to see and shape data quickly. Do not use it as the main transform engine for a large recurring pipeline. It has no SQL model, no column-level lineage, and limited testing. The repeatable path is to prototype the cleaning rules in DataBrew or a notebook, then codify the same rules as SQL in dbt or as a Glue job, keeping DataBrew for profiling and one-off cleanup. Land cleansed output in S3 and `COPY` it rather than writing rows back through JDBC.

Common mistake: pointing DataBrew at the Redshift fact table and writing cleansed rows back with a JDBC write. Row-by-row writes are far slower than `COPY` from S3, and the Spark job holds a connection open while it runs. Another mistake is treating the recipe as the source of truth for transformation logic. A recipe that lives only in the DataBrew console never reaches code review, so a quiet edit changes what the warehouse loads.

## How does Redshift handle query optimization for subqueries and correlated subqueries?

The optimizer decorrelates subqueries where possible, turning `IN`/`EXISTS` predicates into semi-joins and `NOT IN`/`NOT EXISTS` into anti-joins. Scalar correlated subqueries that cannot be decorrelated can be evaluated per row, which is slow on large tables. The usual fix is to rewrite them as joins or window functions.

```sql
-- Correlated scalar subquery: potentially per-row evaluation
select o.order_id,
       (select max(order_date) from analytics.fact_orders p
        where p.customer_id = o.customer_id) as last_order_date
from analytics.fact_orders o;

-- Rewritten with a window function: one pass, no per-row subquery
select
  order_id,
  max(order_date) over (partition by customer_id) as last_order_date
from analytics.fact_orders;
```

Use `EXPLAIN` to confirm the rewrite: look for `XN_HashJoin` with a semi/anti-join step rather than a nested loop over the outer table.

Decorrelation turns a subquery into a join when the correlation is an equality on a key. `IN` and `EXISTS` become semi-joins, which can stop at the first match; `NOT IN` and `NOT EXISTS` become anti-joins. A scalar subquery in the SELECT list is harder. It can be decorrelated into a grouped join when the correlation is a simple equality, but an arbitrary correlated scalar is evaluated per outer row. A large `IN (select ...)` is also rewritten against a distinct set rather than checked row by row.

```sql
-- Anti-join form: orders with no shipment
select o.order_id
from analytics.fact_orders o
where not exists (
  select 1 from analytics.fact_shipments s
  where s.order_id = o.order_id
);

-- The same shape written as a left anti-join
select o.order_id
from analytics.fact_orders o
left join analytics.fact_shipments s on s.order_id = o.order_id
where s.order_id is null;
```
Both forms let the optimizer build one hash table on the inner side and stream the outer table past it.

Common mistake: writing `NOT IN` against a subquery whose column can be NULL. If the subquery returns even one NULL, the comparison is unknown for every row and the query returns nothing. Use `NOT EXISTS`, or add `where col is not null`. The other mistake is nesting a correlated scalar subquery in a query that already touches the same table, which reads it twice when one grouped join or window function would do the job in a single pass.

## Can you use Redshift with AWS Lake Formation for data lake management and access control?

Yes. Lake Formation centralizes permissions on S3/Glue Catalog data, and Redshift Spectrum honors those grants, so an external table can be queried only by principals allowed by Lake Formation. Combined with IAM roles, this lets governance live in one place while the same S3 data is queried by Redshift, Athena, and Spark.

```sql
-- Spectrum external schema over a Lake Formation-governed database
create external schema spectrum_governed
from data catalog
database 'governed_lake'
iam_role 'arn:aws:iam::123456789012:role/redshift-spectrum-lf-role'
create external database if not exists;

select count(*) from spectrum_governed.orders where dt = '2026-10-04';
```

The role must be registered with Lake Formation and granted on the relevant tables; if a grant is missing, the query fails with an access error even though the S3 path is reachable.

Lake Formation holds grants on Data Catalog databases, tables, and columns. Spectrum checks those grants when it plans a query, in addition to IAM. The IAM role named in `CREATE EXTERNAL SCHEMA` must be registered with Lake Formation, and the caller needs SELECT on the table or on a tag the table carries. Lake Formation also supports row-level filters and column-level grants. With credential vending, the role gets temporary S3 credentials scoped to the granted data, so the role does not need the S3 path in its own policy.

Cross-account access works through resource links plus Lake Formation grants, and a tag-based model scales better than per-table grants. Keep the two governance layers clear in your head. Native Redshift tables are governed by Redshift grants and IAM. External tables are governed by Lake Formation and IAM. A query can pass one check and fail the other, so an access error on a Spectrum query does not always mean the S3 path is wrong.

Common mistake: granting S3 read on the bucket and assuming that is enough. A role with S3 access but no Lake Formation grant on the table fails the Spectrum query with an access error. The reverse also bites: with credential vending the role can read only the granted partitions even if its S3 policy is broader. Check the resource link too. A link created in another account carries no source grants, so grants must be made on the link as well.

## What is the purpose of the Redshift COPY command's GZIP option?

`GZIP` tells `COPY` that the input files are gzip-compressed, so Redshift decompresses them on read (and can still parallelize across multiple files). It is the classic way to reduce S3 transfer volume for delimited text. For Parquet and ORC, compression is internal to the format, so you do not use `GZIP` there.

```sql
copy analytics.stg_orders
from 's3://acme-lake/raw/orders_2026-10-04/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
csv
gzip
ignoreheader 1
region 'us-east-1';
```

```bash
# Splitting into many gzip files keeps the load parallel
aws s3 cp ./raw/ s3://acme-lake/raw/orders_2026-10-04/ --recursive --exclude "*" --include "*.csv.gz"
```

A single large gzip file loads on one slice and wastes the cluster; many modest gzip files are the right shape.

`COPY` reads each input file independently, so gzip files still load in parallel: one file is one unit of work. `GZIP` states the compression explicitly. Redshift also detects gzip from the file extension and header, so it can load `.gz` files without the option, but naming it avoids surprises when a file has no extension. Decompression happens on the slice as part of the load, so less data moves from S3 and less is scanned. GZIP applies to delimited text. Parquet and ORC carry their own compression, and passing `GZIP` with them is an error.

```sql
-- Many modest .csv.gz files, loaded in parallel across slices
copy analytics.stg_orders
from 's3://acme-lake/raw/orders_2026-10-04/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
csv
gzip
ignoreheader 1
truncatecolumns
maxerror 0;
```
Aim for a file count in the ballpark of the slice count. A handful of very large files leaves most slices idle; thousands of tiny files adds per-file overhead.

Common mistake: gzipping the whole extract into one file. gzip is not splittable, so `COPY` hands that single file to one slice and the rest of the cluster waits. Size each file near the one-file-per-slice target. The other mistake is adding `GZIP` to a `FORMAT AS PARQUET` load. Columnar formats are already compressed internally, and the option makes the load fail.

## How do you diagnose a slow Redshift query?

Slow-query work is a fixed order of checks, cheapest first. Confirm the query really is slow with a query id, read the per-step metrics for one query, check the table layout behind it, and only then rewrite SQL. Most slow queries on a healthy cluster are a layout problem (missing sort key, skewed distribution, stale statistics) or a queue problem (too little memory, too much concurrency), not a bad plan.

Start with the query history. `STL_QUERY` holds one row per query with `elapsed` in microseconds, so the ranking query is short:

```sql
select query, userid, label, starttime, elapsed / 1000000.0 as elapsed_s, aborted
from stl_query
where starttime > dateadd(hour, -24, getdate())
  and query > 0
order by elapsed desc
limit 20;
```

`STL_` views are step and segment level and are trimmed over time. Copy what you need into a permanent audit table if you want a longer window. For query text use `STL_QUERYTEXT` keyed by `query` and `sequence`.

For one query id, get the per-step breakdown. `STL_QUERY_METRICS` reports `run_time` and `cpu_time` in microseconds and `blocks_to_disk` when intermediate results spilled to disk. `SVL_QUERY_SUMMARY` adds `workmem` and the per-step `is_diskbased` flag:

```sql
select query, segment, step_type, run_time, cpu_time, blocks_read, blocks_to_disk
from stl_query_metrics
where query = 918273
order by run_time desc;

select query, stm, seg, step, avgtime, rows, bytes, workmem, is_diskbased
from svl_query_summary
where query = 918273
order by avgtime desc;
```

Then read the table layout with `SVV_TABLE_INFO` (superuser only). `size` is in 1-MB blocks, `skew_rows` is max slice rows divided by min slice rows, `unsorted` is the percent of rows outside sort order, and `stats_off` runs from 0 (current) to 100 (out of date):

```sql
select "table", diststyle, sortkey1, size, tbl_rows,
       skew_rows, unsorted, stats_off, vacuum_sort_benefit
from svv_table_info
where skew_rows > 1.5 or unsorted > 10 or stats_off > 10
order by size desc;
```

In [ ]:
# Flag table-design problems the way SVV_TABLE_INFO reports them.
tables = [
    {"table": "fact_sales", "size_mb": 1_820_000, "skew_rows": 3.4, "unsorted": 28.0, "stats_off": 2.0},
    {"table": "dim_product", "size_mb": 42, "skew_rows": 1.0, "unsorted": 0.0, "stats_off": 0.0},
    {"table": "stg_orders", "size_mb": 610_000, "skew_rows": 1.1, "unsorted": 61.0, "stats_off": 34.0},
]

for t in tables:
    reasons = []
    if t["skew_rows"] > 1.5:
        reasons.append("skewed rows")
    if t["unsorted"] > 10:
        reasons.append("unsorted rows")
    if t["stats_off"] > 10:
        reasons.append("stale stats")
    verdict = ", ".join(reasons) if reasons else "healthy"
    print(f'{t["table"]:<12} {t["size_mb"]:>9,} MB  {verdict}')

flagged = [t["table"] for t in tables if t["skew_rows"] > 1.5]
assert flagged == ["fact_sales"]
assert sum(t["size_mb"] for t in tables) == 2_430_042
assert tables[1]["unsorted"] == 0.0

Check the result cache before you tune anything. A repeated identical query can return from the result cache and look fast without doing any work, and a sudden slowdown can just mean the cache was invalidated by a load. Disable it for a session when you want honest timings:

```sql
set enable_result_cache_for_session to off;
```

Any change to a referenced table invalidates the cached result, and queries that use non-deterministic functions like `random()` are never cached. `SYS_QUERY_HISTORY` reports `result_cache_hit` and `result_cache_query_id`, which points at the query that filled the cache:

```sql
select query_id, result_cache_hit, result_cache_query_id, elapsed_time
from sys_query_history
where result_cache_hit = true
order by start_time desc
limit 20;
```

Common mistake: starting with `EXPLAIN` and rewriting the SQL. `EXPLAIN` shows the plan the optimizer chose, not how long each step took. Read `STL_QUERY_METRICS`, `SVL_QUERY_SUMMARY` and `SVV_TABLE_INFO` first; they usually show the layout or spilling problem in one query.

## How do you choose a distribution style and sort key from query patterns?

Distribution controls data movement; the sort key controls how much data you read. Answer two questions from the real workload, not from the schema diagram. Which column is joined most often and at the largest scale? That suggests `DISTKEY`. Which columns appear in `WHERE` range filters? That suggests `SORTKEY`.

A star-schema table pair shows the pattern. The fact is distributed on its most common join key, and the small dimension is copied to every slice:

```sql
create table analytics.fact_sales (
  sale_id      bigint        encode az64,
  order_ts     timestamp     encode az64 sortkey,
  product_key  integer       encode az64 distkey,
  store_key    integer       encode az64,
  customer_key bigint        encode az64,
  qty          smallint      encode az64,
  amount       decimal(12,2) encode az64
)
diststyle key;

create table analytics.dim_product (
  product_key integer     encode az64 sortkey,
  category    varchar(64) encode lzo,
  brand       varchar(64) encode lzo
)
diststyle all;
```

Use `DISTSTYLE EVEN` when no single join dominates, and `DISTSTYLE AUTO` when you do not yet know the workload. AUTO starts small tables as ALL and can move a large table to EVEN or KEY later.

Worked example. A 2 TB fact on a 12-slice cluster holds about 170 GB per slice. If the nightly query joins the whole fact to `dim_product` on `product_key`, a `KEY(product_key)` distribution lets every slice join its own rows with no network movement. With `EVEN`, the optimizer redistributes the build side on every run. The dimension at 40 MB is cheap to copy, so `ALL` removes its redistribution completely.

The sort key is separate. `SORTKEY (order_ts)` gives each 1-MB column block a tight timestamp range, so `where order_ts >= dateadd(day, -30, getdate())` skips most blocks. Compound sort keys help the leading columns first, so put the range-filter column first. Interleaved sort keys give equal weight to several columns, but they need `VACUUM REINDEX` and only pay off for a few low-cardinality filter columns.

You can change both later without a rebuild. The table stays queryable, though a sort-key change rewrites data and sort-key columns default to `RAW` encoding:

```sql
alter table analytics.fact_sales alter distkey product_key;
alter table analytics.fact_sales alter sortkey (order_ts, store_key);
alter table analytics.fact_sales alter diststyle auto;
```

`ALTER DISTSTYLE`, `ALTER DISTKEY`, `ALTER SORTKEY` and `VACUUM` cannot run on the same table at the same time. Check `SVV_ALTER_TABLE_RECOMMENDATIONS` and `SVL_AUTO_WORKER_ACTION` to see what automatic table optimization has already decided.

Common mistake: distributing on a surrogate primary key that nothing joins on. Every join then redistributes both sides, and rows scatter unevenly because the key is unique. A second mistake is `DISTSTYLE ALL` on a dimension that keeps growing; the copy is multiplied by slice count and loaded on every insert.

## How do you tune Redshift maintenance, COPY and UNLOAD?

Rows become unsorted because `COPY` appends data in file order and deletes leave gaps. Zone maps then lose their value: a block that should hold one narrow range of timestamps instead holds a wide one, so a range filter reads many more blocks. `SVV_TABLE_INFO.unsorted` reports the percentage of rows outside sort order, and `vacuum_sort_benefit` estimates the maximum scan improvement from fixing it.

Worked example. An 800 GB table at 4 percent read for a monthly filter reads about 32 GB when sorted. At 25 percent unsorted, the out-of-order rows are spread across blocks, so the same filter touches closer to 28 percent, or 224 GB. The extra 192 GB is pure maintenance debt.

Run maintenance by need, not by calendar:

```sql
vacuum sort only analytics.fact_sales to 75 percent;
vacuum delete only analytics.fact_sales;
vacuum reindex analytics.dim_product;
analyze analytics.fact_sales;
analyze compression analytics.fact_sales;
```

`VACUUM` reclaims space and re-sorts, `SORT ONLY` only re-sorts, `DELETE ONLY` only reclaims, and `REINDEX` rebuilds an interleaved sort key. `ANALYZE` collects the statistics the optimizer needs for row counts and join order, and `ANALYZE COMPRESSION` only reports suggested encodings. `VACUUM` cannot run inside a transaction block; `ANALYZE` can.

Tune `COPY` for parallelism, not for one big file. Redshift loads one file per slice at a time, so a 900 GB load onto 12 slices with about 3 files per slice means 36 files of roughly 25 GB each. Compressed, columnar files keep the leader node and the network out of the hot path:

```sql
copy analytics.fact_sales
from 's3://acme-lake/curated/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
manifest
format as parquet
compupdate preset
statupdate on
maxerror 0;
```

`MANIFEST` avoids an expensive S3 listing, `COMPUPDATE PRESET` applies the standard encodings without a full scan, and `STATUPDATE ON` refreshes statistics after the load. Use `COMPUPDATE ON` only for a new table whose data shape you have not seen.

`UNLOAD` is the mirror image and writes in parallel across slices:

```sql
unload ('select * from analytics.fact_sales where order_ts >= ''2026-01-01''')
to 's3://acme-lake/exports/fact_sales/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
format as parquet
parallel on
maxfilesize 256 mb
manifest
allowoverwrite
partition by (order_ts);
```

`MAXFILESIZE` keeps output files small enough for downstream readers, and `PARTITION BY` writes Hive-style folders that Spectrum and Athena can prune.

Common mistake: loading one multi-terabyte CSV. Only one slice reads it, the rest of the cluster idles, and the leader spends time splitting the stream. Equally bad is thousands of 1 MB files, which spends more time opening objects than reading them.

## How do you tune Redshift WLM, concurrency scaling and Spectrum?

Workload management divides cluster memory and query slots into queues. Automatic WLM allocates memory as queries arrive and picks concurrency on demand; manual WLM fixes `memory_percent` and `max_concurrency` per queue. WLM is cluster configuration, not table DDL: you set it in the console, through the Redshift API, or in the `wlm_json_configuration` cluster parameter. There is no `ALTER WORKLOAD MANAGEMENT` SQL statement; use the JSON config and the WLM system views.

A workable automatic-WLM config separates ETL, BI, and short queries. `short_query_queue` is the short query acceleration (SQA) express lane, which keeps sub-second dashboard queries from waiting behind long scans:

```json
[
  {
    "auto_wlm": true,
    "name": "default",
    "query_group": ["etl"],
    "priority": "high",
    "short_query_queue": false
  },
  {
    "auto_wlm": true,
    "name": "bi",
    "user_group": ["bi_role"],
    "priority": "normal",
    "short_query_queue": true
  }
]
```

Inspect queue state with `STV_WLM_QUERY_STATE` and `STV_WLM_SERVICE_CLASS_CONFIG`. A query that hit a query monitoring rule appears in `STL_WLM_RULE_ACTION`.

Concurrency scaling adds transient clusters when the main cluster's queues are full, so read queries keep starting instead of waiting. It is a cluster-level setting with `max_concurrency_scaling_clusters`, and queries that ran on a scaling cluster show `compute_type = 'primary-scale'` in `SYS_QUERY_HISTORY`. SQA and scaling both help bursty read workloads, but neither fixes a query that scans too much; fix the layout first. AQUA is an RA3 capability that offloads some scans and aggregation to a cache layer in front of Redshift Managed Storage, so it helps large sequential scans on RA3 without any SQL change.

For Spectrum, pruning decides cost because you pay for bytes scanned from S3. Partition on the column you filter most, keep files at a readable size, and store columnar formats so only referenced columns are read:

```sql
create external schema spectrum
from data catalog
database 'analytics_ext'
iam_role 'arn:aws:iam::123456789012:role/redshift-spectrum-role'
create external database if not exists;

create external table spectrum.orders (
  order_id    bigint,
  order_ts    timestamp,
  customer_id bigint,
  amount      decimal(12,2)
)
partitioned by (dt date)
stored as parquet
location 's3://acme-lake/curated/orders/'
table properties ('numRows'='1200000000');
```

`ANALYZE` does not collect external-table statistics. Set `numRows` explicitly so the optimizer does not assume the external table is always the larger side.

Common mistake: treating concurrency scaling as a substitute for query tuning. If every dashboard query scans the full fact table, scaling just buys more clusters doing the same waste. Fix distribution, sort order and statistics, then use scaling and SQA for genuine bursts.

## Real world use cases for Redshift

### 1. Retail: store-level sales reporting on RA3

A grocery chain with 800 stores loads 6 TB/month of POS line items and nightly inventory snapshots. The fact table is distributed on `product_key` and sorted by `date_key`; small dimensions use `DISTSTYLE ALL`; cold history older than 24 months lives as Parquet in S3 and is queried through Spectrum.

```sql
create table analytics.fact_pos_lines (
  line_id     bigint,
  date_key    integer,
  store_key   integer,
  product_key integer,
  quantity    integer,
  net_amount  numeric(12,2)
)
diststyle key distkey (product_key)
compound sortkey (date_key, store_key);

-- Store/day rollup served to BI
create materialized view analytics.mv_store_daily
auto refresh yes
as
select date_key, store_key, sum(net_amount) as revenue, sum(quantity) as units
from analytics.fact_pos_lines
group by date_key, store_key;

-- Cold history joined with hot data, no load required
select p.category, sum(s.net_amount) as revenue
from spectrum_lake.pos_history s
join analytics.dim_product p on s.product_key = p.product_key
where s.dt between '2024-01-01' and '2024-12-31'
group by p.category;
```

Why this design: the `product_key` distribution colocates the fact with a `DISTSTYLE ALL` product dimension, so the dominant join has no redistribution; the date-first sort key gives zone-map pruning for the "last N days by store" dashboard; the materialized view keeps the interactive query off the raw fact; and Spectrum keeps the 24-month storage bill on S3 instead of RA3 managed storage.

### 2. Fintech: daily reconciliation with strict freshness and auditability

A payments processor ingests 120 million card transactions per day and must reconcile them against processor settlement files by 06:00, with seven years of queryable history for audits. The load is a single parallel `COPY` from encrypted Parquet, followed by a `MERGE`-based exception step and an `ANALYZE`.

```sql
copy analytics.stg_transactions
from 's3://acme-secure/txn/2026-10-04/'
iam_role 'arn:aws:iam::123456789012:role/redshift-copy-role'
format as parquet
compupdate preset
maxerror 0;

merge into analytics.fact_transactions t
using (
  select txn_id, account_id, txn_ts, amount, processor_amount, ingested_at,
         row_number() over (partition by txn_id order by ingested_at desc) as rn
  from analytics.stg_transactions
) s
on t.txn_id = s.txn_id and s.rn = 1
when matched then update set t.processor_amount = s.processor_amount
when not matched then insert (txn_id, account_id, txn_ts, amount, processor_amount, ingested_at)
values (s.txn_id, s.account_id, s.txn_ts, s.amount, s.processor_amount, s.ingested_at);

analyze analytics.fact_transactions;

select count(*) as breaks
from analytics.fact_transactions
where amount <> processor_amount and txn_date = current_date - 1;
```

Why this design: `MAXERROR 0` makes a bad settlement file fail the run instead of loading partially; the dedup window plus `MERGE` makes reruns idempotent; the sorting is on the transaction timestamp so audit range queries prune; and audit logging plus cross-region snapshot copy satisfy the retention and DR requirements.

### 3. Marketing: multi-channel attribution and spend reporting

An e-commerce company spends across Google, Meta, and TikTok and loads about 40 GB/day of click and impression data plus 2 GB/day of ad spend. Ad platform files land in S3 as Parquet; attribution runs nightly in Redshift against order data; dashboards query a materialized view.

```sql
create table analytics.fact_touchpoints (
  touch_id    bigint,
  user_id     bigint,
  channel     varchar(32),
  campaign_id varchar(64),
  event_ts    timestamp,
  event_date  date
)
diststyle key distkey (user_id)
compound sortkey (event_date, channel);

create table analytics.fact_spend (
  spend_date  date,
  channel     varchar(32),
  campaign_id varchar(64),
  spend       numeric(12,2)
)
diststyle even sortkey (spend_date, channel);

create materialized view analytics.mv_roas_daily
auto refresh yes
as
select
  t.event_date,
  t.channel,
  count(distinct t.user_id) as reach,
  sum(s.spend) as spend
from analytics.fact_touchpoints t
left join analytics.fact_spend s
  on s.channel = t.channel and s.spend_date = t.event_date
group by t.event_date, t.channel;
```

Why this design: distributing both facts on the channel/date grain keeps the nightly attribution join local; sorting on `event_date` keeps the 30-day lookback scans bounded; and the materialized view gives the marketing team sub-second ROAS without re-running attribution.

### 4. SaaS: product and revenue analytics with governed self-service

A B2B SaaS company stores 3 TB of event and subscription data in Redshift and exposes it to analysts and to embedded customer dashboards. Row-level security isolates tenants, column-level grants hide PII, and WLM separates BI from ETL.

```sql
-- Tenant isolation and column-level least privilege
create role tenant_reader;
grant usage on schema product to role tenant_reader;
grant select (event_date, account_id, event_name, event_count) on product.fact_usage to role tenant_reader;

create rls policy tenant_isolation
  with (account_id varchar(64))
  using (account_id = current_setting('app.account_id'));

attach rls policy tenant_isolation
  on product.fact_usage to role tenant_reader;

-- WLM separation: dashboards get priority, ETL gets a low-priority queue
-- (configured on the cluster; see the WLM question)
```

Why this design: row-level security lets one physical table serve every tenant without duplicating data; WLM priority ensures an ETL backfill cannot degrade customer-facing dashboards; and concurrency scaling absorbs the morning dashboard spike without permanently adding nodes.

### 5. Logistics: fleet telemetry and SLA monitoring

A logistics operator tracks 9,000 vehicles reporting every 15 seconds, about 50 GB/day compressed. Events land in S3 via Firehose, load into a date-sorted fact table, and a nightly job computes on-time delivery SLA by lane. Data older than 90 days is archived with `UNLOAD` to Parquet and dropped from the cluster.

```sql
create table analytics.fact_telemetry (
  vehicle_id  bigint,
  event_ts    timestamp,
  event_date  date,
  lat         numeric(9,6),
  lon         numeric(9,6),
  speed_kph   numeric(6,2),
  fuel_pct    numeric(5,2)
)
diststyle even
compound sortkey (event_date, vehicle_id);

-- Archive before dropping the hot copy
unload ('select * from analytics.fact_telemetry where event_date < current_date - 90')
to 's3://acme-archive/telemetry/'
iam_role 'arn:aws:iam::123456789012:role/redshift-unload-role'
format as parquet
partition by (event_date)
manifest
allowoverwrite;

delete from analytics.fact_telemetry where event_date < current_date - 90;
vacuum delete only analytics.fact_telemetry;
analyze analytics.fact_telemetry;
```

Why this design: a date-first compound sort key makes the "last 24 hours for a lane" range query prune hard; `UNLOAD` to Parquet keeps archive costs on S3 while remaining queryable through Spectrum; and a weekly `VACUUM DELETE ONLY` reclaims the space from the archive delete without a full resort.

### 6. Media: ad events and campaign reporting
A streaming ad exchange buffers billable events through Firehose into S3 as Parquet partitioned by `dt` and `hour`, and a `COPY` loads the hot window into RA3. Older partitions stay in S3 and are queried through Spectrum. The fact is distributed on `advertiser_id` because most dashboards slice by advertiser, and it is sorted by `event_ts`.
```sql
create table analytics.fact_ad_events (
  event_id      bigint,
  event_ts      timestamp,
  advertiser_id integer,
  campaign_id   integer,
  impressions   integer,
  spend         numeric(12,4)
)
diststyle key distkey (advertiser_id)
compound sortkey (event_ts, campaign_id);
```
The `event_ts` range prunes blocks, the distribution key colocates the join with the advertiser dimension, and Spectrum keeps the long tail of hourly partitions off cluster storage.

### 7. SaaS: multi-tenant usage analytics
A SaaS vendor keeps tenant usage in one large fact table. `tenant_id` is the distribution key, so each tenant's rows are colocated and a per-tenant dashboard reads a narrow set of slices. Very small tenants would skew that distribution, so group them into a shared pseudo-tenant. Interactive dashboards run on a separate WLM queue with short query acceleration, and Concurrency Scaling adds transient clusters when many tenants query at once.
```sql
create table analytics.fact_usage (
  tenant_id   integer,
  usage_date  date,
  feature_key integer,
  events      bigint
)
diststyle key distkey (tenant_id)
compound sortkey (usage_date);
```
Isolation lives in the presentation layer, not the physical layout, so the metric views filter every query on the caller's tenant.

Common mistake: copying one domain's physical design into every domain. `DISTSTYLE KEY` on a column that is not in the dominant join, or that is badly skewed, moves the bottleneck into redistribution at query time. `DISTSTYLE ALL` on a dimension that keeps growing copies it to every slice and inflates the table. And skipping `ANALYZE` after a large load leaves stale row counts, so the optimizer can pick a poor join order even though the schema is right.

## Redshift exercises

These exercises are standard-library Python. They model the arithmetic behind Redshift design choices: distribution, skew, sort order, load parallelism, maintenance debt, queue sizing and Spectrum cost. Run them with the repository virtual environment.

### Try it 1. Pick a distribution style from join patterns

**Task.** Write `pick_diststyle(fact_gb, top_join_column, join_share, dim_gb)` for a fact table. Use `KEY(column)` when the dominant join column appears in at least 60 percent of queries, `ALL` when the joined dimension is under 1 GB, and `EVEN` otherwise.

**Hint.** Check the rules in order: dominant join key first, then tiny dimension, then the default.

**Expected result.** `KEY(product_key)`, `ALL`, `EVEN`.

In [ ]:
def pick_diststyle(fact_gb, top_join_column, join_share, dim_gb):
    # KEY colocates the dominant join; ALL copies a tiny dimension; EVEN otherwise.
    if join_share >= 0.6 and top_join_column:
        return f"KEY({top_join_column})"
    if dim_gb < 1:
        return "ALL"
    return "EVEN"


cases = [
    (pick_diststyle(2000, "product_key", 0.82, 0.4), "KEY(product_key)"),
    (pick_diststyle(2000, "product_key", 0.20, 0.4), "ALL"),
    (pick_diststyle(2000, "product_key", 0.20, 80.0), "EVEN"),
]

for got, _ in cases:
    print(got)

for got, expected in cases:
    assert got == expected

### Try it 2. Compute slice skew from row counts

**Task.** Given rows per slice, compute the skew ratio as max rows divided by min rows, the ideal even share, and how many rows sit on the hottest slice above that share.

**Hint.** This is the same `skew_rows` definition as `SVV_TABLE_INFO`. Flag it when the ratio is above 1.5.

**Expected result.** Ratio 1.031, ideal share 9,950,000 rows, excess 150,000 rows.

In [ ]:
rows = [10_000_000, 9_800_000, 10_100_000, 9_900_000]

skew_ratio = max(rows) / min(rows)
ideal_share = sum(rows) / len(rows)
excess = max(rows) - ideal_share

print(f"skew_rows = {skew_ratio:.3f}")
print(f"ideal share = {ideal_share:,.0f} rows")
print(f"excess on hottest slice = {excess:,.0f} rows")

assert round(skew_ratio, 3) == 1.031
assert ideal_share == 9_950_000.0
assert excess == 150_000.0
assert skew_ratio < 1.5

### Try it 3. Choose a sort key from filter columns

**Task.** Write `pick_sortkey(columns)` where `columns` maps a column name to `(filter_count, is_range)`. Order range-filter columns first, then by filter count, so the leading sort-key column is the one that benefits most.

**Hint.** Sort with a key tuple of `(is_range, filter_count)` descending.

**Expected result.** `['order_ts', 'region', 'customer_id']`.

In [ ]:
def pick_sortkey(columns):
    # Range filters benefit most from sort order; then filter frequency.
    ordered = sorted(
        columns.items(),
        key=lambda item: (item[1][1], item[1][0]),
        reverse=True,
    )
    return [name for name, _ in ordered]


columns = {
    "order_ts": (420, True),
    "region": (390, False),
    "customer_id": (120, False),
}

result = pick_sortkey(columns)
print("sortkey =", result)

assert result == ["order_ts", "region", "customer_id"]
assert columns[result[0]][1] is True

### Try it 4. Estimate COPY file count for parallelism

**Task.** A 900 GB load targets a 12-slice cluster with about 3 files per slice. Compute the number of files and the size of each file in MiB. Use 1 GB = 1024 MiB.

**Hint.** `files = slices * files_per_slice`, then `file_mib = total_gb * 1024 / files`.

**Expected result.** 36 files of 25,600 MiB each.

In [ ]:
total_gb = 900
slices = 12
files_per_slice = 3

files = slices * files_per_slice
file_mib = total_gb * 1024 / files

print(f"files = {files}")
print(f"each file = {file_mib:,.0f} MiB")

assert files == 36
assert file_mib == 25_600.0
assert files % slices == 0

### Try it 5. Compute unsorted-rows impact from a table scan

**Task.** An 800 GB table is 25 percent unsorted. A range filter normally reads 4 percent of a sorted table. Model the unsorted rows as spread across all blocks, so the effective read fraction is `sorted_fraction * target + unsorted_fraction`. Compute the bytes read and the extra bytes caused by the unsorted rows.

**Hint.** `sorted_fraction = 1 - unsorted_fraction`. Extra bytes are the difference from the sorted-only read.

**Expected result.** Effective fraction 0.28, 224 GB read, 192 GB extra.

In [ ]:
table_gb = 800
unsorted_fraction = 0.25
target_fraction = 0.04

sorted_fraction = 1 - unsorted_fraction
effective_fraction = sorted_fraction * target_fraction + unsorted_fraction
gb_read = table_gb * effective_fraction
gb_sorted = table_gb * target_fraction

print(f"effective read fraction = {effective_fraction:.2f}")
print(f"bytes read = {gb_read:.0f} GB")
print(f"extra from unsorted rows = {gb_read - gb_sorted:.0f} GB")

assert round(effective_fraction, 2) == 0.28
assert round(gb_read, 6) == 224.0
assert round(gb_read - gb_sorted, 6) == 192.0

### Try it 6. Size a WLM queue from concurrency

**Task.** A BI queue receives 2 queries per second and each runs for 15 seconds. Use Little's law to find the average concurrency, then add a 50 percent safety margin and round up to set `max_concurrency`.

**Hint.** `concurrency = arrival_rate * avg_seconds`; slots are `ceil(concurrency * 1.5)`.

**Expected result.** Average concurrency 30, queue slots 45.

In [ ]:
import math

arrival_per_second = 2.0
avg_seconds = 15.0
safety = 1.5

concurrency = arrival_per_second * avg_seconds
slots = math.ceil(concurrency * safety)

print(f"average concurrency = {concurrency:.0f}")
print(f"queue max_concurrency = {slots}")

assert concurrency == 30.0
assert slots == 45
assert slots > concurrency

### Try it 7. Compare Spectrum and local table scan cost

**Task.** A 100 TB dataset is scanned with Spectrum at an illustrative 5 USD per TB scanned. A local table reads 5 percent of the data thanks to zone maps; the Spectrum table reads 20 percent after partition pruning. Compute both scan costs and the difference.

**Hint.** `cost = table_tb * fraction * usd_per_tb`. The local table has storage cost too, so this compares scan cost only.

**Expected result.** Local 25.0 USD, Spectrum 100.0 USD, Spectrum 75.0 USD more.

In [ ]:
table_tb = 100
local_fraction = 0.05
spectrum_fraction = 0.20
usd_per_tb = 5.0

local_cost = table_tb * local_fraction * usd_per_tb
spectrum_cost = table_tb * spectrum_fraction * usd_per_tb

print(f"local scan cost = {local_cost:.1f} USD")
print(f"Spectrum scan cost = {spectrum_cost:.1f} USD")
print(f"difference = {spectrum_cost - local_cost:.1f} USD")

assert local_cost == 25.0
assert spectrum_cost == 100.0
assert spectrum_cost - local_cost == 75.0